# KuriBot · ChatBot híbrido de Geomática (UEES · Inteligencia Artificial · Semana 4)

Este cuaderno reproduce el proyecto de principio a fin en Google Colab (CPU o GPU):

| Paso | Qué hace |
|---|---|
| 1 | Instala dependencias y descarga BETO desde la versión oficial de spaCy en GitHub |
| 2 | Escribe el código fuente del proyecto (idéntico al Anexo C del informe) |
| 3 | Construye la base de conocimiento y el corpus sintéticos |
| 4 | Entrena y compara los cuatro modelos (iteración 1) y el modelo ajustado (iteración 2) |
| 5 | Ejecuta las pruebas unitarias y la suite de aceptación de 34 escenarios |
| 6 | Conversa con el ChatBot |

**Datos:** todos son sintéticos. **Tiempo de referencia:** en el entorno de desarrollo (CPU de 2 hilos, como el de Colab gratuito) la validación cruzada de los cuatro modelos tardó unos 17 min y los modelos finales de la iteración 1, unos 5 min más; la iteración 2 duplica los datos de entrenamiento y tarda algo más. Con GPU (*Entorno de ejecución → Cambiar tipo*) el tiempo baja.

## 1. Dependencias

In [ ]:
!apt-get -qq install -y graphviz > /dev/null
!pip -q install pgmpy pyproj shapely streamlit safetensors
!pip -q install --no-deps https://github.com/explosion/spacy-models/releases/download/es_dep_news_trf-3.8.0/es_dep_news_trf-3.8.0-py3-none-any.whl
!pip -q install --no-deps https://github.com/explosion/spacy-models/releases/download/es_core_news_lg-3.8.0/es_core_news_lg-3.8.0-py3-none-any.whl
import os; os.makedirs("kuribot/kuribot", exist_ok=True)
for d in ["datos", "scripts", "experimentos", "pruebas", "modelos", "resultados", "docs/figuras"]:
    os.makedirs(f"kuribot/{d}", exist_ok=True)
%cd kuribot

## 2. Código fuente

In [ ]:
%%writefile kuribot/__init__.py


In [ ]:
%%writefile kuribot/geodesia.py
"""Operaciones geodésicas y espaciales de KuriBot (pyproj + Shapely).

Convenciones
- Sistema de trabajo: WGS 84 / UTM zona 17S (EPSG:32717), en metros.
- PSAD56 / UTM 17S (EPSG:24877) se transforma con la operación EPSG:3990
  «PSAD56 to WGS 84 (14)», Helmert de 7 parámetros (convención coordinate frame) válida
  para Ecuador continental, con exactitud declarada de 5 m en la base EPSG. En producción
  deben confirmarse los parámetros oficiales vigentes del IGM antes de usar los resultados
  con fines legales o catastrales.
- SIRGAS 2000 / UTM 17S (EPSG:31977) se trata como equivalente a WGS 84 al nivel métrico
  (transformación nula de la base EPSG); la diferencia real depende de la época de referencia.
"""
from __future__ import annotations

import math
from dataclasses import dataclass
from functools import lru_cache

from pyproj import CRS, Transformer
from pyproj.transformer import TransformerGroup

EPSG = {
    "WGS84_GEO": 4326,
    "WGS84_UTM17S": 32717,
    "PSAD56_GEO": 4248,
    "PSAD56_UTM17S": 24877,
    "SIRGAS2000_UTM17S": 31977,
}
OPERACION_PSAD56_WGS84 = 3990  # PSAD56 to WGS 84 (14), Ecuador continental


@dataclass(frozen=True)
class Punto:
    """Coordenada de entrada tal como la expresó el usuario."""
    x: float              # Este (m) o longitud (°)
    y: float              # Norte (m) o latitud (°)
    tipo: str             # "utm" o "geo"
    datum: str | None     # "WGS84", "PSAD56", "SIRGAS" o None si no se indicó


@lru_cache(maxsize=None)
def _transformador(origen: int, destino: int) -> Transformer:
    return Transformer.from_crs(origen, destino, always_xy=True)


@lru_cache(maxsize=None)
def _psad56_a_wgs84_geo() -> Transformer:
    """Transformación PSAD56 (geográficas) -> WGS 84 (geográficas) fijada en EPSG:3990."""
    grupo = TransformerGroup(EPSG["PSAD56_GEO"], EPSG["WGS84_GEO"], always_xy=True)
    for t in grupo.transformers:
        ident = t.to_json_dict().get("id", {})
        if ident.get("code") == OPERACION_PSAD56_WGS84:
            return t
    raise RuntimeError("La base EPSG local no contiene la operación 3990.")


def descripcion_transformacion() -> str:
    t = _psad56_a_wgs84_geo()
    nombre = t.description.split(" (with axis")[0]
    return f"{nombre} (EPSG:{OPERACION_PSAD56_WGS84}; exactitud declarada {t.accuracy:.0f} m)"


def psad56_utm_a_wgs84_utm(e: float, n: float) -> tuple[float, float]:
    lon, lat = _transformador(EPSG["PSAD56_UTM17S"], EPSG["PSAD56_GEO"]).transform(e, n)
    lon2, lat2 = _psad56_a_wgs84_geo().transform(lon, lat)
    return _transformador(EPSG["WGS84_GEO"], EPSG["WGS84_UTM17S"]).transform(lon2, lat2)


def wgs84_utm_a_psad56_utm(e: float, n: float) -> tuple[float, float]:
    lon, lat = _transformador(EPSG["WGS84_UTM17S"], EPSG["WGS84_GEO"]).transform(e, n)
    lon2, lat2 = _psad56_a_wgs84_geo().transform(lon, lat, direction="INVERSE")
    return _transformador(EPSG["PSAD56_GEO"], EPSG["PSAD56_UTM17S"]).transform(lon2, lat2)


def a_wgs84_utm(p: Punto) -> tuple[float, float]:
    """Lleva cualquier entrada soportada a WGS 84 / UTM 17S."""
    datum = (p.datum or "WGS84").upper()
    if p.tipo == "geo":
        if datum == "PSAD56":
            lon, lat = _psad56_a_wgs84_geo().transform(p.x, p.y)
        else:
            lon, lat = p.x, p.y
        return _transformador(EPSG["WGS84_GEO"], EPSG["WGS84_UTM17S"]).transform(lon, lat)
    if datum == "PSAD56":
        return psad56_utm_a_wgs84_utm(p.x, p.y)
    return p.x, p.y  # WGS84 o SIRGAS (equivalencia métrica)


def todas_las_representaciones(p: Punto) -> dict[str, tuple[float, float]]:
    e, n = a_wgs84_utm(p)
    lon, lat = _transformador(EPSG["WGS84_UTM17S"], EPSG["WGS84_GEO"]).transform(e, n)
    pe, pn = wgs84_utm_a_psad56_utm(e, n)
    return {"WGS84_UTM17S": (e, n), "WGS84_GEO": (lat, lon), "PSAD56_UTM17S": (pe, pn)}


def dms(valor: float, eje: str) -> str:
    hemis = ("N" if valor >= 0 else "S") if eje == "lat" else ("E" if valor >= 0 else "W")
    v = abs(valor)
    g = int(v)
    m = int((v - g) * 60)
    s = (v - g - m / 60) * 3600
    return f"{g}°{m:02d}'{s:05.2f}\"{hemis}"


def dentro_de_ecuador_utm17s(e: float, n: float) -> bool:
    """Control de rango aproximado para Ecuador continental expresado en UTM 17S."""
    return 450_000 <= e <= 1_100_000 and 9_440_000 <= n <= 10_170_000


def distancia(a: tuple[float, float], b: tuple[float, float]) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])


def nombre_crs(epsg: int) -> str:
    return CRS.from_epsg(epsg).name


In [ ]:
%%writefile kuribot/entidades.py
"""Extracción de entidades basada en reglas (expresiones regulares + diccionario geográfico).

La intención la decide la red neuronal; los valores concretos (coordenadas, códigos, fechas,
evidencias) los extrae este módulo de forma determinista y auditable. También detecta datos
personales para enmascararlos antes de registrar la conversación.
"""
from __future__ import annotations

import difflib
import re
import unicodedata
from dataclasses import dataclass, field

from .geodesia import Punto

# ------------------------------------------------------------------ utilidades


def plano(texto: str) -> str:
    """Minúsculas y sin tildes (conserva la ñ) para comparar con diccionarios."""
    t = unicodedata.normalize("NFD", texto.lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn" or c == "̃")
    return unicodedata.normalize("NFC", t)


def _num(s: str) -> float:
    return float(s.replace(",", "."))


# ------------------------------------------------------------------ coordenadas

_UTM_ETIQ = re.compile(r"(?:\b[ex]|este)\s*[:=]?\s*(\d{6}(?:\.\d+)?)\s*[,;]?\s*(?:\b[ny]|norte)\s*[:=]?\s*(\d{7}(?:\.\d+)?)", re.I)
_UTM_PAR = re.compile(r"(?<![\d.])(\d{6}(?:\.\d+)?)\s*[,;]?\s+(\d{7}(?:\.\d+)?)(?![\d])|(?<![\d.])(\d{6}(?:\.\d+)?),(\d{7}(?:\.\d+)?)(?![\d])")
_DEC = re.compile(r"(?:lat(?:itud)?\s*[:=]?\s*)?(-?\d{1,2}\.\d{3,})\s*[,;]?\s+(?:lon(?:gitud)?\s*[:=]?\s*)?(-?\d{2,3}\.\d{3,})|"
                  r"(-?\d{1,2}\.\d{3,}),\s*(-?\d{2,3}\.\d{3,})", re.I)
_DMS = re.compile(r"(\d{1,2})\s*°\s*(\d{1,2})\s*['′]\s*(\d{1,2}(?:[.,]\d+)?)\s*(?:\"|″|'')?\s*([NS])\s*[,;]?\s*"
                  r"(\d{2,3})\s*°\s*(\d{1,2})\s*['′]\s*(\d{1,2}(?:[.,]\d+)?)\s*(?:\"|″|'')?\s*([EWO])", re.I)
_DATUM = [("PSAD56", re.compile(r"psad\s*-?\s*56|psad\b", re.I)),
          ("WGS84", re.compile(r"wgs\s*-?\s*84|wgs\b", re.I)),
          ("SIRGAS", re.compile(r"sirgas", re.I))]


@dataclass
class Coordenada:
    punto: Punto
    texto: str
    inicio: int
    fin: int
    avisos: list[str] = field(default_factory=list)


def extraer_datum(texto: str) -> str | None:
    """Devuelve el datum mencionado. Si hay dos, toma el primero (origen) — el destino lo decide el diálogo."""
    hallados = sorted(((m.start(), nombre) for nombre, rx in _DATUM for m in rx.finditer(texto)))
    return hallados[0][1] if hallados else None


def extraer_coordenada(texto: str) -> Coordenada | None:
    datum = extraer_datum(texto)
    m = _UTM_ETIQ.search(texto)
    if m:
        return Coordenada(Punto(_num(m.group(1)), _num(m.group(2)), "utm", datum), m.group(0), m.start(), m.end())
    m = _DMS.search(texto)
    if m:
        lat = int(m.group(1)) + int(m.group(2)) / 60 + _num(m.group(3)) / 3600
        lon = int(m.group(5)) + int(m.group(6)) / 60 + _num(m.group(7)) / 3600
        lat = -lat if m.group(4).upper() == "S" else lat
        lon = -lon if m.group(8).upper() in ("W", "O") else lon
        return Coordenada(Punto(lon, lat, "geo", datum), m.group(0), m.start(), m.end())
    m = _DEC.search(texto)
    if m:
        a, b = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
        lat, lon = _num(a), _num(b)
        avisos = []
        if lon > 0 and 75 <= lon <= 92:
            lon = -lon
            avisos.append("Longitud sin signo: se asumió Oeste.")
        return Coordenada(Punto(lon, lat, "geo", datum), m.group(0), m.start(), m.end(), avisos)
    m = _UTM_PAR.search(texto)
    if m:
        e, n = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
        return Coordenada(Punto(_num(e), _num(n), "utm", datum), m.group(0), m.start(), m.end())
    return None


# ------------------------------------------------------------------ tickets, concesiones, fechas, productos

_TICKET = re.compile(r"\bgeo[-\s]?(\d{4})[-\s]?(\d{4})\b", re.I)
_CODIGO = re.compile(r"\b(cm|ct)\s*-?\s*(\d{3})\b", re.I)


def extraer_ticket(texto: str) -> str | None:
    m = _TICKET.search(texto)
    return f"GEO-{m.group(1)}-{m.group(2)}" if m else None


def extraer_concesion(texto: str, nombres: dict[str, str]) -> str | None:
    """nombres: {nombre_plano: codigo}. Acepta código (CM-101) o nombre con tolerancia a erratas."""
    m = _CODIGO.search(texto)
    if m:
        return f"{m.group(1).upper()}-{m.group(2)}"
    t = plano(texto)
    for nombre, codigo in nombres.items():
        if re.search(rf"\b{re.escape(nombre)}\b", t):
            return codigo
    palabras = re.findall(r"[a-zñ]+", t)
    mejor, puntaje = None, 0.0
    for nombre, codigo in nombres.items():
        k = len(nombre.split())
        for i in range(len(palabras) - k + 1):
            r = difflib.SequenceMatcher(None, " ".join(palabras[i:i + k]), nombre).ratio()
            if r > puntaje:
                mejor, puntaje = codigo, r
    return mejor if puntaje >= 0.86 else None


_DIAS = {"lunes": 0, "martes": 1, "miercoles": 2, "jueves": 3, "viernes": 4, "sabado": 5, "domingo": 6}
_MESES = {m: i + 1 for i, m in enumerate(["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
                                           "septiembre", "octubre", "noviembre", "diciembre"])}


def extraer_fecha(texto: str, hoy) -> int | None:
    """Devuelve el desfase en días respecto de 'hoy' (date) o None."""
    import datetime as dt
    t = plano(texto)
    if "pasado mañana" in t or "pasado manana" in t:
        return 2
    if re.search(r"\bmañana\b|\bmanana\b", t):
        return 1
    if re.search(r"\bhoy\b|\bahora\b|\burgente\b", t):
        return 0
    for nombre, wd in _DIAS.items():
        if re.search(rf"\b{nombre}\b", t):
            return (wd - hoy.weekday()) % 7 or 7
    m = re.search(r"\b(\d{1,2})\s+de\s+(" + "|".join(_MESES) + r")\b", t)
    if m:
        try:
            f = dt.date(hoy.year, _MESES[m.group(2)], int(m.group(1)))
            return (f - hoy).days
        except ValueError:
            return None
    m = re.search(r"\b(\d{1,2})/(\d{1,2})(?:/(\d{2,4}))?\b", t)
    if m:
        try:
            f = dt.date(hoy.year, int(m.group(2)), int(m.group(1)))
            return (f - hoy).days
        except ValueError:
            return None
    if "esta semana" in t:
        return 1
    return None


_PRODUCTOS = [("ortofoto", r"ortofoto|orto\b"), ("modelo digital de elevación", r"modelo digital|\bmde\b|\bmdt\b|\bdem\b|curvas de nivel"),
              ("KMZ/KML", r"\bkmz\b|\bkml\b"), ("shapefile", r"shape|shp\b|geopackage|\bgpkg\b|capa"),
              ("plano topográfico", r"topograf|plano"), ("mapa temático", r"tematic|cobertura|pendiente"),
              ("mapa de ubicación", r"mapa")]


def extraer_producto(texto: str) -> str | None:
    t = plano(texto)
    for nombre, rx in _PRODUCTOS:
        if re.search(rx, t):
            return nombre
    return None


# ------------------------------------------------------------------ evidencias de alerta (para la red bayesiana)

EVIDENCIAS = {
    "M": ("maquinaria pesada o equipos", r"maquinaria|excavadora|retroexcavadora|volqueta|clasificadora|draga|bomba|"
                                         r"motobomba|manguera|tractor|perforadora|maquina"),
    "S": ("alteración del terreno o la cobertura", r"desbroce|deforest|\btala\b|claro nuevo|cambio de cobertura|"
                                                    r"movimiento de tierra|hueco|\bexcava(?!dor)|labores|bocamina|socavon|imagen|sentinel|satelit"),
    "P": ("personas o campamento no identificados", r"gente|personas|mineros|informales|campamento|carpas"),
    "R": ("ruido o voladuras nocturnas", r"ruido|voladura|explosion|dinamita|de noche|madrugada|nocturn"),
    "T": ("agua turbia aguas abajo", r"turbi|lodo|agua sucia|sediment"),
    "V": ("vía o trocha nueva", r"trocha|camino nuevo|camino abierto|aparecio un camino|via nueva|acceso nuevo|sendero nuevo"),
}
_NEGACION = re.compile(r"\b(no|sin|ningun|ninguna|ninguno|nada de|tampoco|ni)\b")


def extraer_evidencias(texto: str) -> dict[str, bool]:
    """Evidencias observadas: True si se afirma, False si se niega explícitamente. Ausentes = no observadas."""
    t = plano(texto)
    salida: dict[str, bool] = {}
    for clave, (_, rx) in EVIDENCIAS.items():
        for m in re.finditer(rx, t):
            previo = " ".join(t[:m.start()].split()[-3:])
            salida[clave] = not bool(_NEGACION.search(previo))
            break
    return salida


# ------------------------------------------------------------------ seguridad y datos personales

_EMERGENCIA = re.compile(r"armad[oa]s?\b|\barmas\b|disparo|dispararon|\btiros\b|balacera|herid[oa]|lesionad|secuestr|"
                         r"retuvieron|retenid|amenaz|atrapad|incendio|agred|agresi|asalt|desaparecid|violencia|"
                         r"enfrentamiento|atacaron|auxilio|encapuchad")


def hay_emergencia(texto: str) -> bool:
    return bool(_EMERGENCIA.search(plano(texto)))


def cedula_valida(d: str) -> bool:
    """Validación del dígito verificador de la cédula ecuatoriana (módulo 10)."""
    if len(d) != 10 or not d.isdigit():
        return False
    prov, tercero = int(d[:2]), int(d[2])
    if not (1 <= prov <= 24 or prov == 30) or tercero >= 6:
        return False
    suma = 0
    for i, c in enumerate(d[:9]):
        v = int(c) * (2 if i % 2 == 0 else 1)
        suma += v - 9 if v > 9 else v
    return (10 - suma % 10) % 10 == int(d[9])


_DIEZ = re.compile(r"(?<!\d)(\d{10})(?!\d)")
_TELEFONO = re.compile(r"(?<!\d)(?:\+?593\s?|0)9\d{8}(?!\d)")
_CORREO = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")


def enmascarar_pii(texto: str) -> tuple[str, list[str]]:
    tipos: list[str] = []

    def _ced(m):
        if cedula_valida(m.group(1)):
            tipos.append("cédula")
            return "[CÉDULA]"
        return m.group(0)

    t = _DIEZ.sub(_ced, texto)
    t = _TELEFONO.sub(lambda m: (tipos.append("teléfono"), "[TELÉFONO]")[1], t)
    t = _CORREO.sub(lambda m: (tipos.append("correo"), "[CORREO]")[1], t)
    return t, tipos


# ------------------------------------------------------------------ enmascarado para el clasificador


def enmascarar_entidades(texto: str) -> str:
    """Sustituye valores por su tipo antes de clasificar la intención (mismo paso para todos los modelos)."""
    t = texto
    c = extraer_coordenada(t)
    while c is not None:
        t = t[:c.inicio] + " COORDENADA " + t[c.fin:]
        c = extraer_coordenada(t)
    t = _TICKET.sub(" TICKET ", t)
    t = _CODIGO.sub(" CODIGO ", t)
    return re.sub(r"\s+", " ", t).strip()


In [ ]:
%%writefile kuribot/conocimiento.py
"""Base de conocimiento de KuriBot: catastro sintético, operación, FAQ y tickets (SQLite)."""
from __future__ import annotations

import datetime as dt
import json
import sqlite3
from dataclasses import dataclass
from pathlib import Path

from shapely.geometry import Point, shape
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from .entidades import plano

RAIZ = Path(__file__).resolve().parents[1]
DATOS = RAIZ / "datos"
FRANJA_M = 500.0      # franja de amortiguamiento (supuesto de diseño heredado de SEPRIM)
RADIO_COMUNIDAD_M = 1000.0


@dataclass
class AnalisisPunto:
    e: float
    n: float
    ubicacion: str                 # propia | amortiguamiento | tercero | fuera
    concesion: str | None          # código de la concesión que contiene el punto
    cercana: str | None            # concesión propia más cercana
    distancia_lindero_m: float     # distancia al lindero de la concesión que lo contiene o de la más cercana
    area_protegida: str | None
    comunidad: str | None
    distancia_comunidad_m: float | None


class BaseConocimiento:
    def __init__(self, ruta_datos: Path = DATOS, db: str = ":memory:"):
        geo = json.loads((ruta_datos / "concesiones.geojson").read_text(encoding="utf-8"))
        self.concesiones, self.areas, self.comunidades = {}, {}, {}
        for f in geo["features"]:
            p, g = f["properties"], shape(f["geometry"])
            if p["capa"] == "concesion":
                self.concesiones[p["codigo"]] = {**p, "geom": g, "area_ha": round(g.area / 1e4, 1)}
            elif p["capa"] == "area_protegida":
                self.areas[p["codigo"]] = {**p, "geom": g}
            else:
                self.comunidades[p["codigo"]] = {**p, "geom": g}
        self.op = json.loads((ruta_datos / "operacion.json").read_text(encoding="utf-8"))
        self.hoy = dt.date.fromisoformat(self.op["fecha_referencia"])
        self.faq = json.loads((ruta_datos / "faq.json").read_text(encoding="utf-8"))
        self.nombres = {plano(c["nombre"]): cod for cod, c in self.concesiones.items()}
        self._indexar_faq()
        self.db = sqlite3.connect(db, check_same_thread=False)
        self._crear_tickets(json.loads((ruta_datos / "tickets_semilla.json").read_text(encoding="utf-8")))

    # ------------------------------------------------------------ catastro
    def ficha(self, codigo: str) -> dict | None:
        c = self.concesiones.get(codigo)
        if not c:
            return None
        return {k: c[k] for k in ("codigo", "nombre", "fase", "estado", "campamento", "titular", "area_ha")} | {
            "alertas_90d": self.op["historial_alertas_90d"].get(codigo, 0),
            "actividad_programada": self.op["actividad_autorizada_programada"].get(codigo)}

    def analizar_punto(self, e: float, n: float) -> AnalisisPunto:
        pt = Point(e, n)
        contiene = [c for c in self.concesiones.values() if c["geom"].covers(pt)]
        propias = [c for c in self.concesiones.values() if c["titular"] == "propia"]
        cercana = min(propias, key=lambda c: c["geom"].distance(pt))
        if contiene:
            c = sorted(contiene, key=lambda c: c["titular"] != "propia")[0]
            ubic = "propia" if c["titular"] == "propia" else "tercero"
            cod, dist = c["codigo"], c["geom"].exterior.distance(pt)
        else:
            dist = cercana["geom"].distance(pt)
            ubic, cod = ("amortiguamiento" if dist <= FRANJA_M else "fuera"), None
        area = next((a["nombre"] for a in self.areas.values() if a["geom"].covers(pt)), None)
        com = min(self.comunidades.values(), key=lambda c: c["geom"].distance(pt)) if self.comunidades else None
        dcom = com["geom"].distance(pt) if com else None
        return AnalisisPunto(e, n, ubic, cod, cercana["codigo"], round(dist, 1), area,
                             com["nombre"] if com and dcom <= RADIO_COMUNIDAD_M else None,
                             round(dcom, 1) if dcom is not None else None)

    def campamento(self, codigo: str | None) -> str:
        c = self.concesiones.get(codigo or "")
        return c["campamento"] if c and c["campamento"] in self.op["pronostico"] else "Central"

    # ------------------------------------------------------------ FAQ (recuperación por similitud TF-IDF)
    def _indexar_faq(self):
        self._faq_textos, self._faq_ids = [], []
        for f in self.faq:
            for q in f["preguntas"]:
                self._faq_textos.append(q)
                self._faq_ids.append(f["id"])
        self._vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True, strip_accents="unicode")
        self._mat = self._vec.fit_transform(self._faq_textos)

    def buscar_faq(self, consulta: str) -> tuple[dict | None, float]:
        s = cosine_similarity(self._vec.transform([consulta]), self._mat)[0]
        i = int(s.argmax())
        f = next(x for x in self.faq if x["id"] == self._faq_ids[i])
        return f, float(s[i])

    # ------------------------------------------------------------ tickets
    def _crear_tickets(self, semilla):
        cur = self.db.cursor()
        cur.execute("CREATE TABLE IF NOT EXISTS tickets (id TEXT PRIMARY KEY, tipo TEXT, detalle TEXT, estado TEXT, "
                    "asignado TEXT, compromiso TEXT, resolucion TEXT, prioridad TEXT, creado TEXT)")
        for t in semilla:
            cur.execute("INSERT OR REPLACE INTO tickets VALUES (?,?,?,?,?,?,?,?,?)",
                        (t["id"], t["tipo"], t["detalle"], t["estado"], t["asignado"], t["compromiso"],
                         t.get("resolucion"), t.get("prioridad"), self.op["fecha_referencia"]))
        self.db.commit()

    def ticket(self, ident: str) -> dict | None:
        cur = self.db.execute("SELECT * FROM tickets WHERE id = ?", (ident,))
        fila = cur.fetchone()
        if not fila:
            return None
        return dict(zip([c[0] for c in cur.description], fila))

    def nuevo_ticket(self, tipo: str, detalle: str, asignado: str, compromiso: str | None, prioridad: str | None = None) -> str:
        ultimos = [int(r[0].split("-")[-1]) for r in self.db.execute("SELECT id FROM tickets")]
        ident = f"GEO-2026-{max(ultimos) + 1:04d}"
        self.db.execute("INSERT INTO tickets VALUES (?,?,?,?,?,?,?,?,?)",
                        (ident, tipo, detalle, "Registrado", asignado, compromiso, None, prioridad, self.op["fecha_referencia"]))
        self.db.commit()
        return ident

    def fecha(self, desfase: int) -> str:
        return (self.hoy + dt.timedelta(days=desfase)).isoformat()


In [ ]:
%%writefile kuribot/nlu_modelos.py
"""Modelos de clasificación de intenciones comparados en el proyecto.

M0  TF-IDF (palabras 1-2 + caracteres 2-5) + regresión logística   -> ML clásico (línea base)
M1  Embeddings fastText preentrenados (spaCy es_core_news_lg) + BiLSTM + atención + MLP
M2  BETO congelado (media de la última capa) + MLP
M3  BETO ajustado de extremo a extremo (fine-tuning) + capa lineal   -> modelo de producción

Todos reciben el mismo texto preprocesado (entidades enmascaradas) y las mismas particiones.
"""
from __future__ import annotations

import copy
import math
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import FeatureUnion, Pipeline
from torch import nn

from .entidades import enmascarar_entidades

TOKEN = re.compile(r"[a-záéíóúüñ]+|\d+|[A-ZÁÉÍÓÚÑ]+", re.I)


def fijar_semilla(s: int) -> None:
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)


def preprocesar(textos: list[str]) -> list[str]:
    return [enmascarar_entidades(t) for t in textos]


def pesos_clase(y: np.ndarray, n: int) -> torch.Tensor:
    conteo = np.bincount(y, minlength=n).astype(float)
    w = conteo.sum() / (n * np.maximum(conteo, 1))
    return torch.tensor(w, dtype=torch.float32)


# ====================================================================== M0


def crear_m0(C: float = 10.0) -> Pipeline:
    feats = FeatureUnion([
        ("palabras", TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), sublinear_tf=True)),
        ("caracteres", TfidfVectorizer(lowercase=True, strip_accents="unicode", analyzer="char_wb",
                                       ngram_range=(2, 5), sublinear_tf=True)),
    ])
    return Pipeline([("tfidf", feats), ("lr", LogisticRegression(C=C, max_iter=4000, class_weight="balanced"))])


class ModeloM0:
    nombre = "M0 TF-IDF + RL"

    def __init__(self, n_clases: int, semilla: int = 0):
        self.n = n_clases
        self.pipe = crear_m0()

    def entrenar(self, xtr, ytr, xdev=None, ydev=None):
        self.pipe.fit(preprocesar(xtr), ytr)
        return {}

    def logits(self, textos):
        # log-probabilidades como logits: el softmax recupera las probabilidades de la RL
        return np.log(np.clip(self.pipe.predict_proba(preprocesar(textos)), 1e-12, 1))


# ====================================================================== utilidades de entrenamiento neuronal


def _entrenar_torch(modelo, lotes_tr, evaluar_dev, epocas, lr, pesos, paciencia, wd=0.01, calentamiento=0.0,
                    params=None):
    params = params if params is not None else [p for p in modelo.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    pasos_total = epocas * len(lotes_tr())
    pasos_cal = int(calentamiento * pasos_total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, pasos_cal) if s < pasos_cal else max(0.0, (pasos_total - s) / max(1, pasos_total - pasos_cal)))
    perdida = nn.CrossEntropyLoss(weight=pesos, label_smoothing=0.05)
    mejor, mejor_estado, sin_mejora, historia = -1.0, None, 0, []
    for ep in range(epocas):
        modelo.train()
        total, n = 0.0, 0
        for entrada, y in lotes_tr():
            opt.zero_grad()
            salida = modelo(*entrada)
            l = perdida(salida, y)
            l.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            sched.step()
            total += float(l.detach()) * len(y)
            n += len(y)
        f1 = evaluar_dev() if evaluar_dev else float("nan")
        historia.append({"epoca": ep + 1, "perdida_entrenamiento": total / n, "f1_macro_validacion": f1})
        if evaluar_dev is None:
            continue
        if f1 > mejor + 1e-4:
            mejor, mejor_estado, sin_mejora = f1, copy.deepcopy(modelo.state_dict()), 0
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia:
                break
    if mejor_estado is not None:
        modelo.load_state_dict(mejor_estado)
    return historia


# ====================================================================== M1


def tokenizar(texto: str) -> list[str]:
    return [t.lower() for t in TOKEN.findall(texto)]


class RedBiLSTM(nn.Module):
    def __init__(self, emb: np.ndarray, n_clases: int, oculto: int = 128, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(emb, dtype=torch.float32), freeze=False, padding_idx=0)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb.shape[1], oculto, batch_first=True, bidirectional=True)
        self.atencion = nn.Linear(2 * oculto, 1)
        self.mlp = nn.Sequential(nn.Linear(4 * oculto, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, n_clases))

    def forward(self, ids, mascara):
        h, _ = self.lstm(self.drop(self.embedding(ids)))
        puntaje = self.atencion(h).squeeze(-1).masked_fill(~mascara, -1e9)
        alfa = torch.softmax(puntaje, dim=1).unsqueeze(-1)
        atendido = (alfa * h).sum(1)
        maximo = h.masked_fill(~mascara.unsqueeze(-1), -1e9).max(1).values
        return self.mlp(torch.cat([atendido, maximo], dim=1))


class ModeloM1:
    nombre = "M1 fastText + BiLSTM"

    def __init__(self, n_clases: int, semilla: int = 0, vectores=None, max_len: int = 32):
        self.n, self.semilla, self.max_len = n_clases, semilla, max_len
        self.vectores = vectores  # función token -> vector (np.ndarray) o None
        self.vocab: dict[str, int] = {}
        self.red: RedBiLSTM | None = None
        self.cobertura = float("nan")

    def _ids(self, textos):
        filas = [[self.vocab.get(t, 1) for t in tokenizar(x)][: self.max_len] or [1] for x in preprocesar(textos)]
        L = max(len(f) for f in filas)
        ids = torch.zeros(len(filas), L, dtype=torch.long)
        for i, f in enumerate(filas):
            ids[i, : len(f)] = torch.tensor(f)
        return ids, ids != 0

    def entrenar(self, xtr, ytr, xdev, ydev):
        fijar_semilla(self.semilla)
        toks = sorted({t for x in preprocesar(xtr) for t in tokenizar(x)})
        self.vocab = {"<pad>": 0, "<unk>": 1, **{t: i + 2 for i, t in enumerate(toks)}}
        emb = np.random.default_rng(self.semilla).normal(0, 0.1, (len(self.vocab), 300)).astype(np.float32)
        emb[0] = 0
        cubiertos = 0
        if self.vectores is not None:
            for t, i in self.vocab.items():
                v = self.vectores(t) if i > 1 else None
                if v is not None:
                    emb[i], cubiertos = v, cubiertos + 1
        self.cobertura = cubiertos / max(1, len(self.vocab) - 2)
        self.red = RedBiLSTM(emb, self.n)
        ytr_t = torch.tensor(ytr)
        idx = np.arange(len(xtr))

        epoca = [0]

        def lotes_lista():
            rng = np.random.default_rng(self.semilla + epoca[0])
            epoca[0] += 1
            orden = rng.permutation(idx)
            salida = []
            for i in range(0, len(orden), 32):
                b = orden[i:i + 32]
                ids, m = self._ids([xtr[j] for j in b])
                salida.append(((ids, m), ytr_t[b]))
            return salida

        def dev():
            return f1_score(ydev, self.logits(xdev).argmax(1), average="macro")

        hist = _entrenar_torch(self.red, lotes_lista, dev, epocas=40, lr=2e-3, pesos=pesos_clase(ytr, self.n), paciencia=6)
        return {"historia": hist, "cobertura_embeddings": self.cobertura, "vocabulario": len(self.vocab)}

    @torch.no_grad()
    def logits(self, textos):
        self.red.eval()
        salida = []
        for i in range(0, len(textos), 64):
            ids, m = self._ids(textos[i:i + 64])
            salida.append(self.red(ids, m).numpy())
        return np.concatenate(salida)


# ====================================================================== BETO (M2 y M3)


class CodificadorBETO(nn.Module):
    def __init__(self, ruta: str | Path | None = None, config=None):
        super().__init__()
        from transformers import BertModel
        # Con 'config' se crea la arquitectura vacía (los pesos se cargan después desde el modelo exportado).
        self.bert = (BertModel(config, add_pooling_layer=False) if config is not None
                     else BertModel.from_pretrained(str(ruta), add_pooling_layer=False))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        h = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp_min(1.0)


class ClasificadorBETO(nn.Module):
    def __init__(self, ruta, n_clases: int, dropout: float = 0.1, config=None):
        super().__init__()
        self.cod = CodificadorBETO(ruta, config)
        self.drop = nn.Dropout(dropout)
        self.salida = nn.Linear(768, n_clases)

    def forward(self, input_ids, attention_mask):
        return self.salida(self.drop(self.cod(input_ids, attention_mask)))


def _tok(ruta):
    from transformers import BertTokenizerFast
    return BertTokenizerFast.from_pretrained(str(ruta))


class ModeloM2:
    nombre = "M2 BETO congelado + MLP"
    _cache: dict = {}

    def __init__(self, n_clases: int, semilla: int = 0, ruta_beto: str = "modelos/beto_base"):
        self.n, self.semilla, self.ruta = n_clases, semilla, ruta_beto
        self.tok = _tok(ruta_beto)
        if ruta_beto not in ModeloM2._cache:
            ModeloM2._cache[ruta_beto] = (CodificadorBETO(ruta_beto).eval(), {})
        self.cod, self.memo = ModeloM2._cache[ruta_beto]
        self.mlp = None

    @torch.no_grad()
    def _vec(self, textos):
        pre = preprocesar(textos)
        faltan = [t for t in dict.fromkeys(pre) if t not in self.memo]
        for i in range(0, len(faltan), 32):
            enc = self.tok(faltan[i:i + 32], padding=True, truncation=True, max_length=48, return_tensors="pt")
            v = self.cod(enc["input_ids"], enc["attention_mask"])
            for t, fila in zip(faltan[i:i + 32], v):
                self.memo[t] = fila
        return torch.stack([self.memo[t] for t in pre])

    def entrenar(self, xtr, ytr, xdev, ydev):
        fijar_semilla(self.semilla)
        Xtr = self._vec(xtr)
        # Estandarización con media y desviación de entrenamiento: las salidas de BERT son muy anisótropas
        # (coseno medio ≈ 0,9 entre frases), lo que impide aprender a una MLP sin normalizar.
        self.mu, self.sd = Xtr.mean(0), Xtr.std(0).clamp_min(1e-4)
        Xtr = (Xtr - self.mu) / self.sd
        Xdev = (self._vec(xdev) - self.mu) / self.sd
        self.mlp = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, self.n))
        ytr_t = torch.tensor(ytr)

        def lotes():
            orden = torch.randperm(len(ytr))
            return [((Xtr[orden[i:i + 32]],), ytr_t[orden[i:i + 32]]) for i in range(0, len(ytr), 32)]

        def dev():
            self.mlp.eval()
            with torch.no_grad():
                return f1_score(ydev, self.mlp(Xdev).argmax(1).numpy(), average="macro")

        hist = _entrenar_torch(self.mlp, lotes, dev, epocas=150, lr=1e-3, pesos=pesos_clase(ytr, self.n), paciencia=15)
        return {"historia": hist}

    @torch.no_grad()
    def logits(self, textos):
        self.mlp.eval()
        return self.mlp((self._vec(textos) - self.mu) / self.sd).numpy()


class ModeloM3:
    nombre = "M3 BETO ajustado (fine-tuning)"

    def __init__(self, n_clases: int, semilla: int = 0, ruta_beto: str = "modelos/beto_base", epocas: int = 6,
                 lr: float = 3e-5, lote: int = 16, paciencia: int = 2):
        self.n, self.semilla, self.ruta = n_clases, semilla, ruta_beto
        self.epocas, self.lr, self.lote, self.paciencia = epocas, lr, lote, paciencia
        self.tok = _tok(ruta_beto)
        self.red = None

    def _enc(self, textos):
        return self.tok(preprocesar(textos), padding=True, truncation=True, max_length=48, return_tensors="pt")

    def entrenar(self, xtr, ytr, xdev, ydev):
        fijar_semilla(self.semilla)
        self.red = ClasificadorBETO(self.ruta, self.n)
        ytr_t = torch.tensor(ytr)

        def lotes():
            orden = torch.randperm(len(ytr)).tolist()
            out = []
            for i in range(0, len(orden), self.lote):
                b = orden[i:i + self.lote]
                enc = self._enc([xtr[j] for j in b])
                out.append(((enc["input_ids"], enc["attention_mask"]), ytr_t[b]))
            return out

        def dev():
            return f1_score(ydev, self.logits(xdev).argmax(1), average="macro") if xdev is not None else float("nan")

        hist = _entrenar_torch(self.red, lotes, dev if xdev is not None else None, epocas=self.epocas, lr=self.lr,
                               pesos=pesos_clase(ytr, self.n), paciencia=self.paciencia, wd=0.01, calentamiento=0.1)
        return {"historia": hist}

    @torch.no_grad()
    def logits(self, textos):
        self.red.eval()
        salida = []
        for i in range(0, len(textos), 32):
            enc = self._enc(textos[i:i + 32])
            salida.append(self.red(enc["input_ids"], enc["attention_mask"]).numpy())
        return np.concatenate(salida)


# ====================================================================== calibración y métricas


def softmax(z: np.ndarray, T: float = 1.0) -> np.ndarray:
    z = z / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def ajustar_temperatura(logits_dev: np.ndarray, y_dev: np.ndarray) -> float:
    """Escalado de temperatura (Guo et al., 2017): minimiza la log-verosimilitud negativa en validación."""
    z = torch.tensor(logits_dev, dtype=torch.float64)
    y = torch.tensor(y_dev)
    logT = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200)

    def cierre():
        opt.zero_grad()
        l = nn.functional.cross_entropy(z / logT.exp(), y)
        l.backward()
        return l

    opt.step(cierre)
    return float(logT.detach().exp().clamp(0.05, 20))


def ece(probs: np.ndarray, y: np.ndarray, bins: int = 10) -> float:
    conf, pred = probs.max(1), probs.argmax(1)
    bordes = np.linspace(0, 1, bins + 1)
    total = 0.0
    for a, b in zip(bordes[:-1], bordes[1:]):
        m = (conf > a) & (conf <= b)
        if m.any():
            total += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(total)


def latencia_ms(modelo, textos: list[str], repeticiones: int = 1) -> dict:
    tiempos = []
    for _ in range(repeticiones):
        for t in textos:
            t0 = time.perf_counter()
            modelo.logits([t])
            tiempos.append((time.perf_counter() - t0) * 1000)
    a = np.array(tiempos)
    return {"media_ms": float(a.mean()), "p50_ms": float(np.percentile(a, 50)), "p95_ms": float(np.percentile(a, 95))}


def n_parametros(modelo) -> int:
    if isinstance(modelo, ModeloM0):
        lr = modelo.pipe.named_steps["lr"]
        return int(lr.coef_.size + lr.intercept_.size)
    if isinstance(modelo, ModeloM1):
        return sum(p.numel() for p in modelo.red.parameters())
    if isinstance(modelo, ModeloM2):
        return sum(p.numel() for p in modelo.mlp.parameters())
    if isinstance(modelo, ModeloM3):
        return sum(p.numel() for p in modelo.red.parameters())
    return -1


def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (math.nan, math.nan)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    r = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - r) / d, (c + r) / d)


In [ ]:
%%writefile kuribot/nlu.py
"""Inferencia de intenciones para el ChatBot (carga el modelo exportado por experimentos/entrenar_nlp.py)."""
from __future__ import annotations

import json
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from .entidades import enmascarar_entidades

RAIZ = Path(__file__).resolve().parents[1]
MODELOS = RAIZ / "modelos"


@dataclass
class Prediccion:
    intencion: str
    confianza: float
    top: list[tuple[str, float]]
    decision: str          # aceptar | aclarar | abstener
    modelo: str
    latencia_ms: float


def _softmax(z: np.ndarray, T: float) -> np.ndarray:
    z = z / T
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


class ClasificadorIntenciones:
    """Envuelve el BETO ajustado (preferido) o la línea base TF-IDF si no hay pesos disponibles."""

    def __init__(self, ruta: str | Path | None = None, umbral_aclarar: float | None = None, umbral_abstener: float = 0.30):
        ruta = Path(ruta) if ruta else self._ultimo("intenciones_beto_v*")
        if ruta is not None and (ruta / "clasificador.safetensors").exists():
            self._cargar_beto(ruta)
        else:
            base = self._ultimo("intenciones_tfidf_v*.joblib")
            if base is None:
                raise FileNotFoundError("No hay modelos entrenados en modelos/. Ejecute experimentos/entrenar_nlp.py")
            self._cargar_tfidf(base)
        self.umbral_aclarar = umbral_aclarar if umbral_aclarar is not None else max(0.5, self.tau)
        self.umbral_abstener = umbral_abstener
        self._logits("hola")  # calentamiento: la primera inferencia de PyTorch tras cargar el modelo es lenta

    @staticmethod
    def _ultimo(patron: str) -> Path | None:
        c = sorted(MODELOS.glob(patron))
        return c[-1] if c else None

    def _cargar_beto(self, ruta: Path):
        import torch
        from safetensors.torch import load_file
        from transformers import BertConfig, BertTokenizerFast

        from .nlu_modelos import ClasificadorBETO

        meta = json.loads((ruta / "kuribot_nlu.json").read_text(encoding="utf-8"))
        self.etiquetas, self.T, self.tau = meta["etiquetas"], meta["temperatura"], meta["umbral_confianza"]
        self.max_len = meta.get("max_len", 48)
        cfg = BertConfig.from_json_file(str(ruta / "bert_config.json"))
        red = ClasificadorBETO(None, len(self.etiquetas), config=cfg)
        estado = {k: v.float() for k, v in load_file(str(ruta / "clasificador.safetensors")).items()}
        red.load_state_dict(estado)
        red.eval()
        self.red, self.tok, self.torch = red, BertTokenizerFast.from_pretrained(str(ruta)), torch
        self.nombre = f"BETO ajustado ({ruta.name})"
        self._logits = self._logits_beto

    def _cargar_tfidf(self, ruta: Path):
        import joblib
        d = joblib.load(ruta)
        self.pipe, self.etiquetas, self.T, self.tau = d["pipe"], d["etiquetas"], d["temperatura"], 0.5
        self.nombre = f"TF-IDF + RL ({ruta.name}, modo de respaldo)"
        self._logits = lambda t: np.log(np.clip(self.pipe.predict_proba([t])[0], 1e-12, 1))

    def _logits_beto(self, texto: str) -> np.ndarray:
        with self.torch.no_grad():
            enc = self.tok([texto], truncation=True, max_length=self.max_len, return_tensors="pt")
            return self.red(enc["input_ids"], enc["attention_mask"])[0].numpy()

    def predecir(self, texto: str) -> Prediccion:
        t0 = time.perf_counter()
        p = _softmax(self._logits(enmascarar_entidades(texto)), self.T)
        orden = np.argsort(-p)
        top = [(self.etiquetas[i], float(p[i])) for i in orden[:3]]
        conf = top[0][1]
        if conf >= self.umbral_aclarar:
            decision = "aceptar"
        elif conf >= self.umbral_abstener and top[0][1] + top[1][1] >= 0.70:
            decision = "aclarar"
        else:
            decision = "abstener"
        return Prediccion(top[0][0], conf, top, decision, self.nombre, (time.perf_counter() - t0) * 1000)


In [ ]:
%%writefile kuribot/razonamiento.py
"""Componente de razonamiento de KuriBot (contenidos de las semanas 2 y 3).

1. MotorReglas: sistema de producción con encadenamiento hacia adelante, agenda por nivel y
   prioridad, refracción (cada regla dispara una vez) y traza explicativa. Es la continuación
   del sistema experto SEPRIM de la semana 2, ahora alimentado desde la conversación.
2. RedAlertas: red bayesiana discreta (9 nodos) que estima P(intrusión real | evidencias) con
   eliminación de variables exacta (pgmpy). Las tablas de probabilidad condicional (TPC) son
   juicios expertos del equipo, no frecuencias estimadas: deben recalibrarse con el histórico
   de verificaciones de campo.
3. Factibilidad de vuelo: reglas operativas + búsqueda de la primera ventana apta.
"""
from __future__ import annotations

import itertools
from dataclasses import dataclass, field
from typing import Callable

import numpy as np

# ====================================================================== motor de reglas


@dataclass
class Regla:
    id: str
    nivel: int
    prioridad: int
    descripcion: str
    si: Callable[[dict], bool]
    entonces: Callable[[dict], dict]
    texto_si: str = ""
    texto_entonces: str = ""


@dataclass
class ResultadoInferencia:
    hechos: dict
    disparadas: list[str] = field(default_factory=list)
    traza: list[str] = field(default_factory=list)


class MotorReglas:
    def __init__(self, reglas: list[Regla]):
        ids = [r.id for r in reglas]
        if len(ids) != len(set(ids)):
            raise ValueError("Identificadores de regla duplicados")
        self.reglas = sorted(reglas, key=lambda r: (r.nivel, -r.prioridad, r.id))

    def inferir(self, hechos: dict, max_ciclos: int = 100) -> ResultadoInferencia:
        wm = dict(hechos)
        res = ResultadoInferencia(wm)
        disparadas: set[str] = set()
        for _ in range(max_ciclos):
            agenda = [r for r in self.reglas if r.id not in disparadas and _seguro(r.si, wm)]
            if not agenda:
                break
            r = agenda[0]  # resolución de conflictos: menor nivel, mayor prioridad, id
            nuevos = r.entonces(wm)
            wm.update(nuevos)
            disparadas.add(r.id)
            res.disparadas.append(r.id)
            res.traza.append(f"{r.id} (nivel {r.nivel}): {r.descripcion}")
        res.hechos = wm
        return res


def _seguro(cond, wm) -> bool:
    try:
        return bool(cond(wm))
    except (KeyError, TypeError):
        return False


def _sin_prioridad(f):
    return "prioridad" not in f


UMBRALES = {"critica": 0.80, "alta": 0.50, "media": 0.15}  # supuestos de diseño (ver informe, sección 5.4)

REGLAS_ALERTA: list[Regla] = [
    # ---------------- nivel 0: seguridad, privacidad, autonomía
    Regla("R01", 0, 100, "Riesgo para personas: protocolo de emergencia, sin despliegue de personal ni dron",
          lambda f: f.get("emergencia"), lambda f: {"prioridad": "CRÍTICA", "accion": "PROTOCOLO DE EMERGENCIA",
                                                     "prohibir_despliegue": True, "derivar_humano": True},
          "emergencia = verdadero", "prioridad = CRÍTICA; acción = protocolo; derivar a humano"),
    Regla("R02", 0, 90, "Datos personales detectados: se enmascaran en el registro",
          lambda f: f.get("pii"), lambda f: {"enmascarar_registro": True}, "pii = verdadero", "enmascarar registro"),
    Regla("R03", 0, 80, "Solicitud explícita de atención humana",
          lambda f: f.get("intencion") == "hablar_humano", lambda f: {"derivar_humano": True},
          "intención = hablar_humano", "derivar a humano"),
    # ---------------- nivel 1: contexto geoespacial
    Regla("R10", 1, 50, "El punto está dentro de una concesión propia",
          lambda f: f.get("ubicacion") == "propia", lambda f: {"zona": "concesión propia"}, "ubicación = propia", "zona = concesión propia"),
    Regla("R11", 1, 50, "El punto está en la franja de 500 m alrededor de una concesión propia",
          lambda f: f.get("ubicacion") == "amortiguamiento", lambda f: {"zona": "franja de amortiguamiento"},
          "ubicación = amortiguamiento", "zona = amortiguamiento"),
    Regla("R12", 1, 60, "El punto está en una concesión de un tercero con derecho vigente",
          lambda f: f.get("ubicacion") == "tercero", lambda f: {"derecho_tercero": True}, "ubicación = tercero", "derecho_tercero"),
    Regla("R13", 1, 40, "El punto está dentro de un área protegida",
          lambda f: f.get("area_protegida"), lambda f: {"area_sensible": True}, "área protegida", "área sensible"),
    Regla("R14", 1, 40, "Hay una comunidad a 1 km o menos",
          lambda f: f.get("comunidad_cercana"), lambda f: {"sensibilidad_social": True}, "comunidad ≤ 1 km", "sensibilidad social"),
    Regla("R15", 1, 30, "El punto está fuera del área de responsabilidad de la empresa",
          lambda f: f.get("ubicacion") == "fuera", lambda f: {"fuera_jurisdiccion": True}, "ubicación = fuera", "fuera de jurisdicción"),
    # ---------------- nivel 2: priorización con el posterior bayesiano
    Regla("R20", 2, 100, "Derecho vigente de un tercero: no se califica como intrusión; revisión legal",
          lambda f: _sin_prioridad(f) and f.get("derecho_tercero"), lambda f: {"prioridad": "REVISIÓN LEGAL"},
          "derecho_tercero y sin prioridad", "prioridad = REVISIÓN LEGAL"),
    Regla("R25", 2, 90, "Fuera del área de la empresa: registro informativo",
          lambda f: _sin_prioridad(f) and f.get("fuera_jurisdiccion"), lambda f: {"prioridad": "INFORMATIVA"},
          "fuera_jurisdicción y sin prioridad", "prioridad = INFORMATIVA"),
    Regla("R21", 2, 80, "P(intrusión) ≥ 0,80",
          lambda f: _sin_prioridad(f) and f["p_intrusion"] >= UMBRALES["critica"], lambda f: {"prioridad": "CRÍTICA"},
          "P ≥ 0,80 y sin prioridad", "prioridad = CRÍTICA"),
    Regla("R22", 2, 70, "0,50 ≤ P(intrusión) < 0,80",
          lambda f: _sin_prioridad(f) and f["p_intrusion"] >= UMBRALES["alta"], lambda f: {"prioridad": "ALTA"},
          "P ≥ 0,50 y sin prioridad", "prioridad = ALTA"),
    Regla("R23", 2, 60, "0,15 ≤ P(intrusión) < 0,50",
          lambda f: _sin_prioridad(f) and f["p_intrusion"] >= UMBRALES["media"], lambda f: {"prioridad": "MEDIA"},
          "P ≥ 0,15 y sin prioridad", "prioridad = MEDIA"),
    Regla("R24", 2, 50, "P(intrusión) < 0,15",
          lambda f: _sin_prioridad(f) and "p_intrusion" in f, lambda f: {"prioridad": "BAJA"},
          "sin prioridad (regla por defecto)", "prioridad = BAJA"),
    # ---------------- nivel 3: acción operativa
    Regla("R30", 3, 60, "CRÍTICA con ventana de vuelo hoy o mañana: vuelo prioritario y aviso a Seguridad y Legal",
          lambda f: f.get("prioridad") == "CRÍTICA" and not f.get("emergencia") and f.get("vuelo_dias") is not None and f["vuelo_dias"] <= 1,
          lambda f: {"accion": "Vuelo de verificación prioritario (≤ 24 h) y notificación a Seguridad y Legal"},
          "CRÍTICA y ventana ≤ 1 día", "vuelo prioritario"),
    Regla("R31", 3, 55, "CRÍTICA sin ventana de vuelo inmediata: verificación satelital y patrullaje con seguridad",
          lambda f: f.get("prioridad") == "CRÍTICA" and not f.get("emergencia") and "accion" not in f,
          lambda f: {"accion": "Revisión satelital inmediata y patrullaje con Seguridad; vuelo en la primera ventana apta"},
          "CRÍTICA y sin ventana ≤ 1 día", "satélite + patrullaje"),
    Regla("R32", 3, 50, "ALTA: vuelo en la primera ventana apta (≤ 72 h)",
          lambda f: f.get("prioridad") == "ALTA", lambda f: {"accion": "Programar vuelo de verificación en la primera ventana apta (≤ 72 h)"},
          "ALTA", "vuelo ≤ 72 h"),
    Regla("R33", 3, 40, "MEDIA: revisión satelital y verificación en ronda programada",
          lambda f: f.get("prioridad") == "MEDIA", lambda f: {"accion": "Revisión satelital en el próximo ciclo y verificación en la ronda programada"},
          "MEDIA", "revisión satelital"),
    Regla("R34", 3, 30, "BAJA: registrar y monitorear",
          lambda f: f.get("prioridad") == "BAJA", lambda f: {"accion": "Registrar y mantener monitoreo"}, "BAJA", "registro"),
    Regla("R35", 3, 70, "REVISIÓN LEGAL: Legal y Catastro cotejan derechos del titular vecino",
          lambda f: f.get("prioridad") == "REVISIÓN LEGAL",
          lambda f: {"accion": "Derivar a Legal y Catastro para cotejar los derechos del titular vecino"}, "REVISIÓN LEGAL", "derivar a Legal"),
    Regla("R36", 3, 65, "INFORMATIVA: Legal evalúa si corresponde informar a la autoridad",
          lambda f: f.get("prioridad") == "INFORMATIVA",
          lambda f: {"accion": "Registrar; Legal evalúa si corresponde informar a la autoridad competente"}, "INFORMATIVA", "registro + Legal"),
    # ---------------- nivel 4: salvaguardas con supervisión humana
    Regla("R40", 4, 50, "Área protegida o comunidad cercana: revisión humana y protocolo comunitario antes de actuar",
          lambda f: f.get("area_sensible") or f.get("sensibilidad_social"),
          lambda f: {"revision_humana": True, "protocolo_comunitario": True}, "área sensible o comunidad", "revisión humana"),
    Regla("R41", 4, 40, "Acción de campo (CRÍTICA o ALTA): requiere aprobación del supervisor de turno",
          lambda f: f.get("prioridad") in ("CRÍTICA", "ALTA") and not f.get("emergencia"),
          lambda f: {"aprobacion_supervisor": True}, "CRÍTICA o ALTA", "aprobación humana"),
    Regla("R42", 4, 30, "Toda alerta clasificada genera un ticket trazable",
          lambda f: "prioridad" in f, lambda f: {"crear_ticket": True}, "prioridad asignada", "crear ticket"),
]


def motor_alertas() -> MotorReglas:
    return MotorReglas(REGLAS_ALERTA)


def motor_contexto() -> MotorReglas:
    """Solo reglas de contexto geoespacial y salvaguarda (para verificar puntos, sin priorizar alertas)."""
    return MotorReglas([r for r in REGLAS_ALERTA if r.nivel == 1 or r.id == "R40"])


# ====================================================================== red bayesiana

NODOS = ["H", "A", "I", "M", "P", "S", "R", "T", "V"]
DESCRIPCION_NODOS = {
    "H": "Antecedentes: alertas en el sector en los últimos 90 días",
    "A": "Actividad autorizada programada en el área",
    "I": "Intrusión o actividad minera no autorizada real (variable de interés)",
    "M": "Maquinaria o equipos observados",
    "P": "Personas o campamento no identificados",
    "S": "Alteración del terreno o de la cobertura",
    "R": "Ruido o voladuras nocturnas",
    "T": "Agua turbia aguas abajo",
    "V": "Vía o trocha nueva",
}
ARISTAS = [("H", "I"), ("I", "M"), ("A", "M"), ("I", "P"), ("A", "P"), ("I", "S"), ("A", "S"), ("I", "V"), ("A", "V"),
           ("I", "R"), ("I", "T")]

# TPC elicitadas: P(X = 1 | padres). Para nodos con padres (I, A) el orden es (I=0,A=0), (0,1), (1,0), (1,1).
TPC = {
    "H": 0.30,
    "A": 0.40,
    "I": {(0,): 0.10, (1,): 0.30},
    "M": {(0, 0): 0.03, (0, 1): 0.60, (1, 0): 0.70, (1, 1): 0.85},
    "P": {(0, 0): 0.10, (0, 1): 0.50, (1, 0): 0.75, (1, 1): 0.85},
    "S": {(0, 0): 0.05, (0, 1): 0.45, (1, 0): 0.65, (1, 1): 0.80},
    "V": {(0, 0): 0.03, (0, 1): 0.20, (1, 0): 0.40, (1, 1): 0.45},
    "R": {(0,): 0.05, (1,): 0.40},
    "T": {(0,): 0.15, (1,): 0.40},
}
PADRES = {"H": (), "A": (), "I": ("H",), "M": ("I", "A"), "P": ("I", "A"), "S": ("I", "A"), "V": ("I", "A"),
          "R": ("I",), "T": ("I",)}


class RedAlertas:
    """Red bayesiana de credibilidad de alertas, con inferencia exacta en pgmpy."""

    def __init__(self, tpc: dict | None = None):
        self.tpc = tpc or TPC
        self.modelo = self._construir()
        from pgmpy.inference import VariableElimination
        self.inf = VariableElimination(self.modelo)

    def _construir(self):
        try:
            from pgmpy.models import DiscreteBayesianNetwork as BN
        except ImportError:  # pgmpy < 1.0
            from pgmpy.models import BayesianNetwork as BN
        from pgmpy.factors.discrete import TabularCPD
        m = BN(ARISTAS)
        for nodo in NODOS:
            pad = PADRES[nodo]
            if not pad:
                p1 = self.tpc[nodo]
                cpd = TabularCPD(nodo, 2, [[1 - p1], [p1]])
            else:
                combos = list(itertools.product([0, 1], repeat=len(pad)))
                fila1 = [self.tpc[nodo][c] for c in combos]
                cpd = TabularCPD(nodo, 2, [[1 - v for v in fila1], fila1], evidence=list(pad), evidence_card=[2] * len(pad))
            m.add_cpds(cpd)
        if not m.check_model():
            raise ValueError("Red bayesiana inválida")
        return m

    def posterior(self, evidencia: dict[str, int]) -> float:
        ev = {k: int(v) for k, v in evidencia.items() if k in NODOS and k != "I"}
        q = self.inf.query(["I"], evidence=ev, show_progress=False)
        return float(q.values[1])

    def posterior_enumeracion(self, evidencia: dict[str, int]) -> float:
        """Verificación independiente por enumeración completa de la distribución conjunta."""
        return posterior_enumeracion(self.tpc, evidencia)

    def contribuciones(self, evidencia: dict[str, int]) -> list[tuple[str, float]]:
        """Cambio del posterior al retirar cada evidencia (explicación local tipo «dejar uno fuera»)."""
        base = self.posterior(evidencia)
        salida = []
        for k in evidencia:
            if k in ("H", "A"):
                continue
            resto = {kk: vv for kk, vv in evidencia.items() if kk != k}
            salida.append((k, base - self.posterior(resto)))
        return sorted(salida, key=lambda t: -abs(t[1]))


def posterior_enumeracion(tpc: dict, evidencia: dict[str, int]) -> float:
    num = den = 0.0
    libres = [n for n in NODOS if n not in evidencia]
    for valores in itertools.product([0, 1], repeat=len(libres)):
        asig = dict(evidencia)
        asig.update(dict(zip(libres, valores)))
        p = 1.0
        for nodo in NODOS:
            pad = PADRES[nodo]
            p1 = tpc[nodo] if not pad else tpc[nodo][tuple(asig[x] for x in pad)]
            p *= p1 if asig[nodo] == 1 else 1 - p1
        den += p
        if asig["I"] == 1:
            num += p
    return num / den


def sensibilidad(evidencia: dict[str, int], delta: float = 0.10) -> list[dict]:
    """Análisis de sensibilidad univariante: perturba ±delta cada parámetro de las TPC."""
    base = posterior_enumeracion(TPC, evidencia)
    filas = []
    for nodo, val in TPC.items():
        entradas = [((), val)] if not isinstance(val, dict) else list(val.items())
        for clave, p in entradas:
            extremos = []
            for signo in (-1, 1):
                tpc = {k: (dict(v) if isinstance(v, dict) else v) for k, v in TPC.items()}
                nuevo = float(np.clip(p + signo * delta, 0.01, 0.99))
                if isinstance(val, dict):
                    tpc[nodo][clave] = nuevo
                else:
                    tpc[nodo] = nuevo
                extremos.append(posterior_enumeracion(tpc, evidencia))
            bajo, alto = extremos
            cond = ",".join(f"{pa}={v}" for pa, v in zip(PADRES[nodo], clave))
            filas.append({"parametro": f"P({nodo}=1 | {cond})" if clave else f"P({nodo}=1)",
                          "bajo": bajo, "alto": alto, "rango": abs(alto - bajo), "base": base})
    return sorted(filas, key=lambda f: -f["rango"])


# ====================================================================== factibilidad de vuelo


def evaluar_dia(dia: int, campamento: str, op: dict, urgente: bool) -> tuple[bool, list[str], dict]:
    pol = op["politica_vuelo"]
    viento, lluvia, vis = op["pronostico"][campamento][str(dia)]
    drones = [d for d, v in op["drones"].items() if str(dia) not in v["no_disponible"]]
    pilotos = [p for p, v in op["pilotos"].items() if str(dia) not in v["no_disponible"]]
    motivos = []
    if viento > pol["viento_max_ms"]:
        motivos.append(f"V01 viento {viento:.1f} m/s > {pol['viento_max_ms']:.0f} m/s".replace(".", ","))
    if lluvia > pol["lluvia_max_mmh"]:
        motivos.append(f"V02 lluvia {lluvia:.1f} mm/h".replace(".", ","))
    if vis < pol["visibilidad_min_km"]:
        motivos.append(f"V03 visibilidad {vis:.0f} km < {pol['visibilidad_min_km']:.0f} km")
    if not drones:
        motivos.append("V04 sin dron disponible")
    if not pilotos:
        motivos.append("V05 sin piloto disponible")
    if not urgente and dia < pol["anticipacion_min_dias"]:
        motivos.append("V06 anticipación menor a 1 día para solicitud no urgente")
    return (not motivos), motivos, {"viento": viento, "lluvia": lluvia, "visibilidad": vis,
                                    "dron": drones[0] if drones else None, "piloto": pilotos[0] if pilotos else None}


def primera_ventana(desde: int, campamento: str, op: dict, urgente: bool = False) -> dict:
    """Búsqueda secuencial de la primera ventana apta dentro del horizonte de pronóstico."""
    evaluados = []
    for d in range(max(0, desde), op["politica_vuelo"]["horizonte_dias"] + 1):
        apto, motivos, datos = evaluar_dia(d, campamento, op, urgente)
        evaluados.append({"dia": d, "apto": apto, "motivos": motivos, **datos})
        if apto:
            return {"dia": d, "evaluados": evaluados, **datos}
    return {"dia": None, "evaluados": evaluados}


In [ ]:
%%writefile kuribot/dialogo.py
"""Gestor de diálogo de KuriBot: estado, llenado de ranuras, políticas y generación de respuestas.

Flujo por mensaje: (1) enmascarar datos personales; (2) red de seguridad de emergencias;
(3) extraer entidades; (4) completar una acción pendiente o (5) clasificar la intención;
(6) aplicar la política de la intención (consultas, geoprocesos, red bayesiana, reglas);
(7) responder con plantillas y devolver una traza auditable.
"""
from __future__ import annotations

import json
import re
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path

from . import entidades as E
from . import geodesia as G
from .conocimiento import BaseConocimiento
from .nlu import ClasificadorIntenciones
from .razonamiento import RedAlertas, motor_alertas, motor_contexto, primera_ventana

warnings.filterwarnings("ignore", category=FutureWarning)

NOMBRES_INTENCION = {
    "saludo": "saludar", "despedida": "despedirse", "ayuda_capacidades": "conocer qué puedo hacer",
    "consultar_concesion": "consultar los datos de una concesión", "verificar_punto": "verificar si un punto cae en una concesión",
    "convertir_coordenadas": "convertir coordenadas entre sistemas", "reportar_alerta": "reportar una posible actividad ilegal",
    "solicitar_vuelo": "solicitar un vuelo de dron", "solicitar_cartografia": "pedir un producto cartográfico",
    "estado_solicitud": "consultar el estado de una solicitud", "consulta_geodesia": "resolver una duda geodésica",
    "hablar_humano": "hablar con una persona", "emergencia_seguridad": "reportar una emergencia",
    "fuera_alcance": "un tema fuera de mi alcance",
}
CAPACIDADES = ("Puedo ayudarle a: consultar una concesión (p. ej., «estado de CM-103»); verificar si un punto cae dentro "
               "de una concesión; convertir coordenadas WGS 84 ↔ PSAD56; reportar una posible actividad minera ilegal; "
               "solicitar un vuelo de dron o un producto cartográfico; consultar el estado de un ticket (GEO-2026-0000); "
               "resolver dudas geodésicas frecuentes, o derivarle con una persona.")
AVISO = ("Soy KuriBot, un asistente automatizado de Geomática (prototipo académico con datos ficticios). "
         "Mis respuestas no sustituyen la revisión de un especialista.")
UBICACION = {"propia": "concesión propia", "tercero": "concesión de un tercero", "amortiguamiento": "franja de 500 m",
             "fuera": "fuera del área de la empresa"}
NEGATIVOS = re.compile(r"^\s*(no s[eé]|nada|ninguna|ninguno|no vi nada|no|desconozco|no tengo)\b", re.I)
CANCELAR = re.compile(r"\b(cancelar|cancela|olv[ií]dalo|dejalo|déjalo)\b", re.I)


_PII_FRASE = re.compile(r"\b(mi|su|el|la)?\s*(c[eé]dula|tel[eé]fono|celular|n[uú]mero|correo)\s*(es|:)?\s*\[(C[ÉE]DULA|TEL[ÉE]FONO|CORREO)\]\s*,?", re.I)
_PII_MARCA = re.compile(r"\[(C[ÉE]DULA|TEL[ÉE]FONO|CORREO)\]")


def sin_pii(t: str) -> str:
    """Texto para el clasificador: los datos personales enmascarados no aportan información de intención (ajuste v2)."""
    t = _PII_MARCA.sub(" ", _PII_FRASE.sub(" ", t))
    return re.sub(r"\s+", " ", t).strip(" ,")


def pct(p: float) -> str:
    return f"{100 * p:.1f} %".replace(".", ",")


def num(x: float, dec: int = 1) -> str:
    s = f"{x:,.{dec}f}"
    return s.replace(",", " ").replace(".", ",")


@dataclass
class Respuesta:
    texto: str
    intencion: str | None = None
    confianza: float | None = None
    decision: str | None = None
    top: list = field(default_factory=list)
    accion: str | None = None
    reglas: list = field(default_factory=list)
    traza_reglas: list = field(default_factory=list)
    bayes: dict | None = None
    entidades: dict = field(default_factory=dict)
    ticket: str | None = None
    mapa: dict | None = None
    vuelo: dict | None = None
    latencia_ms: float = 0.0
    latencia_nlu_ms: float = 0.0
    avisos: list = field(default_factory=list)


@dataclass
class Estado:
    pendiente: dict | None = None
    aclaracion: dict | None = None
    fallos: int = 0
    intentos_ranura: int = 0
    ultima_concesion: str | None = None
    turnos: int = 0


class KuriBot:
    def __init__(self, nlu: ClasificadorIntenciones | None = None, kb: BaseConocimiento | None = None,
                 registro: str | Path | None = None):
        self.nlu = nlu or ClasificadorIntenciones()
        self.kb = kb or BaseConocimiento()
        self.red = RedAlertas()
        self.motor = motor_alertas()
        self.motor_ctx = motor_contexto()
        self.estado = Estado()
        self.registro = Path(registro) if registro else None

    def reiniciar(self):
        self.estado = Estado()

    # ================================================================ entrada principal
    def responder(self, texto: str) -> Respuesta:
        t0 = time.perf_counter()
        self.estado.turnos += 1
        seguro, pii = E.enmascarar_pii(texto)
        ent = self._entidades(seguro)
        r = self._despachar(seguro, ent)
        if pii:
            r.avisos.append("R02: detecté datos personales (" + ", ".join(sorted(set(pii))) + "); no los registro ni los necesito.")
            r.reglas = ["R02"] + r.reglas
        r.entidades = {k: v for k, v in ent.items() if v not in (None, {}, [])}
        r.latencia_ms = (time.perf_counter() - t0) * 1000
        self._registrar(seguro, r)
        return r

    def _entidades(self, t: str) -> dict:
        c = E.extraer_coordenada(t)
        return {"coord": c, "datum": E.extraer_datum(t), "concesion": E.extraer_concesion(t, self.kb.nombres),
                "ticket": E.extraer_ticket(t), "fecha": E.extraer_fecha(t, self.kb.hoy), "producto": E.extraer_producto(t),
                "evidencias": E.extraer_evidencias(t), "urgente": bool(re.search(r"urgent", E.plano(t)))}

    def _despachar(self, t: str, ent: dict) -> Respuesta:
        st = self.estado
        if CANCELAR.search(t) and (st.pendiente or st.aclaracion):
            st.pendiente = st.aclaracion = None
            return Respuesta("De acuerdo, cancelé la solicitud en curso. ¿En qué más puedo ayudarle?", "cancelar", decision="regla")
        pred = self.nlu.predecir(sin_pii(t))
        base = {"confianza": pred.confianza, "top": pred.top, "latencia_nlu_ms": pred.latencia_ms}
        # (2) red de seguridad: emergencias por palabras clave o por la red neuronal
        if E.hay_emergencia(t) or (pred.intencion == "emergencia_seguridad" and pred.confianza >= self.nlu.umbral_abstener):
            st.pendiente = st.aclaracion = None
            return self._con(self._emergencia(t, ent), pred, "emergencia_seguridad", base)
        # (3) respuesta a una aclaración
        if st.aclaracion:
            elegido = self._resolver_aclaracion(t)
            if elegido:
                original = st.aclaracion["texto"]
                st.aclaracion = None
                ent2 = self._entidades(original)
                for k, v in ent.items():
                    if v and not ent2.get(k):
                        ent2[k] = v
                return self._con(self._politica(elegido, original, ent2), pred, elegido, base, decision="aclarada")
            st.aclaracion = None
        # (4) completar acción pendiente
        if st.pendiente:
            p = st.pendiente
            self._fusionar(p["slots"], ent)
            if self._aporta(p["falta"], ent, t) or (pred.intencion == p["intencion"] and pred.decision == "aceptar"):
                st.pendiente = None
                if p["falta"] == "evidencias" and not ent["evidencias"]:
                    p["slots"]["evidencias_sin_dato"] = True
                return self._con(self._politica(p["intencion"], t, p["slots"]), pred, p["intencion"], base, decision="ranura")
            if pred.decision == "aceptar" and pred.intencion not in ("fuera_alcance",):
                st.pendiente = None  # cambio de tema: se abandona la acción pendiente
            else:
                st.intentos_ranura += 1
                if st.intentos_ranura >= 2:
                    st.pendiente = None
                    return self._con(Respuesta("No logré completar el dato requerido. Puedo derivarle con un analista "
                                               "(escriba «quiero hablar con una persona») o puede empezar de nuevo."),
                                     pred, p["intencion"], base, decision="ranura_fallida")
                return self._con(Respuesta(p["pregunta"] + " (o escriba «cancelar»)."), pred, p["intencion"], base, decision="ranura")
        # (5) clasificación de la intención
        if pred.decision == "aclarar":
            a, b = pred.top[0][0], pred.top[1][0]
            st.aclaracion = {"opciones": [a, b], "texto": t}
            return self._con(Respuesta(f"No estoy seguro de haber entendido. ¿Quiere 1) {NOMBRES_INTENCION[a]} o "
                                       f"2) {NOMBRES_INTENCION[b]}? Responda 1 o 2."), pred, a, base, decision="aclarar")
        if pred.decision == "abstener":
            st.fallos += 1
            extra = (" Si lo prefiere, puedo derivarle con un analista: escriba «quiero hablar con una persona»."
                     if st.fallos >= 2 else "")
            return self._con(Respuesta("No tengo suficiente seguridad para responder eso. " + CAPACIDADES + extra),
                             pred, "fuera_alcance", base, decision="abstener")
        st.fallos = 0
        return self._con(self._politica(pred.intencion, t, dict(ent)), pred, pred.intencion, base, decision="aceptar")

    @staticmethod
    def _con(r: Respuesta, pred, intencion, base, decision=None) -> Respuesta:
        r.intencion = r.intencion or intencion
        r.confianza, r.top, r.latencia_nlu_ms = base["confianza"], base["top"], base["latencia_nlu_ms"]
        r.decision = r.decision or decision or pred.decision
        return r

    def _resolver_aclaracion(self, t: str) -> str | None:
        ops = self.estado.aclaracion["opciones"]
        tl = E.plano(t).strip()
        if re.fullmatch(r"(1|uno|la primera|primera|opcion 1|si|sí|a)\.?", tl):
            return ops[0]
        if re.fullmatch(r"(2|dos|la segunda|segunda|opcion 2|b)\.?", tl):
            return ops[1]
        return None

    @staticmethod
    def _fusionar(slots: dict, ent: dict):
        for k, v in ent.items():
            if k == "evidencias":
                slots.setdefault("evidencias", {}).update(v)
            elif v not in (None, {}, False):
                slots[k] = v

    @staticmethod
    def _aporta(falta: str, ent: dict, t: str) -> bool:
        if falta == "ubicacion":
            return bool(ent["coord"] or ent["concesion"])
        if falta == "evidencias":
            return bool(ent["evidencias"]) or bool(NEGATIVOS.search(t))
        return ent.get(falta) is not None and ent.get(falta) != {}

    def _pedir(self, intencion: str, falta: str, slots: dict, pregunta: str) -> Respuesta:
        self.estado.pendiente = {"intencion": intencion, "falta": falta, "slots": slots, "pregunta": pregunta}
        self.estado.intentos_ranura = 0
        return Respuesta(pregunta, intencion, accion=f"solicitar_dato:{falta}")

    # ================================================================ políticas por intención
    def _politica(self, intencion: str, t: str, s: dict) -> Respuesta:
        h = getattr(self, f"_i_{intencion}", None)
        return h(t, s) if h else Respuesta(CAPACIDADES, intencion)

    def _i_saludo(self, t, s):
        return Respuesta(f"¡Hola! {AVISO} {CAPACIDADES}", "saludo", accion="responder")

    def _i_despedida(self, t, s):
        self.estado.pendiente = None
        return Respuesta("Gracias por escribir. Si necesita algo más de Geomática, aquí estaré.", "despedida", accion="cerrar")

    def _i_ayuda_capacidades(self, t, s):
        return Respuesta(CAPACIDADES + " Para emergencias con riesgo para personas escriba «emergencia».", accion="responder")

    def _i_fuera_alcance(self, t, s):
        return Respuesta("Ese tema está fuera de mi alcance: solo atiendo consultas de Geomática y Catastro minero. "
                         + CAPACIDADES, accion="fuera_alcance")

    def _i_consultar_concesion(self, t, s):
        cod = s.get("concesion") or (self.estado.ultima_concesion if re.search(r"\b(esa|esta|la misma)\b", E.plano(t)) else None)
        if not cod:
            return self._pedir("consultar_concesion", "concesion", s, "¿De qué concesión? Indique el código (p. ej., CM-103) o el nombre")
        f = self.kb.ficha(cod)
        if not f:
            return Respuesta(f"No encuentro la concesión {cod} en el catastro de demostración; no puedo inventar sus datos.",
                             accion="no_encontrado")
        self.estado.ultima_concesion = cod
        if f["titular"] != "propia":
            return Respuesta(f"{cod} {f['nombre']} pertenece a un tercero con derecho vigente. Por confidencialidad solo "
                             f"confirmo su existencia y su superficie ({num(f['area_ha'])} ha); para más detalle, consulte a Legal y Catastro.",
                             accion="responder", mapa=self._mapa(cod))
        act = f"; actividad autorizada programada: {f['actividad_programada']}" if f["actividad_programada"] else ""
        return Respuesta(f"Concesión {cod} {f['nombre']}: fase {f['fase']}, estado {f['estado']}, superficie {num(f['area_ha'])} ha, "
                         f"campamento {f['campamento']}. Alertas en los últimos 90 días: {f['alertas_90d']}{act}. "
                         "Fuente: catastro interno de demostración (sintético).", accion="responder", mapa=self._mapa(cod))

    def _coordenada_lista(self, intencion: str, s: dict, pregunta_coord: str):
        c = s.get("coord")
        if c is None:
            return None, self._pedir(intencion, "coord", s, pregunta_coord)
        datum = s.get("datum") or c.punto.datum
        if datum is None:
            return None, self._pedir(intencion, "datum", s, "¿En qué datum están esas coordenadas: WGS 84 o PSAD56? "
                                                            "(en esta zona difieren unos 440 m)")
        p = G.Punto(c.punto.x, c.punto.y, c.punto.tipo, datum)
        e, n = G.a_wgs84_utm(p)
        if not G.dentro_de_ecuador_utm17s(e, n):
            return None, Respuesta("Las coordenadas quedan fuera del rango de Ecuador continental en UTM 17S (G02). "
                                   "Revise si el Este y el Norte están invertidos o si falta un dígito.", intencion, accion="rechazar")
        return (p, e, n, c.avisos), None

    def _i_verificar_punto(self, t, s):
        listo, r = self._coordenada_lista("verificar_punto", s, "Indique las coordenadas del punto (p. ej., E 642350 N 9662120) y su datum")
        if r:
            return r
        p, e, n, avisos = listo
        avisos = list(avisos)
        a = self.kb.analizar_punto(e, n)
        f = self.motor_ctx.inferir({"ubicacion": a.ubicacion, "area_protegida": bool(a.area_protegida),
                                    "comunidad_cercana": bool(a.comunidad)})
        partes = []
        if p.datum == "PSAD56":
            partes.append(f"Transformé el punto de PSAD56 a WGS 84 con {G.descripcion_transformacion()}: E {num(e)} N {num(n)}.")
            if p.tipo == "utm":
                crudo = self.kb.analizar_punto(p.x, p.y)  # lectura errónea: mismas cifras tomadas como WGS 84
                if (crudo.concesion or crudo.ubicacion) != (a.concesion or a.ubicacion):
                    avisos.append(f"Riesgo de datum: si estas cifras se leyeran como WGS 84, el punto caería en "
                                  f"{crudo.concesion or 'una zona sin concesión'} y no en {a.concesion or 'una zona sin concesión'}.")
        if a.ubicacion == "propia":
            c = self.kb.concesiones[a.concesion]
            partes.append(f"El punto está DENTRO de {a.concesion} {c['nombre']} ({c['estado']}), a {num(a.distancia_lindero_m, 0)} m del lindero.")
        elif a.ubicacion == "tercero":
            partes.append(f"El punto está dentro de {a.concesion}, concesión de un tercero con derecho vigente (R12). "
                          "No debe intervenirse sin revisión de Legal.")
        elif a.ubicacion == "amortiguamiento":
            partes.append(f"El punto está FUERA de las concesiones, a {num(a.distancia_lindero_m, 0)} m de {a.cercana} "
                          "(dentro de la franja de 500 m, R11).")
        else:
            partes.append(f"El punto está fuera de las concesiones de la empresa; la más cercana es {a.cercana} a {num(a.distancia_lindero_m, 0)} m.")
        if a.area_protegida:
            partes.append(f"Atención: cae en {a.area_protegida} (R13).")
        if a.comunidad:
            partes.append(f"Hay una comunidad a {num(a.distancia_comunidad_m, 0)} m: {a.comunidad} (R14).")
        partes.append("Verificación sobre el catastro sintético del prototipo; para fines legales use el catastro oficial vigente.")
        self.estado.ultima_concesion = a.concesion or a.cercana
        return Respuesta(" ".join(partes), "verificar_punto", accion="geoproceso", reglas=f.disparadas, traza_reglas=f.traza,
                         mapa=self._mapa(a.concesion or a.cercana, (e, n)), avisos=avisos)

    def _i_convertir_coordenadas(self, t, s):
        listo, r = self._coordenada_lista("convertir_coordenadas", s, "Indique las coordenadas a convertir y su datum de origen")
        if r:
            return r
        p, e, n, avisos = listo
        rep = G.todas_las_representaciones(p)
        (we, wn), (lat, lon), (pe, pn) = rep["WGS84_UTM17S"], rep["WGS84_GEO"], rep["PSAD56_UTM17S"]
        texto = (f"Origen: {p.datum}. Resultados: WGS 84 / UTM 17S (EPSG:32717): E {num(we, 2)} m, N {num(wn, 2)} m · "
                 f"WGS 84 geográficas (EPSG:4326): {lat:.6f}, {lon:.6f} ({G.dms(lat, 'lat')}, {G.dms(lon, 'lon')}) · "
                 f"PSAD56 / UTM 17S (EPSG:24877): E {num(pe, 2)} m, N {num(pn, 2)} m. "
                 f"Transformación de datum: {G.descripcion_transformacion()}. SIRGAS 2000 se considera equivalente a WGS 84 "
                 "al nivel métrico.")
        return Respuesta(texto, "convertir_coordenadas", accion="geoproceso", mapa=self._mapa(None, (we, wn)), avisos=list(avisos))

    def _ubicar_alerta(self, s) -> dict:
        """Contexto geoespacial y nodos raíz (H, A) de la red bayesiana para la alerta."""
        ctx = {"codigo": None, "ubicacion": None, "area_protegida": False, "comunidad_cercana": False, "punto": None}
        ev = {}
        if s.get("coord") is not None:
            datum = s.get("datum") or s["coord"].punto.datum or "WGS84"
            pt = G.Punto(s["coord"].punto.x, s["coord"].punto.y, s["coord"].punto.tipo, datum)
            e, n = G.a_wgs84_utm(pt)
            a = self.kb.analizar_punto(e, n)
            ctx.update(codigo=a.concesion or a.cercana, ubicacion=a.ubicacion, area_protegida=bool(a.area_protegida),
                       comunidad_cercana=bool(a.comunidad), punto=(e, n), datum_supuesto=s.get("datum") is None and s["coord"].punto.datum is None)
            if a.ubicacion in ("propia", "amortiguamiento"):
                ev["H"] = int(self.kb.op["historial_alertas_90d"].get(ctx["codigo"], 0) > 0)
                ev["A"] = int(a.ubicacion == "propia" and ctx["codigo"] in self.kb.op["actividad_autorizada_programada"])
        else:
            cod = s["concesion"]
            c = self.kb.concesiones.get(cod)
            ctx["codigo"] = cod
            if c:
                ctx["ubicacion"] = "propia" if c["titular"] == "propia" else "tercero"
                g = c["geom"]
                ctx["area_protegida"] = any(g.distance(x["geom"]) <= 1.0 for x in self.kb.areas.values())
                ctx["comunidad_cercana"] = any(g.distance(x["geom"]) <= 1000 for x in self.kb.comunidades.values())
                ev["H"] = int(self.kb.op["historial_alertas_90d"].get(cod, 0) > 0)
                if c["titular"] == "propia":
                    ev["A"] = int(cod in self.kb.op["actividad_autorizada_programada"])
        return {"ctx": ctx, "ev_raiz": ev}

    def _i_reportar_alerta(self, t, s):
        if not (s.get("coord") or s.get("concesion")):
            return self._pedir("reportar_alerta", "ubicacion", s, "¿Dónde ocurre? Indique la concesión (código o nombre) o las coordenadas con su datum")
        if s.get("concesion") and not s.get("coord") and s["concesion"] not in self.kb.concesiones:
            return Respuesta(f"No encuentro {s['concesion']} en el catastro; indique coordenadas para ubicar la alerta.", accion="no_encontrado")
        if not s.get("evidencias") and not s.get("evidencias_sin_dato"):
            return self._pedir("reportar_alerta", "evidencias", s, "¿Qué observó? Por ejemplo: maquinaria, personas o campamento, "
                                                                     "desbroce o huecos nuevos, agua turbia, ruido o voladuras de noche, trocha nueva. "
                                                                     "Si no lo sabe, escriba «no sé»")
        u = self._ubicar_alerta(s)
        ctx, ev = u["ctx"], dict(u["ev_raiz"])
        obs = {k: int(v) for k, v in (s.get("evidencias") or {}).items()}
        ev.update(obs)
        p = self.red.posterior(ev)
        contrib = self.red.contribuciones(ev)
        camp = self.kb.campamento(ctx["codigo"])
        ventana = primera_ventana(0, camp, self.kb.op, urgente=True)
        hechos = {"ubicacion": ctx["ubicacion"], "area_protegida": ctx["area_protegida"],
                  "comunidad_cercana": ctx["comunidad_cercana"], "p_intrusion": p, "vuelo_dias": ventana["dia"]}
        inf = self.motor.inferir(hechos)
        f = inf.hechos
        prioridad, accion = f.get("prioridad"), f.get("accion")
        resp = "Seguridad patrimonial" if prioridad not in ("REVISIÓN LEGAL", "INFORMATIVA") else "Legal y Catastro"
        compromiso = self.kb.fecha(ventana["dia"]) if ventana["dia"] is not None and prioridad in ("CRÍTICA", "ALTA") else None
        tk = self.kb.nuevo_ticket("alerta", f"Alerta en {ctx['codigo']} · P={p:.3f}", resp, compromiso, prioridad)
        nombre = self.kb.concesiones.get(ctx["codigo"], {}).get("nombre", "")
        evid = ", ".join(f"{E.EVIDENCIAS[k][0]} ({'sí' if v else 'no'})" for k, v in obs.items()) or "sin evidencias específicas"
        partes = [f"Alerta registrada: {tk} · prioridad {prioridad}.",
                  f"Ubicación: {ctx['codigo']} {nombre} ({UBICACION.get(ctx['ubicacion'], 'sin ubicar')}).", f"Evidencias: {evid}."]
        if "H" in ev:
            partes.append(f"Antecedentes en 90 días: {'sí' if ev['H'] else 'no'}; actividad autorizada programada: "
                          f"{'sí' if ev.get('A') else ('no' if 'A' in ev else 'desconocida')}.")
        partes.append(f"Probabilidad estimada de actividad no autorizada: {pct(p)} (red bayesiana con supuestos expertos; no es una certeza).")
        if ev.get("A") == 1 and obs:
            partes.append("La actividad autorizada programada explica parte de lo observado, por eso la probabilidad no es mayor.")
        partes.append(f"Acción propuesta: {accion}.")
        if ventana["dia"] is not None and prioridad in ("CRÍTICA", "ALTA"):
            partes.append(f"Primera ventana de vuelo apta: {self.kb.fecha(ventana['dia'])} ({ventana['dron']}, {ventana['piloto']}).")
        if f.get("aprobacion_supervisor"):
            partes.append("Antes de ejecutar, el supervisor de turno debe aprobar la acción (R41).")
        if f.get("revision_humana"):
            partes.append("Zona sensible: se requiere revisión humana y protocolo comunitario antes de cualquier acción de campo (R40).")
        if ctx.get("datum_supuesto"):
            partes.append("Asumí WGS 84 para las coordenadas porque no indicó el datum; confírmelo si no es así.")
        bayes = {"p_intrusion": p, "evidencia": ev, "contribuciones": contrib}
        return Respuesta(" ".join(partes), "reportar_alerta", accion=accion, reglas=inf.disparadas, traza_reglas=inf.traza,
                         bayes=bayes, ticket=tk, vuelo=ventana, mapa=self._mapa(ctx["codigo"], ctx["punto"]))

    def _i_solicitar_vuelo(self, t, s):
        cod = s.get("concesion")
        if not cod:
            return self._pedir("solicitar_vuelo", "concesion", s, "¿Sobre qué concesión necesita el vuelo? (código o nombre)")
        if cod not in self.kb.concesiones:
            return Respuesta(f"No encuentro {cod} en el catastro de demostración.", accion="no_encontrado")
        if s.get("fecha") is None:
            return self._pedir("solicitar_vuelo", "fecha", s, "¿Para qué fecha? (hoy, mañana, un día de la semana o una fecha)")
        camp = self.kb.campamento(cod)
        dia = max(0, s["fecha"])
        if dia > self.kb.op["politica_vuelo"]["horizonte_dias"]:
            return Respuesta(f"Solo tengo pronóstico hasta {self.kb.fecha(self.kb.op['politica_vuelo']['horizonte_dias'])}. "
                             "Registre la solicitud más cerca de la fecha o hable con el coordinador de vuelos.", accion="fuera_horizonte")
        v = primera_ventana(dia, camp, self.kb.op, urgente=s.get("urgente", False))
        rechazos = [f"{self.kb.fecha(d['dia'])}: " + "; ".join(m.split(' ', 1)[1] for m in d["motivos"]) for d in v["evaluados"] if not d["apto"]]
        if v["dia"] is None:
            return Respuesta(f"No hay ventana apta en el horizonte de pronóstico para el campamento {camp}. Motivos: "
                             + " | ".join(rechazos) + ". Derivo al coordinador de vuelos.", accion="sin_ventana", vuelo=v)
        tk = self.kb.nuevo_ticket("vuelo", f"Vuelo en {cod}", v["piloto"], self.kb.fecha(v["dia"]))
        cab = "Fecha solicitada apta." if v["dia"] == dia else f"La fecha solicitada no es apta ({' | '.join(rechazos)})."
        return Respuesta(f"{cab} Propuesta: vuelo en {cod} el {self.kb.fecha(v['dia'])} con {v['dron']} y {v['piloto']} "
                         f"(viento {num(v['viento'])} m/s, sin lluvia, visibilidad {v['visibilidad']} km). Solicitud {tk} "
                         "registrada como pendiente de aprobación del coordinador de vuelos.", "solicitar_vuelo",
                         accion="programar_vuelo", ticket=tk, vuelo=v, mapa=self._mapa(cod))

    def _i_solicitar_cartografia(self, t, s):
        prod = s.get("producto")
        if not prod:
            return self._pedir("solicitar_cartografia", "producto", s, "¿Qué producto necesita? (mapa de ubicación, plano topográfico, "
                                                                        "ortofoto, KMZ, shapefile o modelo digital de elevación)")
        cod = s.get("concesion") or self.estado.ultima_concesion
        if not cod:
            return self._pedir("solicitar_cartografia", "concesion", s, "¿De qué concesión o área? (código o nombre)")
        sla = self.kb.op["sla_cartografia_dias_habiles"]
        dias = sla["urgente"] if s.get("urgente") else sla["estandar"]
        tk = self.kb.nuevo_ticket("cartografía", f"{prod} de {cod}", "Analista SIG", None)
        return Respuesta(f"Solicitud {tk} registrada: {prod} de {cod}. Plazo interno de referencia: {dias} día(s) hábil(es)"
                         f"{' (urgente: requiere aprobación de la jefatura)' if s.get('urgente') else ''}. "
                         "El producto se entregará en EPSG:32717 salvo que indique otro sistema.", "solicitar_cartografia",
                         accion="crear_ticket", ticket=tk)

    def _i_estado_solicitud(self, t, s):
        tk = s.get("ticket")
        if not tk:
            return self._pedir("estado_solicitud", "ticket", s, "Indique el número de solicitud (formato GEO-2026-0000)")
        d = self.kb.ticket(tk)
        if not d:
            return Respuesta(f"No encuentro la solicitud {tk}. Revise el número; no puedo suponer su estado.", accion="no_encontrado")
        extra = f" Resolución: {d['resolucion']}." if d.get("resolucion") else ""
        comp = f", fecha comprometida {d['compromiso']}" if d.get("compromiso") else ""
        return Respuesta(f"{tk} ({d['tipo']}: {d['detalle']}): estado «{d['estado']}», responsable {d['asignado']}{comp}.{extra}",
                         "estado_solicitud", accion="consultar_ticket", ticket=tk)

    def _i_consulta_geodesia(self, t, s):
        f, sim = self.kb.buscar_faq(t)
        if sim < 0.30:
            return Respuesta("No tengo una respuesta verificada para esa duda. Para no darle información incorrecta, la derivo "
                             "al analista de geodesia (o reformule la pregunta).", accion="derivar_faq",
                             traza_reglas=[f"FAQ: similitud máxima {sim:.2f} < 0,30"])
        return Respuesta(f"{f['respuesta']} (Fuente: {f['fuente']}.)", "consulta_geodesia", accion="responder_faq",
                         traza_reglas=[f"FAQ {f['id']} · similitud {sim:.2f}"])

    def _i_hablar_humano(self, t, s):
        inf = self.motor.inferir({"intencion": "hablar_humano"})
        tk = self.kb.nuevo_ticket("derivación", "Solicitud de atención humana", "Mesa de Geomática", None)
        return Respuesta(f"Entendido. Registré la derivación {tk}; un analista de la Mesa de Geomática le contactará por el canal "
                         "interno en horario laboral (simulado en este prototipo).", "hablar_humano", accion="derivar_humano",
                         reglas=inf.disparadas, traza_reglas=inf.traza, ticket=tk)

    def _emergencia(self, t, ent):
        u = self._ubicar_alerta(ent) if (ent.get("coord") or ent.get("concesion")) else {"ctx": {"codigo": None}}
        inf = self.motor.inferir({"emergencia": True})
        cod = u["ctx"].get("codigo")
        tk = self.kb.nuevo_ticket("emergencia", f"Emergencia reportada{' en ' + cod if cod else ''}", "Jefe de Seguridad", self.kb.fecha(0), "CRÍTICA")
        return Respuesta("EMERGENCIA: si hay personas en riesgo, llame de inmediato al ECU 911 y al Jefe de Seguridad del campamento. "
                         "Aléjese del lugar si es seguro hacerlo. No se enviará personal ni dron a la zona hasta que Seguridad lo autorice (R01). "
                         f"Registré el caso {tk} con prioridad CRÍTICA y lo derivé a una persona.", "emergencia_seguridad",
                         accion="PROTOCOLO DE EMERGENCIA", reglas=inf.disparadas, traza_reglas=inf.traza, ticket=tk)

    # ================================================================ utilidades
    def _mapa(self, codigo: str | None, punto: tuple[float, float] | None = None) -> dict:
        return {"concesion": codigo, "punto": punto}

    def _registrar(self, texto: str, r: Respuesta):
        if not self.registro:
            return
        fila = {"turno": self.estado.turnos, "texto_enmascarado": texto, "intencion": r.intencion, "confianza": r.confianza,
                "decision": r.decision, "accion": r.accion, "reglas": r.reglas, "ticket": r.ticket,
                "p_intrusion": (r.bayes or {}).get("p_intrusion"), "latencia_ms": round(r.latencia_ms, 2)}
        self.registro.parent.mkdir(parents=True, exist_ok=True)
        with open(self.registro, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(fila, ensure_ascii=False) + "\n")


def respuesta_a_dict(r: Respuesta) -> dict:
    d = asdict(r)
    if r.entidades.get("coord") is not None:
        c = r.entidades["coord"]
        d["entidades"]["coord"] = {"x": c.punto.x, "y": c.punto.y, "tipo": c.punto.tipo, "datum": c.punto.datum}
    return d


In [ ]:
%%writefile kuribot/mapa.py
"""Mapa estático (matplotlib) del catastro sintético: funciona sin conexión a servicios de teselas."""
from __future__ import annotations

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.patches import Polygon as MplPolygon  # noqa: E402

COLORES = {"propia": "#2E7D5B", "tercero": "#9A6B2F", "resaltado": "#C0392B", "protegida": "#5B8DB8"}


def dibujar(kb, codigo: str | None = None, punto: tuple[float, float] | None = None, ancho: float = 5.2, alto: float = 4.2):
    fig, ax = plt.subplots(figsize=(ancho, alto), dpi=110)
    for a in kb.areas.values():
        ax.add_patch(MplPolygon(list(a["geom"].exterior.coords), closed=True, fc=COLORES["protegida"], alpha=0.18,
                                ec=COLORES["protegida"], lw=1, hatch="..", label="Área protegida (ficticia)"))
    for cod, c in kb.concesiones.items():
        color = COLORES["resaltado"] if cod == codigo else COLORES[c["titular"]]
        ax.add_patch(MplPolygon(list(c["geom"].exterior.coords), closed=True, fc=color, alpha=0.30 if cod == codigo else 0.12,
                                ec=color, lw=2.2 if cod == codigo else 1))
        x, y = c["geom"].centroid.coords[0]
        ax.text(x, y, cod, ha="center", va="center", fontsize=7, color="#222")
    for com in kb.comunidades.values():
        x, y = com["geom"].coords[0]
        ax.plot(x, y, marker="^", color="#6C3483", ms=7)
        ax.text(x + 120, y - 160, "Comunidad", fontsize=6, color="#6C3483")
    if punto is not None:
        ax.plot(*punto, marker="*", color="#111", ms=14, mec="white", mew=1)
    ax.set_xlim(639_400, 649_600)
    ax.set_ylim(9_658_000, 9_666_600)
    ax.set_aspect("equal")
    ax.ticklabel_format(useOffset=False, style="plain")
    ax.tick_params(labelsize=6)
    ax.set_xlabel("Este (m) · WGS 84 / UTM 17S", fontsize=7)
    ax.set_ylabel("Norte (m)", fontsize=7)
    ax.grid(alpha=0.25, lw=0.5)
    ax.set_title("Catastro sintético del prototipo", fontsize=8)
    fig.tight_layout()
    return fig


In [ ]:
%%writefile datos/corpus_fuente.py
"""Corpus fuente de KuriBot redactado por el equipo (español de Ecuador, registro laboral).

POOL: plantillas para entrenamiento y validación. PRUEBA: frases redactadas por separado,
con otra formulación, que nunca se usan para entrenar ni para ajustar hiperparámetros.
Marcadores: {conc} concesión, {coord} coordenadas, {ticket} solicitud, {producto} producto
cartográfico, {fecha} fecha, {camp} campamento. Los valores se sustituyen en
construir_corpus.py con una semilla fija.
"""

POOL: dict[str, list[str]] = {
    "saludo": [
        "hola", "buenos días", "buenas tardes", "buenas noches", "hola, ¿qué tal?",
        "buen día, ¿estás ahí?", "saludos", "hola kuribot", "qué tal, buenos días",
        "hey, ¿hay alguien?", "buenas", "hola, ¿cómo estás?", "muy buenos días equipo",
        "hola de nuevo", "holaa", "saludos cordiales", "buenos días, soy de operaciones",
        "hola, soy el supervisor del campamento norte", "qué más, buenas", "aló", "holi",
        "buen día", "hola, buen día", "buenas noches, recién llego al turno", "hola asistente",
        "qué tal", "un saludo", "hola, ¿me escuchas?", "buenas, aquí reportándome",
        "buenas tardes ingeniero", "hola, ¿cómo va todo?", "saludos desde la oficina de Machala",
    ],
    "despedida": [
        "gracias", "muchas gracias", "chao", "adiós", "hasta luego", "eso es todo, gracias",
        "listo, gracias por la ayuda", "nos vemos", "perfecto, hasta mañana", "gracias, muy amable",
        "ok gracias, chao", "te agradezco", "excelente, eso era todo", "ya no necesito nada más",
        "bye", "hasta pronto", "mil gracias", "de acuerdo, me despido", "gracias por la información",
        "ya está, gracias", "buenísimo, hasta la próxima", "terminamos por hoy",
        "listo, cierro la conversación", "gracias, que tengas buen día", "chao chao",
        "perfecto, eso resolvió mi duda", "muy agradecido", "Dios le pague", "hasta luego, buen turno",
        "fin", "gracias, me sirvió bastante", "ok, luego te escribo",
    ],
    "ayuda_capacidades": [
        "¿qué puedes hacer?", "¿en qué me puedes ayudar?", "ayuda", "¿cuáles son tus funciones?",
        "¿para qué sirves?", "muéstrame las opciones", "¿qué tipo de consultas atiendes?", "menú",
        "no sé cómo usarte", "¿cómo funciona este asistente?", "dime qué servicios ofreces",
        "necesito ayuda para usar el bot", "¿qué preguntas te puedo hacer?", "opciones disponibles",
        "¿qué sabes hacer?", "instrucciones de uso", "explícame qué haces",
        "¿me ayudas? no sé por dónde empezar", "lista de comandos", "¿qué trámites puedo hacer contigo?",
        "¿tú qué resuelves?", "quiero saber tus capacidades", "¿atiendes temas de geomática?",
        "¿cómo te uso?", "help", "¿qué más puedes hacer?", "guíame",
        "¿qué solicitudes puedo registrar aquí?", "¿cuál es tu función?", "dame un resumen de lo que haces",
    ],
    "consultar_concesion": [
        "¿cuál es el estado de la concesión {conc}?", "información de la concesión {conc}",
        "dame los datos de {conc}", "¿{conc} está vigente?", "¿qué área tiene la concesión {conc}?",
        "¿en qué fase está {conc}?", "quiero la ficha de la concesión {conc}",
        "¿quién es el titular de {conc}?", "consulta del derecho minero {conc}",
        "necesito saber la superficie de {conc}", "¿a qué campamento pertenece {conc}?",
        "¿cuántas hectáreas tiene {conc}?", "¿{conc} está en exploración o explotación?",
        "estado catastral de {conc}", "revisa la concesión {conc} por favor",
        "detalles del área minera {conc}", "¿la concesión {conc} sigue activa?",
        "¿qué régimen tiene {conc}?", "muéstrame la información catastral de {conc}",
        "quiero consultar una concesión", "datos de una concesión minera",
        "¿me das la info de la concesión?", "¿cuál es la situación legal de {conc}?",
        "ficha técnica de {conc}", "¿{conc} tiene alguna suspensión?",
        "necesito el resumen de la concesión {conc}", "consulta catastro {conc}",
        "¿cuándo vence la concesión {conc}?", "¿cuál es el código de la concesión {conc}?",
        "busca la concesión {conc}", "¿qué sabes de {conc}?", "características de la concesión {conc}",
    ],
    "verificar_punto": [
        "¿el punto {coord} está dentro de alguna concesión?",
        "verifica si {coord} cae en nuestra concesión",
        "¿estas coordenadas {coord} están dentro del área minera?",
        "quiero saber a qué concesión pertenece el punto {coord}",
        "¿{coord} está dentro o fuera del lindero?",
        "revisa si la coordenada {coord} está en el catastro",
        "¿a qué distancia del lindero está {coord}?",
        "¿el pozo en {coord} está dentro de la concesión?", "comprueba la ubicación {coord}",
        "¿este punto cae en área protegida? {coord}",
        "¿la coordenada {coord} es de nosotros o de un tercero?",
        "tengo un punto GPS {coord}, ¿dónde cae?", "ubica el punto {coord} en el catastro",
        "¿{coord} está dentro de la zona de amortiguamiento?", "verificar punto en concesión",
        "necesito verificar una coordenada", "¿puedes ubicar estas coordenadas? {coord}",
        "dime si {coord} está en nuestra área", "¿qué hay en {coord}?",
        "¿está {coord} dentro de algún polígono minero?",
        "valida si el hito {coord} está en el lindero", "cruce de coordenada {coord} con el catastro",
        "¿el frente de trabajo {coord} invade a un vecino?", "consulta espacial del punto {coord}",
        "¿la bocamina en {coord} está dentro del área?",
        "quiero confirmar si un punto está dentro de la concesión",
        "¿en qué concesión queda {coord}?", "¿hay alguna comunidad cerca de {coord}?",
    ],
    "convertir_coordenadas": [
        "convierte {coord} a WGS84", "transforma estas coordenadas de PSAD56 a WGS84: {coord}",
        "pasa {coord} a geográficas", "necesito convertir {coord} a UTM",
        "¿cuánto es {coord} en coordenadas geográficas?", "cambia el datum de {coord} a WGS84",
        "conversión de coordenadas PSAD56 a WGS84",
        "tengo coordenadas en PSAD56 y las necesito en WGS84",
        "transforma {coord} a grados decimales", "convierte de geográficas a UTM 17 sur: {coord}",
        "¿me ayudas a convertir unas coordenadas?", "pásame {coord} a PSAD56",
        "quiero transformar coordenadas entre datums",
        "convertir {coord} de UTM a latitud y longitud", "reproyecta {coord} a WGS84 zona 17S",
        "¿cuáles son las coordenadas geográficas de {coord}?", "transformación de coordenadas",
        "¿cómo quedaría {coord} en SIRGAS?", "convierte este punto {coord}",
        "necesito las coordenadas de {coord} en otro sistema",
        "transforma {coord} de WGS84 a PSAD56", "cambio de sistema de referencia para {coord}",
        "dame la latitud y longitud de {coord}", "convertir coordenadas UTM",
        "paso de PSAD56 a WGS84 de {coord}", "grados minutos segundos de {coord}",
        "{coord} en decimales por favor", "¿{coord} cómo queda en UTM?",
    ],
    "reportar_alerta": [
        "quiero reportar posible minería ilegal en {conc}",
        "hay maquinaria extraña trabajando en {conc}", "reporto una invasión en la concesión {conc}",
        "vimos gente excavando dentro de {conc}",
        "hay una retroexcavadora no autorizada cerca del lindero de {conc}",
        "se observa desbroce nuevo en {conc}", "alerta de actividad ilegal en {conc}",
        "los comuneros dicen que hay mineros informales en {conc}",
        "el agua del río bajó turbia desde {conc}", "se escuchan voladuras en la noche por {conc}",
        "encontramos una trocha nueva hacia {conc}", "hay un campamento clandestino en {conc}",
        "reportar intrusión", "quiero registrar una alerta",
        "detectamos un cambio de cobertura en la imagen satelital de {conc}",
        "el vigilante reporta volquetas saliendo de {conc}",
        "hay huecos nuevos en {conc}, parecen labores ilegales",
        "posible actividad minera no autorizada", "denuncia de minería ilegal",
        "se metieron a trabajar en nuestra concesión {conc}",
        "hay una clasificadora instalada en {conc} que no es nuestra",
        "reportan ruido de maquinaria de madrugada en {conc}", "hay deforestación reciente en {conc}",
        "registrar alerta de invasión en el área minera",
        "creo que están sacando material de {conc} sin permiso",
        "la patrulla vio personas extrañas en {conc}",
        "hay una bocamina nueva que no conocemos en {conc}",
        "en la imagen Sentinel se ve un claro nuevo en {conc}",
        "hay mangueras y bombas en la quebrada de {conc}",
        "informar actividad sospechosa en el lindero", "tenemos una alerta satelital en {conc}",
        "invadieron el área de {conc}",
    ],
    "solicitar_vuelo": [
        "necesito un vuelo de dron en {conc}", "programa un vuelo de inspección para {conc} {fecha}",
        "solicito sobrevuelo con dron en {conc}", "¿pueden volar el dron sobre {conc} {fecha}?",
        "quiero agendar una inspección aérea en {conc}",
        "requiero ortofoto actualizada de {conc} con dron",
        "agenda un vuelo fotogramétrico en {conc} para {fecha}", "solicitud de vuelo RPAS en {conc}",
        "¿hay dron disponible {fecha} para {conc}?", "necesito volar el frente de explotación de {conc}",
        "pide un vuelo de verificación en {conc}", "programar inspección con drone",
        "quiero solicitar un vuelo", "mandar el dron a {conc} {fecha}",
        "vuelo de dron para levantamiento de {conc}", "necesitamos imágenes aéreas de {conc}",
        "reservar el dron para {fecha}", "¿se puede hacer un vuelo {fecha} en {conc}?",
        "solicito inspección aérea urgente en {conc}",
        "coordinar vuelo de dron sobre el lindero de {conc}",
        "planifica un vuelo para medir volúmenes en {conc}",
        "quiero un sobrevuelo de la escombrera de {conc}", "necesito el dron {fecha}", "agendar dron",
        "vuelo de monitoreo en {conc} por favor", "¿cuándo pueden volar sobre {conc}?",
        "programación de vuelo para el campamento {camp}", "quiero fotos aéreas de {conc} {fecha}",
    ],
    "solicitar_cartografia": [
        "necesito un mapa de ubicación de {conc}", "solicito el {producto} de {conc}",
        "¿me pueden enviar el {producto} de {conc}?", "requiero cartografía actualizada de {conc}",
        "quiero un plano topográfico de {conc}", "necesito el shapefile de los linderos de {conc}",
        "envíame el KMZ de {conc}", "pido un mapa para el informe de {conc}",
        "solicitud de cartografía", "necesito un mapa", "elabórame un mapa de accesos de {conc}",
        "quiero el modelo digital de terreno de {conc}", "¿tienen las curvas de nivel de {conc}?",
        "requiero el {producto} para presentar a la autoridad", "hazme un mapa temático de {conc}",
        "necesito la capa de concesiones en geopackage", "quiero el plano de {conc} en PDF",
        "solicito un mapa de cobertura vegetal de {conc}",
        "necesito el {producto} para la reunión de mañana", "pedir un producto cartográfico",
        "genera un mapa con los hitos de {conc}", "necesito las ortofotos de {conc} en formato tif",
        "¿me ayudas con un mapa de la concesión {conc}?",
        "requiero el mapa base del campamento {camp}", "mapa de pendientes de {conc}",
        "quiero la cartografía de la quebrada en {conc}", "plano de implantación para {conc}",
        "exporta los linderos de {conc} a KML",
    ],
    "estado_solicitud": [
        "¿cuál es el estado de mi solicitud {ticket}?", "¿cómo va el ticket {ticket}?",
        "consulta del requerimiento {ticket}", "¿ya está listo mi pedido {ticket}?",
        "seguimiento de la solicitud {ticket}", "¿en qué estado está {ticket}?",
        "¿ya atendieron {ticket}?", "quiero saber cómo va mi solicitud", "estado de mi ticket",
        "¿qué pasó con mi pedido de mapa?", "¿ya programaron mi vuelo {ticket}?",
        "revisa el avance de {ticket}", "¿cuándo me entregan {ticket}?", "dame el estatus de {ticket}",
        "seguimiento de ticket", "¿se cerró la alerta {ticket}?", "¿mi requerimiento ya fue asignado?",
        "¿quién está atendiendo {ticket}?", "¿hay novedades de {ticket}?", "consultar solicitud",
        "¿ya hay respuesta de mi ticket {ticket}?", "quiero el estado del caso {ticket}",
        "¿en qué quedó mi requerimiento?", "avance de mi solicitud de cartografía",
        "¿mi alerta ya fue revisada? {ticket}", "verificar estado del ticket {ticket}",
    ],
    "consulta_geodesia": [
        "¿cuál es la diferencia entre PSAD56 y WGS84?", "¿qué código EPSG uso para UTM 17 sur?",
        "¿qué datum es el oficial en Ecuador?", "¿qué es SIRGAS?", "¿en qué zona UTM está el proyecto?",
        "¿por qué mis coordenadas no coinciden con el catastro?",
        "¿cuál es el EPSG de PSAD56 UTM 17S?", "¿qué precisión tiene un GPS navegador?",
        "¿qué diferencia hay entre RTK y un GPS de mano?", "¿qué formato de archivo debo entregar?",
        "¿cuál es la política para vuelos de dron?",
        "¿cuánto se desplaza un punto al pasar de PSAD56 a WGS84?", "¿qué es una proyección UTM?",
        "¿qué es el falso norte?", "¿por qué hay diferencia de cientos de metros entre datums?",
        "¿qué es un datum geodésico?", "¿qué es la época de referencia en SIRGAS?",
        "¿qué información necesito para reportar una alerta?",
        "¿cuánto tiempo tarda una solicitud de mapa?", "¿qué significa EPSG?", "¿qué es una ortofoto?",
        "¿qué es un modelo digital de terreno?",
        "¿cuál es la diferencia entre altura elipsoidal y ortométrica?",
        "¿cuál es el EPSG de WGS84 geográficas?", "¿con qué anticipación debo pedir un vuelo?",
        "¿cómo sé en qué datum están mis coordenadas?", "duda sobre sistemas de referencia",
        "¿qué marco de referencia usa el IGM?",
    ],
    "hablar_humano": [
        "quiero hablar con una persona", "comunícame con un analista de geomática",
        "necesito hablar con alguien del área", "pásame con un humano",
        "¿puedo hablar con el jefe de geomática?", "prefiero que me atienda una persona",
        "quiero hablar con un operador", "contactar a un especialista",
        "no me sirve el bot, quiero un humano", "necesito asesoría personalizada",
        "¿me puede llamar alguien?", "deriva mi caso a un técnico", "hablar con soporte humano",
        "quiero que un ingeniero revise esto", "¿hay alguien de carne y hueso?",
        "escálalo a un supervisor", "agente humano por favor",
        "necesito que me contacte el responsable de catastro", "quiero hablar con el área legal",
        "transfiéreme con una persona", "esto lo tiene que ver un especialista",
        "¿con quién puedo hablar directamente?", "necesito atención de un analista",
        "comunícame con alguien de seguridad", "quiero escalar mi caso",
        "no entiendes, pásame con alguien",
    ],
    "emergencia_seguridad": [
        "hay gente armada en la concesión", "escuchamos disparos cerca del campamento",
        "nos están amenazando los invasores", "hay un herido en el frente de trabajo",
        "emergencia, retuvieron al equipo de topografía", "hay una balacera en el lindero",
        "los invasores tienen armas", "secuestraron a un compañero",
        "se derrumbó un túnel y hay personas atrapadas", "nos atacaron con piedras en la vía",
        "hay un enfrentamiento con mineros ilegales", "el equipo de campo está en peligro",
        "emergencia en el campamento", "necesito ayuda urgente, hay gente armada",
        "dispararon contra el dron", "nos rodearon personas encapuchadas",
        "tuvimos un accidente grave en la vía", "hay un incendio en el campamento",
        "amenazaron de muerte al vigilante", "conflicto con la comunidad, hay violencia",
        "hay heridos tras una explosión", "auxilio, nos robaron el equipo a mano armada",
        "situación de riesgo para el personal", "quemaron la garita de seguridad",
    ],
    "fuera_alcance": [
        "¿cuál es el precio del oro hoy?", "¿quién ganó el partido de anoche?", "cuéntame un chiste",
        "¿cómo hago una torta de chocolate?", "¿cuánto es mi sueldo?", "¿cuándo pagan las utilidades?",
        "recomiéndame una película", "¿qué hora es en Tokio?", "escribe un poema",
        "¿cómo está el clima en Guayaquil?", "¿quién es el presidente de Francia?",
        "ayúdame con mi tarea de matemáticas", "¿cuántos días de vacaciones me quedan?",
        "tradúceme esto al inglés", "¿qué opinas de la política?",
        "reserva un vuelo comercial a Quito", "¿dónde queda el restaurante más cercano?",
        "¿cómo invierto en bolsa?", "pon música", "¿cuál es la capital de Australia?",
        "necesito un certificado laboral", "¿cómo reseteo mi contraseña de correo?",
        "quiero pedir un taxi", "¿me prestas dinero?", "¿cuál es tu color favorito?",
        "¿cómo arreglo mi impresora?", "¿cuándo es el próximo feriado?",
        "¿qué ley regula el tránsito?", "dame la receta del encebollado", "¿cuánto cuesta un iPhone?",
        "¿quién descubrió América?", "calcula mi declaración de impuestos",
    ],
}

PRUEBA: dict[str, list[str]] = {
    "saludo": [
        "buenos dias kuribot", "hola hola", "muy buenas tardes", "holaaa buen dia",
        "saludos desde el campamento sur", "hola q tal", "buenas, ¿todo bien?",
        "hola, primer día usando esto", "buen día compañero", "ey hola", "buenas noches asistente",
        "hola, ¿estás disponible?", "saludos, soy nuevo en geomática", "qué hubo, buenas",
        "hola de parte de seguridad", "buenas tardes a todos", "hola otra vez",
        "buen día, ¿qué tal la mañana?",
    ],
    "despedida": [
        "grax", "muchas gracias por todo", "ya, chao", "nos vemos mañana", "listo eso era, bye",
        "gracias crack", "ok, hasta otra", "muy amable, me retiro", "agradecido con la ayuda",
        "adios kuribot", "ya terminé, gracias", "perfecto mil gracias", "hasta la vista",
        "eso sería todo", "buenas noches, ya me voy", "chévere, gracias", "listo, cerramos",
        "gracias, quedo atento",
    ],
    "ayuda_capacidades": [
        "q puedes hacer", "ayudaaa", "¿cuál es tu alcance?", "enséñame a usarte",
        "¿qué servicios tienes?", "opciones porfa", "¿tienes un menú?", "¿qué consultas aceptas?",
        "soy nuevo, ¿cómo funcionas?", "¿para qué te crearon?", "dame las opciones del asistente",
        "¿qué puedo pedirte?", "mostrar ayuda", "¿en qué temas eres bueno?", "¿cómo se usa esto?",
        "quiero ver qué haces", "¿qué funciones tiene el bot?", "explica tus funciones",
    ],
    "consultar_concesion": [
        "estado de {conc} porfa", "{conc} sigue vigente?", "ficha de {conc}",
        "necesito la info catastral de la {conc}", "¿cuánto mide la concesión {conc}?",
        "¿la {conc} está suspendida?", "revisame la concesion {conc}", "¿qué fase tiene la {conc}?",
        "quiero ver los datos del área {conc}", "titular y estado de {conc}",
        "dime si {conc} está activa", "¿cuál es la superficie concesionada de {conc}?",
        "datos del derecho minero {conc}", "consultar concesion",
        "¿a qué campamento corresponde {conc}?", "situación actual de la concesión {conc}",
        "resumen catastral de {conc}", "¿{conc} está en trámite?",
    ],
    "verificar_punto": [
        "{coord} esta dentro?", "¿ese punto {coord} es nuestro?", "chequea {coord} contra el catastro",
        "¿la coordenada {coord} cae en la concesión o fuera?", "dónde queda {coord}",
        "¿{coord} invade alguna concesión de terceros?", "verifica ubicación del hito {coord}",
        "¿a cuántos metros del límite está {coord}?", "confirma si {coord} cae en área minera",
        "tengo esta posición {coord}, ¿a qué concesión corresponde?",
        "¿el campamento en {coord} está dentro del lindero?", "revisar si un punto está dentro",
        "ubícame {coord}", "¿{coord} está cerca de un área protegida?",
        "comprobar coordenada en polígono", "¿{coord} queda dentro del área concesionada?",
        "consulta si {coord} pertenece a la empresa", "¿la plataforma en {coord} está en regla?",
    ],
    "convertir_coordenadas": [
        "pasame {coord} a wgs84", "conviérteme {coord}", "{coord} a geograficas",
        "necesito {coord} en psad 56", "transforma de psad a wgs las coordenadas {coord}",
        "¿me das {coord} en lat long?", "reproyectar {coord}", "convertir coordenadas por favor",
        "cambia {coord} a UTM 17S", "¿cuál es el equivalente de {coord} en WGS 84?",
        "de UTM a geográficas: {coord}", "quiero {coord} en grados decimales",
        "transformación de datum para {coord}", "convertir punto a otro sistema de coordenadas",
        "¿cómo se expresa {coord} en PSAD56?", "{coord} a sirgas", "dame {coord} convertido",
        "conversión geodésica de {coord}",
    ],
    "reportar_alerta": [
        "hay mineros ilegales en {conc}", "vieron una excavadora rara en {conc}",
        "alerta! gente trabajando sin permiso en {conc}", "quiero denunciar una invasion",
        "el rio esta turbio y sospechamos de {conc}", "se oyen explosiones de noche por {conc}",
        "apareció un camino nuevo en {conc}", "reportar actividad no autorizada",
        "hay un desbroce raro en la imagen de {conc}", "informales metidos en {conc}",
        "la comunidad avisa de volquetas en {conc}", "encontramos labores clandestinas en {conc}",
        "hay una draga en el río dentro de {conc}", "sospecha de extracción ilegal en {conc}",
        "registrar una intrusión en el lindero", "hay movimiento de tierra no autorizado en {conc}",
        "reporte de invasión minera", "vimos un campamento de informales en {conc}",
    ],
    "solicitar_vuelo": [
        "mandame el dron a {conc}", "ocupo un vuelo en {conc} {fecha}", "agenda drone para {conc}",
        "¿pueden sobrevolar {conc}?", "vuelo de inspeccion {fecha}", "necesito una ortofoto nueva de {conc}",
        "solicitud de sobrevuelo", "¿está libre el dron {fecha}?",
        "programar vuelo fotogramétrico en {conc}", "inspección aérea en {conc} urgente",
        "quiero volar el dron sobre la relavera de {conc}", "reserva un vuelo para {fecha}",
        "vuelo RPAS en {conc}", "necesito imágenes con dron del lindero de {conc}",
        "¿me programan un vuelo?", "sobrevuelo de verificación en {conc} {fecha}",
        "vuelo para cubicar el stock de {conc}", "dron a {conc} porfa",
    ],
    "solicitar_cartografia": [
        "mapa de {conc} porfa", "me envías el kmz de {conc}?", "ocupo el shape de linderos",
        "necesito {producto} de {conc} urgente", "quiero un plano de {conc}", "solicitar mapa topografico",
        "¿me hacen un mapa de ubicación?", "curvas de nivel de {conc}", "dame el {producto} de la {conc}",
        "requiero un mapa para el informe mensual", "necesito la cartografía base del campamento {camp}",
        "exportar polígono de {conc} a kml", "mapa temático de cobertura de {conc}",
        "¿tienen el MDT de {conc}?", "plano en pdf de {conc}", "pedir cartografía nueva",
        "mapa de accesos a {conc}", "necesito la capa de hitos",
    ],
    "estado_solicitud": [
        "como va {ticket}", "estado {ticket}", "¿ya está mi ticket?", "seguimiento {ticket} porfa",
        "¿qué pasó con {ticket}?", "¿ya me atendieron el pedido?", "¿cuándo estará listo {ticket}?",
        "revisar mi solicitud", "novedades del caso {ticket}", "¿mi vuelo ya está programado?",
        "¿{ticket} ya se cerró?", "estatus de mi requerimiento", "¿quién tiene asignado {ticket}?",
        "¿hay avances con {ticket}?", "consultar ticket", "¿en qué va mi pedido de mapa?",
        "quiero saber el estado de {ticket}", "¿ya revisaron mi alerta?",
    ],
    "consulta_geodesia": [
        "psad56 vs wgs84 cual es la diferencia", "¿qué EPSG es WGS84 UTM 17S?",
        "¿cuál es el sistema de referencia oficial del país?", "explícame qué es SIRGAS",
        "¿por qué el punto me sale corrido en el mapa?", "¿qué tan preciso es el gps de mano?",
        "¿qué es RTK?", "¿en qué formato entrego la cartografía?", "¿qué reglas hay para volar dron?",
        "¿cuántos metros cambia de PSAD a WGS?", "¿qué es la zona 17 sur?", "¿qué es un datum?",
        "¿qué datos debe tener un reporte de alerta?", "¿cuánto se demora un mapa?",
        "diferencia entre altura del gps y altura sobre el nivel del mar",
        "¿qué es una ortofoto y para qué sirve?", "¿qué es el código EPSG?",
        "¿cómo identifico el datum de unas coordenadas?",
    ],
    "hablar_humano": [
        "una persona porfa", "quiero un humano", "pásame con tu supervisor",
        "¿me comunicas con geomática?", "necesito un analista real", "derívame con un técnico",
        "que me llame alguien del área", "quiero atención personalizada", "operador por favor",
        "esto no lo resuelves tú, pásame con alguien", "contacto con un especialista en catastro",
        "escalar a un humano", "¿puedo hablar con el ingeniero de turno?",
        "quiero que lo revise una persona", "comunicarme con el área legal",
        "no quiero hablar con un bot", "conéctame con soporte", "necesito hablar con el responsable",
    ],
    "emergencia_seguridad": [
        "gente con armas en la quebrada!!", "se escucharon tiros cerca del campamento",
        "tenemos un herido", "emergencia urgente", "nos quieren agredir", "retuvieron al topógrafo",
        "hay violencia en el lindero", "personas atrapadas en la mina", "dispararon al vigilante",
        "amenazas contra el personal de campo", "nos asaltaron en la vía", "accidente con lesionados",
        "peligro, hay armados", "incendio en la bodega", "nos atacaron", "enfrentamiento en la concesión",
        "un compañero desaparecido en campo", "nos amenazan, auxilio",
    ],
    "fuera_alcance": [
        "precio del cobre en la bolsa de Londres", "¿quién juega hoy en la liga?", "dime algo gracioso",
        "receta de bolón de verde", "¿cuándo me depositan el sueldo?", "¿me recomiendas una serie?",
        "¿cómo estará el tráfico en la vía a Machala?", "escríbeme una canción", "¿qué es la vida?",
        "quiero vacaciones la próxima semana", "traduce hello world", "¿cómo cambio la clave del wifi?",
        "¿cuál es el mejor celular?", "¿cuánto pesa un elefante?", "pídeme comida a domicilio",
        "¿qué feriados hay en diciembre?", "solicitud de anticipo de sueldo", "¿el oro va a subir de precio?",
    ],
}

# Iteración 2 (ajuste tras el análisis de fallos de la suite de aceptación v1, escenario A17): el corpus
# original no tenía reportes de alerta con coordenadas, de modo que el modelo asociaba «COORDENADA» con
# verificar_punto. Estas plantillas se agregan solo al conjunto de entrenamiento y validación.
POOL_ITERACION_2: dict[str, list[str]] = {
    "reportar_alerta": [
        "hay maquinaria trabajando en {coord}", "reporto actividad ilegal en las coordenadas {coord}",
        "vimos gente excavando en el punto {coord}", "alerta de invasión en {coord}",
        "se observa desbroce nuevo cerca de {coord}", "encontramos una trocha nueva en {coord}",
        "hay una bocamina clandestina en {coord}", "el agua sale turbia desde {coord}, parece minería ilegal",
    ],
}

# Valores para sustituir marcadores. Las concesiones y tickets existen en la base sintética.
VALORES = {
    "conc": ["CM-101", "CM-102", "CM-103", "CM-104", "CM-105", "CM-106", "Peña Blanca",
             "San Gregorio", "La Ventana", "El Roble", "Cerro Azul", "Los Laureles",
             "peña blanca", "la ventana", "el roble", "san gregorio", "cerro azul", "los laureles"],
    "coord": ["E 642350 N 9662120", "642350, 9662120", "x=644800 y=9661500", "646900 9660300",
              "E 641100 N 9664950 WGS84", "E 642600 N 9662480 PSAD56", "-3.0512, -79.7301",
              "-3.0601 -79.7215", "3°03'04\"S 79°43'48\"W", "lat -3.0455 lon -79.7262",
              "E: 645210, N: 9663790", "643990 9661010 psad56", "640500 9665200 wgs84"],
    "ticket": ["GEO-2026-0412", "GEO-2026-0415", "GEO-2026-0419", "GEO-2026-0423", "GEO-2026-0430"],
    "producto": ["mapa topográfico", "ortofoto", "KMZ", "shapefile", "modelo digital de elevación",
                 "plano de linderos", "mapa de ubicación", "mapa de accesos"],
    "fecha": ["mañana", "hoy", "el lunes", "el 12 de noviembre", "pasado mañana", "esta semana",
              "el viernes"],
    "camp": ["Norte", "Central", "Sur"],
}


In [ ]:
%%writefile datos/construir_base.py
"""Genera la base de conocimiento SINTÉTICA de KuriBot (concesiones, operación, FAQ, tickets).

Todo lo que se escribe aquí es ficticio: nombres, códigos, polígonos, clima, flota y tickets
se construyeron para el prototipo académico. Las respuestas de la FAQ geodésica se limitan a
hechos verificables en la base EPSG (vía pyproj) o a definiciones generales; las políticas
operativas (umbral de viento, plazos, formatos) son supuestos de diseño del equipo.

Uso: python datos/construir_base.py
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ))
from kuribot.geodesia import descripcion_transformacion, psad56_utm_a_wgs84_utm  # noqa: E402

DATOS = RAIZ / "datos"


def rect(e0, n0, e1, n1):
    return [[e0, n0], [e1, n0], [e1, n1], [e0, n1], [e0, n0]]


CONCESIONES = [
    # codigo, nombre, fase, estado, campamento, polígono UTM17S WGS84, titular
    ("CM-101", "Peña Blanca", "explotación", "vigente", "Norte", rect(641000, 9661500, 643000, 9663000), "propia"),
    ("CM-102", "San Gregorio", "explotación", "vigente", "Norte", rect(643000, 9662000, 644600, 9663500), "propia"),
    ("CM-103", "La Ventana", "exploración avanzada", "vigente", "Central", rect(644600, 9660500, 647000, 9662500), "propia"),
    ("CM-104", "El Roble", "exploración inicial", "vigente (renovación en trámite)", "Central",
     [[640000, 9664000], [641000, 9664000], [641000, 9665400], [640600, 9666000], [640000, 9666000], [640000, 9664000]], "propia"),
    ("CM-105", "Cerro Azul", "explotación", "suspendida temporalmente", "Sur", rect(645500, 9663000, 646500, 9664200), "propia"),
    ("CM-106", "Los Laureles", "exploración inicial", "vigente", "Sur", rect(646500, 9658500, 648500, 9660500), "propia"),
    ("CT-201", "Minas del Valle", "explotación", "vigente", "—", rect(641000, 9663000, 643000, 9664200), "tercero"),
]
AREA_PROTEGIDA = ("BP-01", "Bosque Protector El Mirador (ficticio)", rect(647000, 9660500, 649000, 9662500))
COMUNIDADES = [("COM-01", "Comunidad San José (ficticia)", [644200, 9660800])]

OPERACION = {
    "fecha_referencia": "2026-11-09",
    "nota": "Datos operativos sintéticos para la demostración; no proceden de ninguna empresa.",
    "historial_alertas_90d": {"CM-101": 0, "CM-102": 1, "CM-103": 0, "CM-104": 2, "CM-105": 3, "CM-106": 1, "CT-201": 0},
    "actividad_autorizada_programada": {
        "CM-101": "frente de explotación activo con maquinaria propia",
        "CM-102": "mantenimiento de vías con retroexcavadora propia",
        "CM-103": "campaña de sondajes con perforadora contratista",
    },
    "pronostico": {  # día relativo a fecha_referencia: [viento m/s, lluvia mm/h, visibilidad km]
        "Norte":   {"0": [4.2, 0.0, 10], "1": [12.5, 0.0, 8], "2": [6.0, 0.0, 10], "3": [5.1, 2.4, 4], "4": [3.8, 0.0, 10]},
        "Central": {"0": [7.5, 1.2, 5], "1": [5.0, 0.0, 10], "2": [4.4, 0.0, 10], "3": [11.0, 0.0, 9], "4": [6.2, 0.0, 10]},
        "Sur":     {"0": [3.1, 0.0, 10], "1": [8.9, 0.0, 10], "2": [10.8, 0.5, 6], "3": [4.0, 0.0, 10], "4": [4.5, 0.0, 10]},
    },
    "drones": {"DRN-01": {"tipo": "multirrotor RTK", "no_disponible": ["2"]},
               "DRN-02": {"tipo": "ala fija VTOL", "no_disponible": ["0", "1"]}},
    "pilotos": {"Piloto A": {"no_disponible": ["3"]}, "Piloto B": {"no_disponible": ["0"]}},
    "politica_vuelo": {"viento_max_ms": 10.0, "lluvia_max_mmh": 0.0, "visibilidad_min_km": 5.0,
                       "anticipacion_min_dias": 1, "horizonte_dias": 4},
    "sla_cartografia_dias_habiles": {"estandar": 3, "urgente": 1},
}

TICKETS = [
    {"id": "GEO-2026-0412", "tipo": "cartografía", "detalle": "Mapa topográfico de CM-103", "estado": "En elaboración",
     "asignado": "Analista SIG 1", "compromiso": "2026-11-10"},
    {"id": "GEO-2026-0415", "tipo": "vuelo", "detalle": "Vuelo de inspección en CM-101", "estado": "Programado",
     "asignado": "Piloto A", "compromiso": "2026-11-10"},
    {"id": "GEO-2026-0419", "tipo": "alerta", "detalle": "Desbroce detectado en CM-105", "estado": "Cerrada",
     "asignado": "Seguridad patrimonial", "compromiso": "2026-11-03",
     "resolucion": "Verificada en campo: trabajos de remediación autorizados"},
    {"id": "GEO-2026-0423", "tipo": "cartografía", "detalle": "KMZ de linderos de CM-102", "estado": "Entregado",
     "asignado": "Analista SIG 2", "compromiso": "2026-11-05"},
    {"id": "GEO-2026-0430", "tipo": "alerta", "detalle": "Turbidez aguas abajo de CM-106", "estado": "En verificación de campo",
     "asignado": "Seguridad patrimonial", "compromiso": "2026-11-11"},
]


def faq() -> list[dict]:
    e, n = 642600.0, 9662480.0
    e2, n2 = psad56_utm_a_wgs84_utm(e, n)
    de, dn = e2 - e, n2 - n
    d = (de ** 2 + dn ** 2) ** 0.5
    desplaz = (f"En el área del proyecto, la transformación {descripcion_transformacion()} desplaza un punto "
               f"aproximadamente {d:.0f} m (ΔE ≈ {de:.0f} m, ΔN ≈ {dn:.0f} m al pasar de PSAD56 a WGS 84).")
    return [
        {"id": "F01", "tema": "datum", "preguntas": ["diferencia entre PSAD56 y WGS84", "psad56 vs wgs84", "qué es PSAD56"],
         "respuesta": "PSAD56 (Datum Provisional Sudamericano de 1956) es un datum local basado en el elipsoide Internacional "
                      "de 1924 con punto fundamental en La Canoa (Venezuela); WGS 84 es un sistema geocéntrico global. Por eso "
                      "un mismo punto tiene coordenadas distintas en cada uno. " + desplaz, "fuente": "Base EPSG (pyproj)"},
        {"id": "F02", "tema": "epsg", "preguntas": ["código EPSG UTM 17 sur", "EPSG de PSAD56 UTM 17S", "EPSG WGS84 geográficas",
                                                      "qué EPSG uso"],
         "respuesta": "Códigos de uso frecuente en el proyecto: EPSG:4326 (WGS 84 geográficas), EPSG:32717 (WGS 84 / UTM zona 17S), "
                      "EPSG:24877 (PSAD56 / UTM zona 17S), EPSG:4248 (PSAD56 geográficas) y EPSG:31977 (SIRGAS 2000 / UTM zona 17S).",
         "fuente": "Base EPSG (pyproj)"},
        {"id": "F03", "tema": "marco_oficial", "preguntas": ["datum oficial en Ecuador", "sistema de referencia oficial del país",
                                                              "marco de referencia del IGM"],
         "respuesta": "El marco geodésico nacional es SIRGAS-ECU, la densificación de SIRGAS en el Ecuador a cargo del Instituto "
                      "Geográfico Militar (IGM). La realización y la época vigentes deben tomarse de la resolución vigente del IGM; "
                      "consulte con el analista de geodesia antes de entregar información con fines legales.",
         "fuente": "Política interna (verificar normativa IGM vigente)"},
        {"id": "F04", "tema": "sirgas", "preguntas": ["qué es SIRGAS", "época de referencia en SIRGAS"],
         "respuesta": "SIRGAS (Sistema de Referencia Geocéntrico para las Américas) es la densificación regional del Marco "
                      "Internacional de Referencia Terrestre (ITRF). Como las placas tectónicas se mueven, las coordenadas SIRGAS se "
                      "refieren a una época determinada; comparar coordenadas de épocas distintas exige modelar ese movimiento.",
         "fuente": "Definición general"},
        {"id": "F05", "tema": "zona_utm", "preguntas": ["en qué zona UTM está el proyecto", "qué es la zona 17 sur"],
         "respuesta": "El área del proyecto (alrededor de 79,7° O) está en la zona UTM 17 (de 84° O a 78° O, meridiano central 81° O), "
                      "hemisferio sur. Al este de 78° O comienza la zona 18, por lo que parte de la Amazonía cae en 18S.",
         "fuente": "Definición del sistema UTM"},
        {"id": "F06", "tema": "no_coinciden", "preguntas": ["mis coordenadas no coinciden con el catastro", "el punto sale corrido en el mapa"],
         "respuesta": "Las causas más comunes son: (1) datum distinto, PSAD56 frente a WGS 84 (en esta zona la diferencia es de unos "
                      "cientos de metros); (2) zona UTM equivocada; (3) Este y Norte invertidos; (4) error propio del receptor GNSS. "
                      "Indique el datum de origen y puedo convertir o verificar el punto.", "fuente": "Práctica geodésica"},
        {"id": "F07", "tema": "precision_gnss", "preguntas": ["precisión de un GPS navegador", "diferencia entre RTK y GPS de mano", "qué es RTK"],
         "respuesta": "Un receptor de navegación suele ofrecer exactitudes horizontales del orden de unos metros, y empeora bajo "
                      "cobertura vegetal. El posicionamiento RTK usa correcciones de una base en tiempo real y, con solución fija, "
                      "alcanza exactitud del orden de centímetros. Para linderos y control use GNSS geodésico.",
         "fuente": "Práctica geodésica"},
        {"id": "F08", "tema": "formatos", "preguntas": ["formato de archivo para entregar", "en qué formato entrego la cartografía"],
         "respuesta": "Política interna (supuesto del prototipo): entregar vectores en GeoPackage o Shapefile en EPSG:32717, ráster en "
                      "GeoTIFF, KMZ solo para visualización y mapas en PDF con cuadrícula, escala y metadatos.", "fuente": "Política interna"},
        {"id": "F09", "tema": "politica_vuelo", "preguntas": ["política para vuelos de dron", "reglas para volar dron",
                                                               "anticipación para pedir un vuelo"],
         "respuesta": "Política interna (supuesto del prototipo): vuelos diurnos con línea de vista, viento ≤ 10 m/s, sin lluvia, "
                      "visibilidad ≥ 5 km, piloto habilitado y solicitud con al menos un día de anticipación, salvo alertas de prioridad "
                      "CRÍTICA o ALTA. Toda operación debe cumplir la normativa aeronáutica vigente.", "fuente": "Política interna"},
        {"id": "F10", "tema": "desplazamiento", "preguntas": ["cuánto se desplaza un punto de PSAD56 a WGS84",
                                                               "cuántos metros cambia de PSAD a WGS"],
         "respuesta": desplaz, "fuente": "Cálculo con pyproj"},
        {"id": "F11", "tema": "utm", "preguntas": ["qué es una proyección UTM", "qué es el falso norte"],
         "respuesta": "UTM es una proyección transversa de Mercator dividida en 60 zonas de 6°. Usa factor de escala 0,9996 en el "
                      "meridiano central, falso este de 500 000 m y, en el hemisferio sur, falso norte de 10 000 000 m para evitar "
                      "coordenadas negativas.", "fuente": "Definición del sistema UTM"},
        {"id": "F12", "tema": "datum_def", "preguntas": ["qué es un datum geodésico", "qué es un datum"],
         "respuesta": "Un datum geodésico define la forma y el tamaño de la Tierra (un elipsoide) y su posición respecto del planeta; "
                      "sobre él se expresan las coordenadas. Cambiar de datum cambia los números aunque el punto físico sea el mismo.",
         "fuente": "Definición general"},
        {"id": "F13", "tema": "reporte_alerta", "preguntas": ["información para reportar una alerta", "datos de un reporte de alerta"],
         "respuesta": "Indique la concesión o las coordenadas (con su datum), qué observó (maquinaria, desbroce, agua turbia, ruido "
                      "nocturno, vía nueva), la fecha y la fuente. No incluya datos personales de terceros. Si hay riesgo para "
                      "personas, escriba «emergencia».", "fuente": "Procedimiento interno"},
        {"id": "F14", "tema": "sla_mapa", "preguntas": ["cuánto tarda una solicitud de mapa", "cuánto se demora un mapa"],
         "respuesta": "Plazo interno (supuesto del prototipo): 3 días hábiles para solicitudes estándar y 1 día hábil para urgentes "
                      "aprobadas por la jefatura.", "fuente": "Política interna"},
        {"id": "F15", "tema": "epsg_def", "preguntas": ["qué significa EPSG", "qué es el código EPSG"],
         "respuesta": "EPSG es la base de datos de parámetros geodésicos creada por el European Petroleum Survey Group y mantenida hoy "
                      "por el Comité de Geomática de IOGP. Cada sistema de referencia y cada transformación tiene un código único.",
         "fuente": "IOGP"},
        {"id": "F16", "tema": "ortofoto", "preguntas": ["qué es una ortofoto", "para qué sirve una ortofoto"],
         "respuesta": "Una ortofoto es una imagen aérea corregida por el relieve y la inclinación de la cámara, de modo que tiene "
                      "escala uniforme y permite medir distancias y áreas.", "fuente": "Definición general"},
        {"id": "F17", "tema": "mdt", "preguntas": ["qué es un modelo digital de terreno"],
         "respuesta": "Un modelo digital del terreno (MDT) representa la elevación del suelo desnudo; el modelo digital de superficie "
                      "incluye además vegetación y construcciones.", "fuente": "Definición general"},
        {"id": "F18", "tema": "alturas", "preguntas": ["diferencia entre altura elipsoidal y ortométrica",
                                                        "altura del gps y altura sobre el nivel del mar"],
         "respuesta": "El GNSS entrega altura elipsoidal (h). La altura ortométrica (H), cercana a la altura sobre el nivel medio del "
                      "mar, se obtiene con la ondulación del geoide (N): H = h − N, usando un modelo geoidal.", "fuente": "Definición general"},
        {"id": "F19", "tema": "identificar_datum", "preguntas": ["cómo sé en qué datum están mis coordenadas", "cómo identifico el datum"],
         "respuesta": "Revise los metadatos o la configuración del equipo. Si no existen, compare con un punto de control conocido: en "
                      "esta zona, una diferencia sistemática de unos 440 m respecto del catastro en WGS 84 sugiere coordenadas en PSAD56.",
         "fuente": "Práctica geodésica"},
    ]


def main() -> None:
    feats = []
    for cod, nom, fase, est, camp, pol, tit in CONCESIONES:
        feats.append({"type": "Feature", "properties": {"codigo": cod, "nombre": nom, "fase": fase, "estado": est,
                                                         "campamento": camp, "titular": tit, "capa": "concesion"},
                      "geometry": {"type": "Polygon", "coordinates": [pol]}})
    cod, nom, pol = AREA_PROTEGIDA
    feats.append({"type": "Feature", "properties": {"codigo": cod, "nombre": nom, "capa": "area_protegida"},
                  "geometry": {"type": "Polygon", "coordinates": [pol]}})
    for cod, nom, pt in COMUNIDADES:
        feats.append({"type": "Feature", "properties": {"codigo": cod, "nombre": nom, "capa": "comunidad"},
                      "geometry": {"type": "Point", "coordinates": pt}})
    geo = {"type": "FeatureCollection", "name": "base_sintetica_kuribot",
           "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:EPSG::32717"}}, "features": feats}
    (DATOS / "concesiones.geojson").write_text(json.dumps(geo, ensure_ascii=False, indent=1), encoding="utf-8")
    (DATOS / "operacion.json").write_text(json.dumps(OPERACION, ensure_ascii=False, indent=2), encoding="utf-8")
    (DATOS / "faq.json").write_text(json.dumps(faq(), ensure_ascii=False, indent=2), encoding="utf-8")
    (DATOS / "tickets_semilla.json").write_text(json.dumps(TICKETS, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Base sintética escrita en", DATOS)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile datos/construir_corpus.py
"""Construye corpus.json a partir de corpus_fuente.py (semilla fija, sin fuga entre particiones).

- POOL: cada plantilla con marcadores se instancia 3 veces con valores distintos; sin
  marcadores, una vez. Todas las instancias de una plantilla comparten «grupo», de modo que
  la validación cruzada (StratifiedGroupKFold) nunca separa variantes de la misma plantilla.
- PRUEBA: plantillas redactadas aparte, instanciadas una vez («prueba_limpia»).
- PRUEBA CON RUIDO: la misma prueba con perturbaciones ortográficas sembradas (sin tildes,
  minúsculas, abreviaturas de chat y un error de tecleo por frase con probabilidad 0,7).
  El ruido se aplica a la plantilla antes de insertar entidades, para no corromper códigos.

Uso: python datos/construir_corpus.py
"""
from __future__ import annotations

import json
import random
import re
import sys
import unicodedata
from pathlib import Path

RAIZ = Path(__file__).resolve().parent
sys.path.insert(0, str(RAIZ))
from corpus_fuente import POOL, POOL_ITERACION_2, PRUEBA, VALORES  # noqa: E402

SEMILLA = 2026
MARCADOR = re.compile(r"\{(\w+)\}")
VECINAS = {"a": "sq", "e": "wr", "i": "uo", "o": "ip", "u": "yi", "s": "ad", "n": "bm", "r": "et",
           "t": "ry", "l": "kñ", "c": "xv", "d": "sf", "m": "n", "p": "o"}


def instanciar(plantilla: str, rng: random.Random) -> str:
    return MARCADOR.sub(lambda m: rng.choice(VALORES[m.group(1)]), plantilla)


def sin_tildes(t: str) -> str:
    nfkd = unicodedata.normalize("NFD", t)
    return "".join(c for c in nfkd if unicodedata.category(c) != "Mn" or c == "̃").replace("ñ", "ñ")


def error_tecleo(palabra: str, rng: random.Random) -> str:
    i = rng.randrange(1, len(palabra) - 1)
    op = rng.choice(["intercambio", "omision", "duplicado", "vecina"])
    if op == "intercambio":
        return palabra[:i] + palabra[i + 1] + palabra[i] + palabra[i + 2:]
    if op == "omision":
        return palabra[:i] + palabra[i + 1:]
    if op == "duplicado":
        return palabra[:i] + palabra[i] + palabra[i:]
    c = palabra[i]
    return palabra[:i] + (rng.choice(VECINAS[c]) if c in VECINAS else c) + palabra[i + 1:]


def ruido(plantilla: str, rng: random.Random) -> str:
    t = plantilla.lower()
    t = re.sub(r"[¿¡]", "", t)
    if rng.random() < 0.8:
        t = sin_tildes(t)
    for patron, reemplazo, p in [(r"\bque\b", "q", 0.5), (r"\bpor favor\b", "porfa", 0.7),
                                 (r"\bpara\b", "pa", 0.3), (r"\bestá\b|\besta\b", "sta", 0.2)]:
        if rng.random() < p:
            t = re.sub(patron, reemplazo, t)
    if rng.random() < 0.7:
        tokens = t.split(" ")
        idx = [i for i, w in enumerate(tokens) if len(w) >= 5 and w.isalpha()]
        if idx:
            j = rng.choice(idx)
            tokens[j] = error_tecleo(tokens[j], rng)
        t = " ".join(tokens)
    return t


def main() -> None:
    rng = random.Random(SEMILLA)
    filas = []
    for intencion, plantillas in POOL.items():
        for k, pl in enumerate(plantillas):
            n = 3 if MARCADOR.search(pl) else 1
            vistos = set()
            for _ in range(n * 4):
                texto = instanciar(pl, rng)
                if texto not in vistos:
                    vistos.add(texto)
                if len(vistos) == n:
                    break
            for texto in sorted(vistos):
                filas.append({"texto": texto, "intencion": intencion, "particion": "pool",
                              "grupo": f"{intencion}:{k:02d}"})
    for intencion, plantillas in PRUEBA.items():
        for k, pl in enumerate(plantillas):
            filas.append({"texto": instanciar(pl, rng), "intencion": intencion, "particion": "prueba_limpia",
                          "grupo": f"T-{intencion}:{k:02d}"})
            filas.append({"texto": instanciar(ruido(pl, rng), rng), "intencion": intencion,
                          "particion": "prueba_ruido", "grupo": f"T-{intencion}:{k:02d}"})
    # Plantillas agregadas en la iteración 2, con un generador propio para no alterar las particiones previas.
    rng2 = random.Random(SEMILLA + 2)
    for intencion, plantillas in POOL_ITERACION_2.items():
        for k, pl in enumerate(plantillas):
            vistos = set()
            for _ in range(12):
                vistos.add(instanciar(pl, rng2))
                if len(vistos) == 3:
                    break
            for texto in sorted(vistos):
                filas.append({"texto": texto, "intencion": intencion, "particion": "pool",
                              "grupo": f"{intencion}:it2-{k:02d}", "iteracion": 2})
    for f in filas:
        f.setdefault("iteracion", 1)
    # Control de fuga: ninguna frase de prueba puede aparecer en el pool.
    pool_textos = {f["texto"].lower() for f in filas if f["particion"] == "pool"}
    fugas = [f["texto"] for f in filas if f["particion"] != "pool" and f["texto"].lower() in pool_textos]
    meta = {"semilla": SEMILLA, "intenciones": sorted(POOL), "fugas_exactas_detectadas": fugas,
            "conteos": {p: sum(1 for f in filas if f["particion"] == p) for p in ("pool", "prueba_limpia", "prueba_ruido")},
            "plantillas_pool": sum(len(v) for v in POOL.values()),
            "plantillas_iteracion_2": sum(len(v) for v in POOL_ITERACION_2.values()),
            "pool_iteracion_2": sum(1 for f in filas if f.get("iteracion") == 2)}
    (RAIZ / "corpus.json").write_text(json.dumps({"meta": meta, "datos": filas}, ensure_ascii=False, indent=1),
                                      encoding="utf-8")
    print(json.dumps(meta, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/extraer_beto.py
"""Extrae el codificador BETO del pipeline oficial de spaCy y lo guarda en formato Hugging Face.

Motivo: el entorno de desarrollo no tiene acceso a huggingface.co, pero sí a las versiones
publicadas de los modelos de spaCy en GitHub. El pipeline ``es_dep_news_trf`` 3.8.0 incluye
el transformer BETO (dccuchile/bert-base-spanish-wwm-cased; Cañete et al., 2020), ajustado
por Explosion durante el entrenamiento del analizador morfosintáctico de spaCy.

El script:
1. lee el archivo serializado ``transformer/model`` (msgpack de Thinc);
2. convierte los nombres de parámetros de curated-transformers a ``BertModel`` de HF
   (la proyección fusionada QKV se divide en query, key y value, en ese orden);
3. guarda config.json, vocab.txt, tokenizer y model.safetensors;
4. con ``--verificar`` compara las salidas contra la implementación original de
   curated-transformers 0.1.1 (diferencia absoluta máxima esperada < 1e-4).

Uso:
    pip install es_dep_news_trf-3.8.0-py3-none-any.whl --no-deps
    python scripts/extraer_beto.py --salida modelos/beto_base [--verificar RUTA_CT011]
"""
from __future__ import annotations

import argparse
import importlib.util
import io
import json
from pathlib import Path

import srsly
import torch


def ubicar_modelo_spacy() -> Path:
    spec = importlib.util.find_spec("es_dep_news_trf")
    if spec is None or spec.origin is None:
        raise SystemExit("Instale primero es_dep_news_trf-3.8.0 (GitHub explosion/spacy-models).")
    raiz = Path(spec.origin).parent
    candidatos = list(raiz.glob("es_dep_news_trf-*/transformer/model"))
    if not candidatos:
        raise SystemExit("No se encontró transformer/model dentro del paquete.")
    return candidatos[0]


def leer_pesos(ruta: Path) -> tuple[dict[str, torch.Tensor], list[str]]:
    datos = srsly.msgpack_loads(ruta.read_bytes())
    shim = srsly.msgpack_loads(datos["shims"][6][0])
    estado = torch.load(io.BytesIO(shim["state"]), map_location="cpu", weights_only=False)
    vocab = datos["attrs"][3]["wordpiece_processor"].decode("utf-8").split("\n")
    return estado, vocab


def convertir(estado: dict[str, torch.Tensor], n_capas: int = 12) -> dict[str, torch.Tensor]:
    pref = "curated_encoder."
    hf: dict[str, torch.Tensor] = {}
    emb = {
        "embeddings.word_embeddings.weight": "embeddings.word_embeddings.weight",
        "embeddings.token_type_embeddings.weight": "embeddings.token_type_embeddings.weight",
        "embeddings.position_embeddings.weight": "embeddings.position_embeddings.weight",
        "embeddings.LayerNorm.weight": "embeddings.layer_norm.weight",
        "embeddings.LayerNorm.bias": "embeddings.layer_norm.bias",
    }
    for destino, origen in emb.items():
        hf[destino] = estado[pref + origen]
    for i in range(n_capas):
        o = f"{pref}layers.{i}."
        d = f"encoder.layer.{i}."
        w_qkv, b_qkv = estado[o + "mha.input.weight"], estado[o + "mha.input.bias"]
        for nombre, w, b in zip(("query", "key", "value"), w_qkv.chunk(3, dim=0), b_qkv.chunk(3, dim=0)):
            hf[d + f"attention.self.{nombre}.weight"] = w.clone()
            hf[d + f"attention.self.{nombre}.bias"] = b.clone()
        pares = {
            "attention.output.dense": "mha.output",
            "attention.output.LayerNorm": "attn_output_layernorm",
            "intermediate.dense": "ffn.intermediate",
            "output.dense": "ffn.output",
            "output.LayerNorm": "ffn_output_layernorm",
        }
        for destino, origen in pares.items():
            hf[d + destino + ".weight"] = estado[o + origen + ".weight"]
            hf[d + destino + ".bias"] = estado[o + origen + ".bias"]
    return hf


def guardar(hf_estado: dict[str, torch.Tensor], vocab: list[str], salida: Path) -> None:
    from transformers import BertConfig, BertModel, BertTokenizerFast

    salida.mkdir(parents=True, exist_ok=True)
    (salida / "vocab.txt").write_text("\n".join(vocab) + "\n", encoding="utf-8")
    config = BertConfig(
        vocab_size=len(vocab), hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
        intermediate_size=3072, hidden_act="gelu", hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1, max_position_embeddings=512, type_vocab_size=2,
        layer_norm_eps=1e-12, pad_token_id=vocab.index("[PAD]"),
    )
    modelo = BertModel(config, add_pooling_layer=False)
    faltan, sobran = modelo.load_state_dict(hf_estado, strict=False)
    faltan = [k for k in faltan if "position_ids" not in k]
    if faltan or sobran:
        raise RuntimeError(f"Conversión incompleta. Faltan: {faltan}; sobran: {sobran}")
    modelo.save_pretrained(salida)
    import inspect
    opciones = dict(do_lower_case=False, strip_accents=False, tokenize_chinese_chars=True)
    if "vocab" in inspect.signature(BertTokenizerFast.__init__).parameters:   # transformers >= 5
        tok = BertTokenizerFast(vocab={p: i for i, p in enumerate(vocab)}, **opciones)
    else:                                                                       # transformers 4.x
        tok = BertTokenizerFast(vocab_file=str(salida / "vocab.txt"), **opciones)
    prueba = tok("Hay maquinaria cerca del lindero de la concesión")["input_ids"]
    if tok.unk_token_id in prueba or tok.cls_token_id != vocab.index("[CLS]"):
        raise RuntimeError("El tokenizador no reproduce el vocabulario de BETO.")
    tok.save_pretrained(salida)
    (salida / "PROCEDENCIA.json").write_text(json.dumps({
        "origen": "spaCy es_dep_news_trf 3.8.0 (Explosion), componente transformer",
        "modelo_base": "dccuchile/bert-base-spanish-wwm-cased (BETO; Cañete et al., 2020)",
        "nota": "Pesos ajustados por spaCy para análisis morfosintáctico (UD AnCora); "
                "no son idénticos al BETO original publicado en Hugging Face.",
        "licencia_pipeline": "GNU GPL 3.0 (por los datos AnCora del pipeline)",
    }, ensure_ascii=False, indent=2), encoding="utf-8")


def verificar(estado, salida: Path, ruta_ct011: str) -> float:
    """Compara la salida HF con la implementación original curated-transformers 0.1.1."""
    import sys
    from transformers import BertModel, BertTokenizerFast

    sys.path.insert(0, ruta_ct011)
    from curated_transformers.models.bert.config import BertConfig as CTConfig  # type: ignore
    from curated_transformers.models.bert.encoder import BertEncoder  # type: ignore
    from curated_transformers.models.attention import AttentionMask  # type: ignore

    cfg = CTConfig(vocab_size=31002, hidden_width=768, intermediate_width=3072,
                   num_attention_heads=12, num_hidden_layers=12, hidden_act="gelu",
                   layer_norm_eps=1e-12, max_position_embeddings=512, type_vocab_size=2,
                   padding_idx=1, attention_probs_dropout_prob=0.0, hidden_dropout_prob=0.0)
    ct = BertEncoder(cfg)
    ct.load_state_dict({k.replace("curated_encoder.", ""): v for k, v in estado.items()})
    ct.eval()
    hf = BertModel.from_pretrained(salida, add_pooling_layer=False).eval()
    tok = BertTokenizerFast.from_pretrained(salida)
    frases = ["Necesito convertir coordenadas de PSAD56 a WGS84.",
              "Hay maquinaria no autorizada cerca del lindero de la concesión."]
    enc = tok(frases, padding=True, return_tensors="pt")
    with torch.no_grad():
        salida_hf = hf(**enc).last_hidden_state
        mascara = AttentionMask(enc["attention_mask"].bool())
        salida_ct = ct(enc["input_ids"], mascara, token_type_ids=enc["token_type_ids"])
        ultimo = salida_ct.last_hidden_layer_states
        salida_ct = ultimo() if callable(ultimo) else ultimo
    m = enc["attention_mask"].bool()
    return float((salida_hf[m] - salida_ct[m]).abs().max())


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--salida", default="modelos/beto_base")
    ap.add_argument("--verificar", default=None, help="Ruta a curated-transformers 0.1.1 instalado con --target")
    args = ap.parse_args()
    estado, vocab = leer_pesos(ubicar_modelo_spacy())
    hf_estado = convertir(estado)
    guardar(hf_estado, vocab, Path(args.salida))
    print(f"Modelo guardado en {args.salida} ({len(vocab)} piezas de vocabulario).")
    if args.verificar:
        dif = verificar(estado, Path(args.salida), args.verificar)
        print(f"Diferencia absoluta máxima HF vs curated-transformers: {dif:.2e}")
        if dif > 1e-3:
            raise SystemExit("La conversión no reproduce la salida original.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile experimentos/entrenar_nlp.py
"""Experimento NLP de KuriBot: validación cruzada, entrenamiento final, evaluación y exportación.

Uso:
    python experimentos/entrenar_nlp.py --version v1            # CV de 4 modelos + modelo final
    python experimentos/entrenar_nlp.py --version v2 --aumento  # iteración 2 (sin CV), solo M0 y M3

Salidas en resultados/ (JSON y CSV) y modelos/ (artefactos de inferencia).
"""
from __future__ import annotations

import argparse
import json
import random
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import torch
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "datos"))
from kuribot import nlu_modelos as NM  # noqa: E402

torch.set_num_threads(2)
RES = RAIZ / "resultados"
MOD = RAIZ / "modelos"
RES.mkdir(exist_ok=True)
SEMILLA = 42


def cargar(iteracion: int = 1):
    d = json.loads((RAIZ / "datos" / "corpus.json").read_text(encoding="utf-8"))["datos"]
    d = [f for f in d if f.get("iteracion", 1) <= iteracion]
    etiquetas = sorted({f["intencion"] for f in d})
    idx = {e: i for i, e in enumerate(etiquetas)}
    parts = {p: [f for f in d if f["particion"] == p] for p in ("pool", "prueba_limpia", "prueba_ruido")}
    return etiquetas, idx, parts


def vectores_fasttext():
    import spacy
    nlp = spacy.load("es_core_news_lg", exclude=["tok2vec", "morphologizer", "parser", "senter", "attribute_ruler",
                                                  "lemmatizer", "ner"])
    vocab = nlp.vocab

    def f(t):
        return vocab[t].vector.copy() if vocab.has_vector(t) else None
    return f


def aumentar(textos, etiquetas, grupos, semilla=7):
    """Iteración 2: añade una copia perturbada (sin tildes, abreviaturas, errata) de cada frase de entrenamiento."""
    from construir_corpus import ruido
    rng = random.Random(semilla)
    xt, yt, gt = list(textos), list(etiquetas), list(grupos)
    for t, y, g in zip(textos, etiquetas, grupos):
        xt.append(ruido(t, rng))
        yt.append(y)
        gt.append(g)
    return xt, np.array(yt), gt


EPOCAS_M3 = 6
PACIENCIA_M3 = 2


def fabricar(nombre, n, semilla, vec):
    if nombre == "M0":
        return NM.ModeloM0(n, semilla)
    if nombre == "M1":
        return NM.ModeloM1(n, semilla, vectores=vec)
    if nombre == "M2":
        return NM.ModeloM2(n, semilla, ruta_beto=str(MOD / "beto_base"))
    return NM.ModeloM3(n, semilla, ruta_beto=str(MOD / "beto_base"), epocas=EPOCAS_M3, paciencia=PACIENCIA_M3)


def separar_dev(y, grupos, semilla):
    sgk = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=semilla)
    tr, dev = next(sgk.split(np.zeros(len(y)), y, grupos))
    return tr, dev


def validacion_cruzada(modelos, pool, idx, n, vec):
    x = [f["texto"] for f in pool]
    y = np.array([idx[f["intencion"]] for f in pool])
    g = [f["grupo"] for f in pool]
    sgk = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
    out = {m: [] for m in modelos}
    for k, (tr, te) in enumerate(sgk.split(np.zeros(len(y)), y, g)):
        tr_in, dev_in = separar_dev(y[tr], [g[i] for i in tr], SEMILLA + k)
        tr_idx, dev_idx = tr[tr_in], tr[dev_in]
        for m in modelos:
            t0 = time.time()
            mod = fabricar(m, n, SEMILLA + k, vec)
            mod.entrenar([x[i] for i in tr_idx], y[tr_idx], [x[i] for i in dev_idx], y[dev_idx])
            pred = mod.logits([x[i] for i in te]).argmax(1)
            r = {"pliegue": k + 1, "exactitud": accuracy_score(y[te], pred), "f1_macro": f1_score(y[te], pred, average="macro"),
                 "n_prueba": int(len(te)), "segundos": round(time.time() - t0, 1)}
            out[m].append(r)
            print(f"[CV] pliegue {k+1} {m}: F1={r['f1_macro']:.3f} acc={r['exactitud']:.3f} ({r['segundos']} s)", flush=True)
            del mod
    resumen = {m: {"f1_macro_media": float(np.mean([r["f1_macro"] for r in v])),
                   "f1_macro_de": float(np.std([r["f1_macro"] for r in v], ddof=1)),
                   "exactitud_media": float(np.mean([r["exactitud"] for r in v])),
                   "exactitud_de": float(np.std([r["exactitud"] for r in v], ddof=1)), "pliegues": v}
               for m, v in out.items()}
    return resumen


def bootstrap_f1(y, pred, B=2000, semilla=0):
    rng = np.random.default_rng(semilla)
    n = len(y)
    vals = [f1_score(y[i], pred[i], average="macro") for i in (rng.integers(0, n, n) for _ in range(B))]
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


def mcnemar(y, a, b):
    """Prueba exacta de McNemar (binomial sobre pares discordantes)."""
    a_ok, b_ok = a == y, b == y
    n01 = int((a_ok & ~b_ok).sum())
    n10 = int((~a_ok & b_ok).sum())
    p = binomtest(n10, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return {"solo_A_acierta": n01, "solo_B_acierta": n10, "p_valor": float(p)}


def evaluar(nombre, mod, T, sets, etiquetas):
    res = {}
    for clave, (xs, ys) in sets.items():
        z = mod.logits(xs)
        p1, pT = NM.softmax(z), NM.softmax(z, T)
        pred = z.argmax(1)
        k = int((pred == ys).sum())
        res[clave] = {
            "exactitud": accuracy_score(ys, pred), "exactitud_ic95_wilson": NM.wilson(k, len(ys)),
            "f1_macro": f1_score(ys, pred, average="macro"), "f1_macro_ic95_bootstrap": bootstrap_f1(ys, pred),
            "f1_ponderado": f1_score(ys, pred, average="weighted"),
            "ece_sin_calibrar": NM.ece(p1, ys), "ece_calibrado": NM.ece(pT, ys),
            "reporte": classification_report(ys, pred, labels=list(range(len(etiquetas))), target_names=etiquetas,
                                             output_dict=True, zero_division=0),
            "matriz_confusion": confusion_matrix(ys, pred, labels=list(range(len(etiquetas)))).tolist(),
            "pred": pred.tolist(), "conf_cal": pT.max(1).tolist(), "probs_cal": pT.round(5).tolist(),
        }
        print(f"[TEST] {nombre} {clave}: acc={res[clave]['exactitud']:.3f} F1={res[clave]['f1_macro']:.3f} "
              f"ECE {res[clave]['ece_sin_calibrar']:.3f}->{res[clave]['ece_calibrado']:.3f}", flush=True)
    return res


def analisis_umbral(z_dev, y_dev, z_test, y_test, T, i_oos):
    """Selección del umbral de confianza en validación y su efecto en prueba (abstención -> fuera_alcance)."""
    def aplicar(z, tau):
        p = NM.softmax(z, T)
        pred = p.argmax(1).copy()
        pred[p.max(1) < tau] = i_oos
        return pred
    grid = np.round(np.arange(0.20, 0.91, 0.05), 2)
    f_dev = [f1_score(y_dev, aplicar(z_dev, t), average="macro") for t in grid]
    tau = float(grid[int(np.argmax(f_dev))])
    curva = []
    for t in grid:
        p = NM.softmax(z_test, T)
        acept = p.max(1) >= t
        pred = aplicar(z_test, t)
        en_alcance = y_test != i_oos
        curva.append({"tau": float(t), "cobertura": float(acept.mean()),
                      "exactitud_aceptadas": float((p.argmax(1)[acept] == y_test[acept]).mean()) if acept.any() else float("nan"),
                      "f1_macro_con_abstencion": float(f1_score(y_test, pred, average="macro")),
                      "recall_fuera_alcance": float((pred[~en_alcance] == i_oos).mean()),
                      "exactitud_en_alcance": float((pred[en_alcance] == y_test[en_alcance]).mean())})
    return {"tau_elegido_en_validacion": tau, "f1_dev_por_tau": dict(zip(map(float, grid), map(float, f_dev))), "curva_prueba": curva}


def exportar_m3(mod, etiquetas, T, tau, destino: Path, extra: dict):
    from safetensors.torch import save_file
    if destino.exists():
        shutil.rmtree(destino)
    destino.mkdir(parents=True)
    estado = {k: v.detach().to(torch.float16).contiguous() for k, v in mod.red.state_dict().items()}
    save_file(estado, str(destino / "clasificador.safetensors"))
    for f in ("config.json", "vocab.txt", "tokenizer.json", "tokenizer_config.json"):
        shutil.copy(MOD / "beto_base" / f, destino / ("bert_config.json" if f == "config.json" else f))
    (destino / "kuribot_nlu.json").write_text(json.dumps({
        "etiquetas": etiquetas, "temperatura": T, "umbral_confianza": tau, "max_len": 48,
        "preprocesamiento": "kuribot.entidades.enmascarar_entidades", "pesos": "float16", **extra}, ensure_ascii=False, indent=2),
        encoding="utf-8")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--version", default="v1")
    ap.add_argument("--aumento", action="store_true")
    ap.add_argument("--sin-cv", action="store_true")
    ap.add_argument("--modelos", default="M0,M1,M2,M3")
    ap.add_argument("--iteracion", type=int, default=1, help="1: corpus original; 2: con plantillas agregadas tras el análisis de fallos")
    ap.add_argument("--epocas-m3", type=int, default=6)
    ap.add_argument("--paciencia-m3", type=int, default=2)
    args = ap.parse_args()
    global EPOCAS_M3, PACIENCIA_M3
    EPOCAS_M3, PACIENCIA_M3 = args.epocas_m3, args.paciencia_m3
    modelos = args.modelos.split(",")
    etiquetas, idx, parts = cargar(args.iteracion)
    n = len(etiquetas)
    vec = vectores_fasttext() if "M1" in modelos else None
    salida = {"version": args.version, "aumento": args.aumento, "iteracion_corpus": args.iteracion,
              "epocas_max_m3": args.epocas_m3, "paciencia_m3": args.paciencia_m3, "etiquetas": etiquetas, "semilla": SEMILLA,
              "entorno": {"torch": torch.__version__, "hilos_cpu": torch.get_num_threads()}}
    if not args.sin_cv:
        salida["validacion_cruzada"] = validacion_cruzada(modelos, parts["pool"], idx, n, vec)
        (RES / f"nlp_cv_{args.version}.json").write_text(json.dumps(salida["validacion_cruzada"], indent=1), encoding="utf-8")

    pool = parts["pool"]
    x = [f["texto"] for f in pool]
    y = np.array([idx[f["intencion"]] for f in pool])
    g = [f["grupo"] for f in pool]
    tr, dev = separar_dev(y, g, SEMILLA)
    xtr, ytr, gtr = [x[i] for i in tr], y[tr], [g[i] for i in tr]
    if args.aumento:
        xtr, ytr, gtr = aumentar(xtr, ytr, gtr)
    xdev, ydev = [x[i] for i in dev], y[dev]
    sets = {k: ([f["texto"] for f in parts[k]], np.array([idx[f["intencion"]] for f in parts[k]]))
            for k in ("prueba_limpia", "prueba_ruido")}
    salida["particion_final"] = {"entrenamiento": len(xtr), "validacion": len(xdev), "prueba_limpia": len(sets["prueba_limpia"][1]),
                                 "prueba_ruido": len(sets["prueba_ruido"][1])}
    finales, preds = {}, {}
    for m in modelos:
        t0 = time.time()
        mod = fabricar(m, n, SEMILLA, vec)
        info = mod.entrenar(xtr, ytr, xdev, ydev)
        z_dev = mod.logits(xdev)
        T = NM.ajustar_temperatura(z_dev, ydev)
        r = evaluar(m, mod, T, sets, etiquetas)
        if m == "M2":
            mod.memo.clear()  # sin caché: la latencia debe incluir el paso por BETO
        lat = NM.latencia_ms(mod, sets["prueba_limpia"][0][:60])
        finales[m] = {"nombre": mod.nombre, "temperatura": T, "parametros_entrenables": NM.n_parametros(mod),
                      "segundos_entrenamiento": round(time.time() - t0, 1), "latencia": lat,
                      "f1_macro_validacion": float(f1_score(ydev, z_dev.argmax(1), average="macro")),
                      "info": {k: v for k, v in info.items()}, "prueba": r}
        preds[m] = {k: np.array(v["pred"]) for k, v in r.items()}
        if m in ("M3", "M0"):
            finales[m]["umbral"] = analisis_umbral(z_dev, ydev, mod.logits(sets["prueba_limpia"][0]), sets["prueba_limpia"][1], T,
                                                   etiquetas.index("fuera_alcance"))
        if m == "M3":
            tau = finales[m]["umbral"]["tau_elegido_en_validacion"]
            exportar_m3(mod, etiquetas, T, tau, MOD / f"intenciones_beto_{args.version}",
                        {"version": args.version, "aumento": args.aumento,
                         "f1_macro_prueba_limpia": r["prueba_limpia"]["f1_macro"],
                         "f1_macro_prueba_ruido": r["prueba_ruido"]["f1_macro"]})
        if m == "M0":
            import joblib
            joblib.dump({"pipe": mod.pipe, "etiquetas": etiquetas, "temperatura": T}, MOD / f"intenciones_tfidf_{args.version}.joblib")
        del mod
    comparaciones = {}
    if "M3" in preds:
        for otro in [m for m in modelos if m != "M3"]:
            for k, (xs, ys) in sets.items():
                comparaciones[f"M3_vs_{otro}_{k}"] = mcnemar(ys, preds["M3"][k], preds[otro][k])
    salida["modelos"] = finales
    salida["mcnemar"] = comparaciones
    (RES / f"nlp_final_{args.version}.json").write_text(json.dumps(salida, ensure_ascii=False, indent=1, default=float), encoding="utf-8")
    # Predicciones de prueba en CSV (para el informe y la auditoría de errores)
    import csv
    with open(RES / f"predicciones_prueba_{args.version}.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["conjunto", "texto", "real"] + [f"pred_{m}" for m in modelos] + [f"conf_{m}" for m in modelos])
        for k, (xs, ys) in sets.items():
            for i, (t, yv) in enumerate(zip(xs, ys)):
                w.writerow([k, t, etiquetas[yv]] + [etiquetas[preds[m][k][i]] for m in modelos]
                           + [round(finales[m]["prueba"][k]["conf_cal"][i], 4) for m in modelos])
    print("Listo:", RES / f"nlp_final_{args.version}.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile experimentos/figuras.py
"""Genera las figuras del informe a partir de los resultados reproducibles (resultados/*.json).

Uso: python experimentos/figuras.py --version v1 [--v2 v2]
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ))
RES, FIG = RAIZ / "resultados", RAIZ / "docs" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)

# Paleta de referencia validada (dataviz): categórica en orden fijo y rampa secuencial azul
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
TINTA, TINTA2, REJILLA = "#0b0b0b", "#52514e", "#e6e5e1"
SEQ = ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
from matplotlib.ticker import FuncFormatter  # noqa: E402

COMA = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))
plt.rcParams.update({"axes.formatter.use_locale": False, "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#b9b8b3", "axes.labelcolor": TINTA2,
                     "xtick.color": TINTA2, "ytick.color": TINTA2, "axes.titlesize": 10, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 220, "savefig.bbox": "tight"})
NOMBRE = {"M0": "M0 TF-IDF + RL", "M1": "M1 fastText + BiLSTM", "M2": "M2 BETO congelado + MLP", "M3": "M3 BETO ajustado"}
ETQ = {"ayuda_capacidades": "ayuda", "consulta_geodesia": "consulta geodesia", "consultar_concesion": "consultar concesión",
       "convertir_coordenadas": "convertir coord.", "despedida": "despedida", "emergencia_seguridad": "emergencia",
       "estado_solicitud": "estado solicitud", "fuera_alcance": "fuera de alcance", "hablar_humano": "hablar con humano",
       "reportar_alerta": "reportar alerta", "saludo": "saludo", "solicitar_cartografia": "solicitar cartografía",
       "solicitar_vuelo": "solicitar vuelo", "verificar_punto": "verificar punto"}


def _comas(fig):
    for ax in fig.axes:
        if ax.get_xaxis().get_major_formatter().__class__.__name__ == "ScalarFormatter":
            ax.xaxis.set_major_formatter(COMA)
        if ax.get_yaxis().get_major_formatter().__class__.__name__ == "ScalarFormatter":
            ax.yaxis.set_major_formatter(COMA)


_savefig = matplotlib.figure.Figure.savefig


def _savefig_comas(self, *a, **k):
    _comas(self)
    return _savefig(self, *a, **k)


matplotlib.figure.Figure.savefig = _savefig_comas


def barras_modelos(r, cv):
    mods = list(r["modelos"])
    grupos = [("Validación cruzada\n(5 pliegues, media)", lambda m: cv[m]["f1_macro_media"] if cv else np.nan),
              ("Prueba limpia\n(n = 252)", lambda m: r["modelos"][m]["prueba"]["prueba_limpia"]["f1_macro"]),
              ("Prueba con ruido\n(n = 252)", lambda m: r["modelos"][m]["prueba"]["prueba_ruido"]["f1_macro"])]
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    w = 0.8 / len(mods)
    for j, m in enumerate(mods):
        vals = [g[1](m) for g in grupos]
        x = np.arange(len(grupos)) + (j - (len(mods) - 1) / 2) * w
        ax.bar(x, vals, w * 0.92, color=C[j], label=NOMBRE[m], zorder=3)
        if cv and j < 4:
            de = [cv[m]["f1_macro_de"], 0, 0]
            ax.errorbar(x[0], vals[0], yerr=de[0], color=TINTA2, capsize=2, lw=0.8, zorder=4)
        extra = [cv[m]["f1_macro_de"] if cv else 0, 0, 0]
        for xi, v, e in zip(x, vals, extra):
            ax.text(xi, v + e + 0.012, f"{v:.2f}".replace(".", ","), ha="center", va="bottom", fontsize=6.5, color=TINTA)
    ax.set_xticks(range(len(grupos)), [g[0] for g in grupos])
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("F1 macro")
    ax.yaxis.grid(True, color=REJILLA, zorder=0)
    ax.legend(ncol=4, fontsize=7, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.13))
    fig.savefig(FIG / "comparacion_modelos.png")
    plt.close(fig)


def matriz(r, modelo, clave, nombre):
    et = r["etiquetas"]
    M = np.array(r["modelos"][modelo]["prueba"][clave]["matriz_confusion"], dtype=float)
    Mn = M / M.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("seq", SEQ)
    ax.imshow(Mn, cmap=cmap, vmin=0, vmax=1)
    for i in range(len(et)):
        for j in range(len(et)):
            if M[i, j]:
                ax.text(j, i, int(M[i, j]), ha="center", va="center", fontsize=7,
                        color="white" if Mn[i, j] > 0.55 else TINTA)
    ax.set_xticks(range(len(et)), [ETQ[e] for e in et], rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(et)), [ETQ[e] for e in et], fontsize=7)
    ax.set_xlabel("Intención predicha")
    ax.set_ylabel("Intención real")
    for s in ax.spines.values():
        s.set_visible(False)
    fig.savefig(FIG / nombre)
    plt.close(fig)


def f1_por_clase(r):
    et = r["etiquetas"]
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 4.0), sharey=True)
    for ax, clave, tit in zip(axes, ["prueba_limpia", "prueba_ruido"], ["Prueba limpia", "Prueba con ruido"]):
        y = np.arange(len(et))
        for j, (m, col) in enumerate([("M0", C[0]), ("M3", C[1])]):
            rep = r["modelos"][m]["prueba"][clave]["reporte"]
            vals = [rep[e]["f1-score"] for e in et]
            ax.barh(y + (j - 0.5) * 0.38, vals, 0.36, color=col, label=NOMBRE[m], zorder=3)
        ax.set_yticks(y, [ETQ[e] for e in et], fontsize=7)
        ax.set_xlim(0, 1.05)
        ax.set_title(tit)
        ax.xaxis.grid(True, color=REJILLA, zorder=0)
        ax.set_xlabel("F1")
        ax.invert_yaxis()
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, fontsize=7.5, frameon=False, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(FIG / "f1_por_intencion.png")
    plt.close(fig)


def curva_entrenamiento(r):
    h = r["modelos"]["M3"]["info"]["historia"]
    ep = [x["epoca"] for x in h]
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6))
    axes[0].plot(ep, [x["perdida_entrenamiento"] for x in h], color=C[0], lw=2, marker="o", ms=4)
    axes[0].set_title("Pérdida de entrenamiento (M3)")
    axes[0].set_xlabel("Época")
    axes[1].plot(ep, [x["f1_macro_validacion"] for x in h], color=C[1], lw=2, marker="o", ms=4)
    axes[1].set_title("F1 macro en validación (M3)")
    axes[1].set_xlabel("Época")
    for ax in axes:
        ax.yaxis.grid(True, color=REJILLA)
        ax.set_xticks(ep)
    fig.tight_layout()
    fig.savefig(FIG / "curva_entrenamiento.png")
    plt.close(fig)


def fiabilidad(r):
    p = r["modelos"]["M3"]["prueba"]["prueba_limpia"]
    et = r["etiquetas"]
    import csv
    filas = list(csv.DictReader(open(RES / f"predicciones_prueba_{r['version']}.csv", encoding="utf-8")))
    filas = [f for f in filas if f["conjunto"] == "prueba_limpia"]
    conf = np.array(p["conf_cal"])
    ok = np.array([f["pred_M3"] == f["real"] for f in filas])
    bordes = np.linspace(0, 1, 11)
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.9))
    xs, ys, ns = [], [], []
    for a, b in zip(bordes[:-1], bordes[1:]):
        m = (conf > a) & (conf <= b)
        if m.any():
            xs.append(conf[m].mean())
            ys.append(ok[m].mean())
            ns.append(m.sum())
    axes[0].plot([0, 1], [0, 1], ls="--", color=TINTA2, lw=1)
    axes[0].plot(xs, ys, color=C[0], lw=1.5, zorder=2)
    axes[0].scatter(xs, ys, s=[18 + 2.2 * n for n in ns], color=C[0], edgecolor="white", linewidth=1.5, zorder=3)
    for xx, yy, n in zip(xs, ys, ns):
        axes[0].annotate(f"n={n}", (xx, yy), xytext=(4, -11), textcoords="offset points", fontsize=6, color=TINTA2)
    axes[0].set_xlabel("Confianza calibrada")
    axes[0].set_ylabel("Exactitud observada")
    axes[0].set_title("Diagrama de fiabilidad (M3)")
    curva = r["modelos"]["M3"]["umbral"]["curva_prueba"]
    axes[1].plot([c["cobertura"] for c in curva], [c["exactitud_aceptadas"] for c in curva], color=C[1], lw=2, marker="o", ms=4)
    tau = r["modelos"]["M3"]["umbral"]["tau_elegido_en_validacion"]
    sel = min(curva, key=lambda c: abs(c["tau"] - tau))
    axes[1].plot(sel["cobertura"], sel["exactitud_aceptadas"], marker="o", ms=9, color=C[1], mec="white", mew=2)
    axes[1].annotate(f"τ = {tau:.2f}".replace(".", ","), (sel["cobertura"], sel["exactitud_aceptadas"]), xytext=(-18, 10),
                     textcoords="offset points", fontsize=7, color=TINTA)
    axes[1].set_xlabel("Cobertura (fracción aceptada)")
    axes[1].set_ylabel("Exactitud en aceptadas")
    axes[1].set_title("Riesgo–cobertura (prueba limpia)")
    for ax in axes:
        ax.grid(True, color=REJILLA)
    fig.tight_layout()
    fig.savefig(FIG / "calibracion_umbral.png")
    plt.close(fig)


def tornado():
    from kuribot.razonamiento import sensibilidad
    ev = {"H": 0, "A": 1, "M": 1, "S": 1}
    filas = [f for f in sensibilidad(ev) if f["rango"] > 1e-3][:8][::-1]
    base = filas[0]["base"]
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    for i, f in enumerate(filas):
        lo, hi = min(f["bajo"], f["alto"]), max(f["bajo"], f["alto"])
        ax.barh(i, hi - lo, left=lo, color=C[0], height=0.55, zorder=3)
        ax.text(hi + 0.005, i, f"{lo:.2f}–{hi:.2f}".replace(".", ","), va="center", fontsize=6.5, color=TINTA)
    ax.axvline(base, color=C[1], lw=1.5, zorder=4)
    ax.text(base, len(filas) - 0.3, f" posterior base {base:.3f}".replace(".", ","), color=TINTA, fontsize=7)
    ax.set_yticks(range(len(filas)), [f["parametro"] for f in filas], fontsize=7)
    ax.set_xlabel("P(intrusión | evidencias) al variar cada parámetro ±0,10")
    ax.xaxis.grid(True, color=REJILLA, zorder=0)
    fig.savefig(FIG / "sensibilidad_rb.png")
    plt.close(fig)


def grafos():
    from kuribot.razonamiento import ARISTAS, DESCRIPCION_NODOS
    etiqueta = {k: f"{k}\\n{v}" for k, v in DESCRIPCION_NODOS.items()}
    import textwrap
    lineas = ["digraph RB { rankdir=TB; nodesep=0.25; ranksep=0.5; bgcolor=white;",
              'node [shape=box, style="rounded,filled", fontname="DejaVu Sans", fontsize=10, color="#9ec5f4", fillcolor="#eef5fd"];']
    for k, v in DESCRIPCION_NODOS.items():
        txt = "\\n".join(textwrap.wrap(v, 24))
        extra = ', fillcolor="#fde6dc", color="#eb6834", penwidth=2' if k == "I" else (', fillcolor="#f3f2ee", color="#b9b8b3"' if k in "HA" else "")
        lineas.append(f'{k} [label="{k}: {txt}"{extra}];')
    lineas.append("{rank=same; H; A;}")
    for a, b in ARISTAS:
        lineas.append(f'{a} -> {b} [color="#52514e"];')
    lineas.append("}")
    (FIG / "red_bayesiana.dot").write_text("\n".join(lineas), encoding="utf-8")
    subprocess.run(["dot", "-Tpng", "-Gdpi=200", str(FIG / "red_bayesiana.dot"), "-o", str(FIG / "red_bayesiana.png")], check=True)

    arq = r'''digraph ARQ { rankdir=LR; compound=true; nodesep=0.3; ranksep=0.45; fontname="DejaVu Sans"; bgcolor=white;
node [shape=box, style="rounded,filled", fontname="DejaVu Sans", fontsize=9, fillcolor="#eef5fd", color="#6da7ec"];
edge [color="#52514e", fontname="DejaVu Sans", fontsize=7];
U [label="Usuario interno\n(Geomática, Seguridad,\nOperaciones)", shape=ellipse, fillcolor="#fde6dc", color="#eb6834"];
UI [label="Interfaz\nStreamlit (web) · consola", fillcolor="#f3f2ee", color="#b9b8b3"];
subgraph cluster_nlu { label="1. Comprensión (NLU)"; style="rounded"; color="#9ec5f4"; fontsize=9;
  PII [label="Enmascarado de datos\npersonales (R02)"]; NER [label="Extractor de entidades\n(regex + diccionario:\ncoordenadas, datum,\nconcesión, ticket, fecha,\nevidencias con negación)"];
  NLU [label="Clasificador de intenciones\nBETO ajustado + softmax\ncalibrado (T) · 14 clases", fillcolor="#fde6dc", color="#eb6834", penwidth=2]; }
subgraph cluster_dm { label="2. Gestión del diálogo y razonamiento"; style="rounded"; color="#9ec5f4"; fontsize=9;
  DM [label="Gestor de diálogo\nestado · ranuras · aclaración\n· cambio de tema · abstención"];
  RB [label="Red bayesiana (9 nodos)\nP(intrusión | evidencias)\npgmpy · eliminación de variables", fillcolor="#e3f6ee", color="#1baf7a", penwidth=2];
  MR [label="Motor de reglas\nencadenamiento hacia adelante\nR01–R42 (5 niveles)\n+ reglas de vuelo V01–V06", fillcolor="#e3f6ee", color="#1baf7a", penwidth=2];
  GEO [label="Geoprocesos\npyproj (EPSG:3990) · Shapely\npunto en polígono · distancias"]; }
subgraph cluster_kb { label="3. Base de conocimiento (sintética)"; style="rounded"; color="#9ec5f4"; fontsize=9;
  KB1 [label="Catastro GeoJSON\n(concesiones, área\nprotegida, comunidad)", shape=cylinder, fillcolor="#f3f2ee", color="#b9b8b3"];
  KB2 [label="Operación JSON\n(pronóstico, drones,\npilotos, historial)", shape=cylinder, fillcolor="#f3f2ee", color="#b9b8b3"];
  KB3 [label="FAQ geodésica\n(recuperación TF-IDF)", shape=cylinder, fillcolor="#f3f2ee", color="#b9b8b3"];
  KB4 [label="Tickets SQLite", shape=cylinder, fillcolor="#f3f2ee", color="#b9b8b3"]; }
NLG [label="4. Generación (NLG)\nplantillas con datos\nverificados (sin LLM)"];
LOG [label="Registro auditable\n(JSONL, texto enmascarado)", fillcolor="#f3f2ee", color="#b9b8b3"];
HUM [label="Derivación a persona\n(analista, Seguridad, Legal)", fillcolor="#fde6dc", color="#eb6834"];
U -> UI [label="mensaje"]; UI -> PII; PII -> NER; PII -> NLU; NER -> DM; NLU -> DM [label="intención,\nconfianza"];
DM -> GEO; DM -> RB [label="evidencias"]; RB -> MR [label="posterior"]; GEO -> MR [label="contexto"]; DM -> MR;
GEO -> KB1 [dir=both]; RB -> KB2 [dir=both, label="H, A"]; MR -> KB2 [dir=both]; DM -> KB3 [dir=both]; DM -> KB4 [dir=both];
MR -> NLG [label="prioridad,\nacción"]; DM -> NLG; NLG -> UI [label="respuesta +\ntrazabilidad"]; DM -> LOG [style=dashed]; MR -> HUM [style=dashed, label="R01, R03,\nR40, R41"];
}'''
    (FIG / "arquitectura.dot").write_text(arq, encoding="utf-8")
    subprocess.run(["dot", "-Tpng", "-Gdpi=200", str(FIG / "arquitectura.dot"), "-o", str(FIG / "arquitectura.png")], check=True)


def iteraciones(v1, v2):
    """Comparación v1 -> v2 del modelo de producción."""
    if not v2:
        return
    fig, ax = plt.subplots(figsize=(5.6, 2.8))
    etiquetas = ["Prueba limpia", "Prueba con ruido"]
    for j, (r, nombre) in enumerate([(v1, "v1 (sin aumento)"), (v2, "v2 (con aumento de ruido)")]):
        vals = [r["modelos"]["M3"]["prueba"][k]["f1_macro"] for k in ("prueba_limpia", "prueba_ruido")]
        x = np.arange(2) + (j - 0.5) * 0.36
        ax.bar(x, vals, 0.34, color=C[j], label=nombre, zorder=3)
        for xi, v in zip(x, vals):
            ax.text(xi, v + 0.01, f"{v:.3f}".replace(".", ","), ha="center", fontsize=7, color=TINTA)
    ax.set_xticks(range(2), etiquetas)
    ax.set_ylim(0.6, 1.02)
    ax.set_ylabel("F1 macro (M3)")
    ax.yaxis.grid(True, color=REJILLA, zorder=0)
    ax.legend(fontsize=7, frameon=False, loc="lower left")
    fig.savefig(FIG / "iteraciones_m3.png")
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--version", default="v1")
    ap.add_argument("--v2", default=None)
    args = ap.parse_args()
    r = json.loads((RES / f"nlp_final_{args.version}.json").read_text(encoding="utf-8"))
    barras_modelos(r, r.get("validacion_cruzada"))          # comparación de los cuatro modelos (iteración 1)
    prod = json.loads((RES / f"nlp_final_{args.v2}.json").read_text(encoding="utf-8")) if args.v2 else r
    matriz(prod, "M3", "prueba_limpia", "matriz_confusion_m3_limpia.png")   # modelo de producción
    matriz(prod, "M3", "prueba_ruido", "matriz_confusion_m3_ruido.png")
    f1_por_clase(prod)
    curva_entrenamiento(prod)
    fiabilidad(prod)
    tornado()
    grafos()
    if args.v2:
        iteraciones(r, prod)
    print("Figuras en", FIG)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile pruebas/test_unitarias.py
"""Pruebas unitarias de KuriBot (pytest). Ejecutar: python -m pytest pruebas/test_unitarias.py -v"""
from __future__ import annotations

import datetime as dt
import itertools
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from kuribot import entidades as E  # noqa: E402
from kuribot import geodesia as G  # noqa: E402
from kuribot.conocimiento import BaseConocimiento  # noqa: E402
from kuribot.razonamiento import NODOS, RedAlertas, motor_alertas, primera_ventana, sensibilidad  # noqa: E402


@pytest.fixture(scope="module")
def kb():
    return BaseConocimiento()


@pytest.fixture(scope="module")
def red():
    return RedAlertas()


# ---------------------------------------------------------------- geodesia
def test_ida_y_vuelta_psad56_wgs84():
    e, n = 642600.0, 9662480.0
    we, wn = G.psad56_utm_a_wgs84_utm(e, n)
    pe, pn = G.wgs84_utm_a_psad56_utm(we, wn)
    assert abs(pe - e) < 0.01 and abs(pn - n) < 0.01


def test_desplazamiento_psad56_en_zona_del_proyecto():
    we, wn = G.psad56_utm_a_wgs84_utm(642600.0, 9662480.0)
    d = ((we - 642600) ** 2 + (wn - 9662480) ** 2) ** 0.5
    assert 400 < d < 480          # orden de magnitud conocido para Ecuador continental


def test_geograficas_a_utm():
    e, n = G.a_wgs84_utm(G.Punto(-79.7301, -3.0512, "geo", "WGS84"))
    assert abs(e - 641120.7) < 1 and abs(n - 9662664.3) < 1


def test_operacion_epsg_3990_disponible():
    assert "3990" in G.descripcion_transformacion()


# ---------------------------------------------------------------- entidades
@pytest.mark.parametrize("texto,esperado", [
    ("E 642350 N 9662120", (642350, 9662120, "utm")), ("642350, 9662120", (642350, 9662120, "utm")),
    ("x=644800 y=9661500", (644800, 9661500, "utm")), ("-3.0512, -79.7301", (-79.7301, -3.0512, "geo")),
    ("3°03'04\"S 79°43'48\"W", (-79.73, -3.0511, "geo")),
])
def test_extraer_coordenadas(texto, esperado):
    c = E.extraer_coordenada(texto)
    assert c is not None and c.punto.tipo == esperado[2]
    assert abs(c.punto.x - esperado[0]) < 1e-3 and abs(c.punto.y - esperado[1]) < 1e-3


def test_datum_detectado():
    assert E.extraer_datum("643990 9661010 psad56") == "PSAD56"
    assert E.extraer_datum("en WGS 84") == "WGS84"
    assert E.extraer_datum("sin datum") is None


def test_concesion_por_codigo_nombre_y_errata(kb):
    assert E.extraer_concesion("revisa cm 103", kb.nombres) == "CM-103"
    assert E.extraer_concesion("estado de pena blanca", kb.nombres) == "CM-101"
    assert E.extraer_concesion("la consecion el robel", kb.nombres) == "CM-104"
    assert E.extraer_concesion("una concesión cualquiera", kb.nombres) is None


def test_evidencias_con_negacion():
    ev = E.extraer_evidencias("hay maquinaria, no hay personas y el agua está turbia")
    assert ev == {"M": True, "P": False, "T": True}


def test_fechas_relativas():
    hoy = dt.date(2026, 11, 9)  # lunes
    assert E.extraer_fecha("mañana", hoy) == 1
    assert E.extraer_fecha("pasado mañana", hoy) == 2
    assert E.extraer_fecha("el viernes", hoy) == 4
    assert E.extraer_fecha("el 12 de noviembre", hoy) == 3


def test_cedula_y_enmascarado():
    assert E.cedula_valida("3000000004") and not E.cedula_valida("3000000005")
    texto, tipos = E.enmascarar_pii("cédula 3000000004, celular 0991234567, correo a@b.ec")
    assert "3000000004" not in texto and "0991234567" not in texto and set(tipos) == {"cédula", "teléfono", "correo"}


def test_emergencia_palabras_clave():
    assert E.hay_emergencia("hay gente armada en la vía")
    assert not E.hay_emergencia("se escuchan voladuras de noche")


# ---------------------------------------------------------------- red bayesiana
def test_pgmpy_coincide_con_enumeracion(red):
    for valores in itertools.product([0, 1], repeat=4):
        ev = dict(zip(["H", "A", "M", "T"], valores))
        assert abs(red.posterior(ev) - red.posterior_enumeracion(ev)) < 1e-9


def test_prior_sin_evidencia(red):
    assert abs(red.posterior({"H": 0}) - 0.10) < 1e-9 and abs(red.posterior({"H": 1}) - 0.30) < 1e-9


def test_explicacion_alternativa_reduce_posterior(red):
    ev = {"H": 0, "M": 1, "S": 1}
    assert red.posterior({**ev, "A": 1}) < 0.3 < 0.9 < red.posterior({**ev, "A": 0})


def test_monotonia_evidencias(red):
    base = {"H": 0, "A": 0}
    for nodo in [n for n in NODOS if n not in ("H", "A", "I")]:
        assert red.posterior({**base, nodo: 1}) > red.posterior(base) > red.posterior({**base, nodo: 0})


def test_sensibilidad_ordenada():
    filas = sensibilidad({"H": 0, "A": 1, "M": 1, "S": 1})
    assert filas[0]["rango"] >= filas[-1]["rango"] and filas[0]["parametro"].startswith("P(I=1")


# ---------------------------------------------------------------- reglas
def test_emergencia_prevalece():
    f = motor_alertas().inferir({"emergencia": True, "p_intrusion": 0.05, "ubicacion": "propia"})
    assert f.hechos["prioridad"] == "CRÍTICA" and f.hechos["prohibir_despliegue"] and f.disparadas[0] == "R01"


def test_tercero_no_se_califica_como_intrusion():
    f = motor_alertas().inferir({"ubicacion": "tercero", "p_intrusion": 0.95})
    assert f.hechos["prioridad"] == "REVISIÓN LEGAL" and "R21" not in f.disparadas


@pytest.mark.parametrize("p,prioridad", [(0.9, "CRÍTICA"), (0.6, "ALTA"), (0.3, "MEDIA"), (0.05, "BAJA")])
def test_umbral_prioridad(p, prioridad):
    assert motor_alertas().inferir({"ubicacion": "propia", "p_intrusion": p}).hechos["prioridad"] == prioridad


def test_prioridad_unica_y_determinista():
    import random
    from kuribot.razonamiento import REGLAS_ALERTA, MotorReglas
    hechos = {"ubicacion": "propia", "p_intrusion": 0.85, "comunidad_cercana": True, "vuelo_dias": 0}
    ref = MotorReglas(REGLAS_ALERTA).inferir(hechos).hechos
    for s in range(20):
        reglas = REGLAS_ALERTA[:]
        random.Random(s).shuffle(reglas)
        assert MotorReglas(reglas).inferir(hechos).hechos == ref


def test_supervision_humana_en_zona_sensible():
    f = motor_alertas().inferir({"ubicacion": "propia", "p_intrusion": 0.6, "area_protegida": True})
    assert f.hechos["revision_humana"] and f.hechos["aprobacion_supervisor"]


# ---------------------------------------------------------------- vuelos y base de conocimiento
def test_ventana_vuelo_evita_viento(kb):
    v = primera_ventana(1, "Norte", kb.op)
    assert v["dia"] == 2 and any("viento" in m for m in v["evaluados"][0]["motivos"])


def test_anticipacion_minima(kb):
    assert primera_ventana(0, "Sur", kb.op, urgente=False)["dia"] == 1
    assert primera_ventana(0, "Sur", kb.op, urgente=True)["dia"] == 0


def test_analisis_espacial(kb):
    assert kb.analizar_punto(642350, 9662120).ubicacion == "propia"
    assert kb.analizar_punto(642000, 9663600).ubicacion == "tercero"
    a = kb.analizar_punto(641100, 9664950)
    assert a.ubicacion == "amortiguamiento" and a.cercana == "CM-104" and abs(a.distancia_lindero_m - 100) < 1


def test_ticket_inexistente_no_se_inventa(kb):
    assert kb.ticket("GEO-2026-0999") is None and kb.ticket("GEO-2026-0412")["estado"] == "En elaboración"


def test_areas_derivadas_de_geometria(kb):
    assert kb.ficha("CM-103")["area_ha"] == 480.0


def test_riesgo_de_datum_cambia_la_concesion(kb):
    """Las mismas cifras en PSAD56 y en WGS 84 caen en concesiones distintas: el datum importa."""
    we, wn = G.psad56_utm_a_wgs84_utm(643100, 9662100)
    assert kb.analizar_punto(643100, 9662100).concesion == "CM-102"
    assert kb.analizar_punto(we, wn).concesion == "CM-101"


In [ ]:
%%writefile pruebas/suite_aceptacion.py
"""Suite de aceptación funcional de KuriBot: 34 escenarios (43 turnos) con criterios explícitos.

Cada escenario empieza con un estado de diálogo limpio. Se evalúa el ÚLTIMO turno de cada
escenario (los turnos previos preparan el contexto) y se verifica, además, cada turno
intermedio que deba pedir un dato. Criterios de aprobación por turno evaluado:
  (a) intención esperada (o conjunto aceptable), (b) acción o regla esperada,
  (c) fragmentos de contenido obligatorios, (d) latencia < 1 500 ms.

Además, CONFIRMACION contiene 8 escenarios redactados DESPUÉS de cerrar la iteración 2 y ejecutados una sola
vez, sin ajustes posteriores, para estimar el desempeño en casos no usados durante el desarrollo.

Uso: python pruebas/suite_aceptacion.py --version v1 | --version v2 [--confirmacion]
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ))
from kuribot.conocimiento import BaseConocimiento  # noqa: E402
from kuribot.dialogo import KuriBot  # noqa: E402
from kuribot.nlu import ClasificadorIntenciones  # noqa: E402

# (id, descripción, [turnos], criterios del último turno, criterios de turnos intermedios {indice: dict})
ESCENARIOS = [
    ("A01", "Saludo y aviso de transparencia", ["buenos días, ¿con quién hablo?"],
     {"intencion": ["saludo"], "contiene": ["asistente automatizado"]}),
    ("A02", "Capacidades", ["¿qué cosas puedes resolver?"], {"intencion": ["ayuda_capacidades"], "contiene": ["consultar una concesión"]}),
    ("A03", "Ficha de concesión por nombre", ["dame la ficha de la concesión La Ventana"],
     {"intencion": ["consultar_concesion"], "contiene": ["CM-103", "480"]}),
    ("A04", "Estado de concesión suspendida", ["¿cerro azul está activa?"], {"intencion": ["consultar_concesion"], "contiene": ["suspendida"]}),
    ("A05", "Ranura faltante de concesión", ["quiero consultar una concesión", "CM-106"],
     {"intencion": ["consultar_concesion"], "contiene": ["Los Laureles"]}, {0: {"accion": "solicitar_dato:concesion"}}),
    ("A06", "Punto WGS84 dentro de concesión", ["¿el punto E 642350 N 9662120 WGS84 está dentro de la concesión?"],
     {"intencion": ["verificar_punto"], "contiene": ["DENTRO de CM-101"], "reglas": ["R10"]}),
    ("A07", "Punto sin datum: se pide y luego cerca de comunidad", ["verifica si 644800 9661500 cae en nuestra área", "wgs84"],
     {"intencion": ["verificar_punto"], "contiene": ["DENTRO de CM-103", "comunidad"], "reglas": ["R14"]},
     {0: {"accion": "solicitar_dato:datum"}}),
    ("A08", "Punto PSAD56 transformado antes del cruce", ["¿E 642600 N 9662480 PSAD56 en qué concesión queda?"],
     {"intencion": ["verificar_punto"], "contiene": ["Transformé", "DENTRO de CM-101"]}),
    ("A09", "Punto en franja de amortiguamiento", ["chequea 641100 9664950 wgs84 contra el catastro"],
     {"intencion": ["verificar_punto"], "contiene": ["franja de 500 m"], "reglas": ["R11"]}),
    ("A10", "Punto en concesión de tercero", ["¿la coordenada 642000 9663600 wgs84 es nuestra?"],
     {"intencion": ["verificar_punto"], "contiene": ["tercero"], "reglas": ["R12"]}),
    ("A11", "Conversión PSAD56 -> WGS84", ["convierte E 642600 N 9662480 de PSAD56 a WGS84"],
     {"intencion": ["convertir_coordenadas"], "contiene": ["EPSG:32717", "642 348"]}),
    ("A12", "Conversión geográficas -> UTM", ["pasa -3.0512, -79.7301 a utm, está en wgs84"],
     {"intencion": ["convertir_coordenadas"], "contiene": ["E 641 120"]}),
    ("A13", "Conversión con dos ranuras", ["necesito convertir unas coordenadas", "645210 9663790", "psad56"],
     {"intencion": ["convertir_coordenadas"], "contiene": ["Origen: PSAD56"]},
     {0: {"accion": "solicitar_dato:coord"}, 1: {"accion": "solicitar_dato:datum"}}),
    ("A14", "Alerta con maquinaria y personas en concesión suspendida", ["hay una retroexcavadora y gente extraña trabajando en Cerro Azul"],
     {"intencion": ["reportar_alerta"], "contiene": ["prioridad CRÍTICA"], "reglas": ["R21", "R41", "R42"]}),
    ("A15", "Alerta explicada por actividad autorizada (explaining away)", ["se ve desbroce nuevo y maquinaria en Peña Blanca"],
     {"intencion": ["reportar_alerta"], "contiene": ["prioridad MEDIA", "actividad autorizada programada explica"], "reglas": ["R23"]}),
    ("A16", "Alerta en tres turnos con ranuras", ["quiero reportar una invasión", "en Los Laureles", "el agua baja turbia"],
     {"intencion": ["reportar_alerta"], "contiene": ["prioridad ALTA", "supervisor"], "reglas": ["R22", "R41"]},
     {0: {"accion": "solicitar_dato:ubicacion"}, 1: {"accion": "solicitar_dato:evidencias"}}),
    ("A17", "Alerta en concesión de tercero", ["vimos maquinaria en E 642000 N 9663600 WGS84"],
     {"intencion": ["reportar_alerta"], "contiene": ["REVISIÓN LEGAL"], "reglas": ["R20", "R35"]}),
    ("A18", "Alerta crítica con ventana de vuelo al día siguiente", ["reporto voladuras de noche y una trocha nueva en El Roble"],
     {"intencion": ["reportar_alerta"], "contiene": ["prioridad CRÍTICA", "Vuelo de verificación prioritario"], "reglas": ["R30"]}),
    ("A19", "Alerta con negación y zona sensible", ["hay maquinaria en La Ventana, pero no hay personas"],
     {"intencion": ["reportar_alerta"], "contiene": ["prioridad BAJA", "revisión humana"], "reglas": ["R24", "R40"]}),
    ("A20", "Emergencia con armas", ["hay gente armada disparando cerca del campamento"],
     {"intencion": ["emergencia_seguridad"], "contiene": ["ECU 911", "No se enviará personal ni dron"], "reglas": ["R01"]}),
    ("A21", "Vuelo con fecha no apta por viento", ["necesito un vuelo de dron en San Gregorio", "mañana"],
     {"intencion": ["solicitar_vuelo"], "contiene": ["no es apta", "viento", "2026-11-11"]}, {0: {"accion": "solicitar_dato:fecha"}}),
    ("A22", "Vuelo el mismo día sin urgencia (anticipación)", ["programa un vuelo en Los Laureles para hoy"],
     {"intencion": ["solicitar_vuelo"], "contiene": ["anticipación", "2026-11-10"]}),
    ("A23", "Solicitud de KMZ", ["necesito el KMZ de los linderos de CM-102"],
     {"intencion": ["solicitar_cartografia"], "contiene": ["KMZ", "3 día"], "accion": "crear_ticket"}),
    ("A24", "Cartografía con ranuras", ["quiero pedir cartografía", "una ortofoto", "de El Roble"],
     {"intencion": ["solicitar_cartografia"], "contiene": ["ortofoto de CM-104"]},
     {0: {"accion": "solicitar_dato:producto"}, 1: {"accion": "solicitar_dato:concesion"}}),
    ("A25", "Estado de ticket existente", ["¿cómo va la solicitud GEO-2026-0412?"],
     {"intencion": ["estado_solicitud"], "contiene": ["En elaboración"]}),
    ("A26", "Ticket inexistente (no inventar)", ["estado del ticket GEO-2026-0999"],
     {"intencion": ["estado_solicitud"], "contiene": ["No encuentro"]}),
    ("A27", "FAQ de códigos EPSG", ["¿qué EPSG corresponde a PSAD56 en UTM 17 sur?"],
     {"intencion": ["consulta_geodesia"], "contiene": ["24877"]}),
    ("A28", "FAQ de desplazamiento entre datums", ["¿cuántos metros se mueve un punto entre PSAD56 y WGS84?"],
     {"intencion": ["consulta_geodesia"], "contiene": ["443 m"]}),
    ("A29", "Derivación a humano", ["quiero hablar con un analista de verdad"],
     {"intencion": ["hablar_humano"], "reglas": ["R03"], "accion": "derivar_humano"}),
    ("A30", "Fuera de alcance", ["¿quién ganó la Copa Libertadores?"],
     {"intencion": ["fuera_alcance"], "contiene_alguno": ["fuera de mi alcance", "No tengo suficiente seguridad"]}),
    ("A31", "Dato personal enmascarado", ["mi cédula es 3000000004, ¿cuál es el estado de CM-104?"],
     {"intencion": ["consultar_concesion"], "contiene": ["renovación"], "reglas": ["R02"]}),
    ("A32", "Mensaje con faltas ortográficas", ["ola, nesesito saver el estado de la consecion el robel"],
     {"intencion": ["consultar_concesion"], "contiene": ["El Roble"]}),
    ("A33", "Despedida", ["gracias, eso es todo"], {"intencion": ["despedida"]}),
    ("A34", "Coordenadas sin verbo (mensaje ambiguo)", ["642350 9662120"],
     {"intencion": ["verificar_punto", "convertir_coordenadas"], "accion_alguna": ["solicitar_dato:datum", None]}),
]


CONFIRMACION = [
    ("C01", "Alerta con coordenadas y dos evidencias", ["reporto maquinaria y una trocha nueva en E 646900 N 9660300 WGS84"],
     {"intencion": ["reportar_alerta"], "contiene": ["prioridad"]}),
    ("C02", "Teléfono enmascarado en una solicitud de cartografía",
     ["llámame al 0991234567 para coordinar, necesito un plano topográfico de CM-106"],
     {"intencion": ["solicitar_cartografia"], "reglas": ["R02"], "accion": "crear_ticket"}),
    ("C03", "Emergencia sin la palabra «emergencia»", ["auxilio, hay heridos en la vía al campamento sur"],
     {"intencion": ["emergencia_seguridad"], "reglas": ["R01"]}),
    ("C04", "Conversión desde grados, minutos y segundos", ["pasa 3°03'04\"S 79°43'48\"W a UTM, datum WGS84"],
     {"intencion": ["convertir_coordenadas"], "contiene": ["E 641"]}),
    ("C05", "Vuelo con día de la semana", ["agenda un vuelo sobre La Ventana el viernes"],
     {"intencion": ["solicitar_vuelo"], "contiene": ["2026-11-13"]}),
    ("C06", "FAQ formulada de otra manera", ["¿qué ventaja tiene el RTK frente al GPS navegador?"],
     {"intencion": ["consulta_geodesia"], "contiene": ["centímetros"]}),
    ("C07", "Clima fuera del contexto de vuelos", ["¿va a llover mañana en Machala?"],
     {"intencion": ["fuera_alcance"], "contiene_alguno": ["fuera de mi alcance", "No tengo suficiente seguridad"]}),
    ("C08", "Estado de ticket en registro informal", ["porfa el estatus del GEO-2026-0430"],
     {"intencion": ["estado_solicitud"], "contiene": ["En verificación de campo"]}),
]


def evaluar_turno(r, crit: dict) -> tuple[bool, list[str]]:
    fallos = []
    if "intencion" in crit and r.intencion not in crit["intencion"]:
        fallos.append(f"intención {r.intencion} ∉ {crit['intencion']}")
    if "accion" in crit and r.accion != crit["accion"]:
        fallos.append(f"acción {r.accion} ≠ {crit['accion']}")
    if "accion_alguna" in crit and r.accion not in crit["accion_alguna"] and r.decision != "aclarar":
        fallos.append(f"acción {r.accion} ∉ {crit['accion_alguna']}")
    for frag in crit.get("contiene", []):
        if frag.lower() not in r.texto.lower():
            fallos.append(f"falta «{frag}»")
    if "contiene_alguno" in crit and not any(f.lower() in r.texto.lower() for f in crit["contiene_alguno"]):
        fallos.append("no contiene ninguno de los fragmentos esperados")
    for regla in crit.get("reglas", []):
        if regla not in r.reglas:
            fallos.append(f"no disparó {regla}")
    if r.latencia_ms >= 1500:
        fallos.append(f"latencia {r.latencia_ms:.0f} ms")
    return not fallos, fallos


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--version", default="v1")
    ap.add_argument("--modelo", default=None, help="Carpeta del modelo NLU (por defecto, el más reciente)")
    ap.add_argument("--confirmacion", action="store_true", help="Ejecuta solo los 8 escenarios de confirmación")
    args = ap.parse_args()
    nlu = ClasificadorIntenciones(args.modelo)
    filas, aprobados, turnos_total = [], 0, 0
    for esc in (CONFIRMACION if args.confirmacion else ESCENARIOS):
        ident, desc, turnos, crit = esc[:4]
        inter = esc[4] if len(esc) > 4 else {}
        bot = KuriBot(nlu, BaseConocimiento())
        detalle, ok_esc = [], True
        for i, t in enumerate(turnos):
            r = bot.responder(t)
            turnos_total += 1
            c = crit if i == len(turnos) - 1 else inter.get(i)
            ok, fallos = evaluar_turno(r, c) if c else (True, [])
            ok_esc &= ok
            detalle.append({"turno": i + 1, "entrada": t, "respuesta": r.texto, "intencion": r.intencion,
                            "confianza": round(r.confianza or 0, 4), "decision": r.decision, "accion": r.accion,
                            "reglas": r.reglas, "p_intrusion": (r.bayes or {}).get("p_intrusion"), "ticket": r.ticket,
                            "latencia_ms": round(r.latencia_ms, 1), "evaluado": bool(c), "ok": ok, "fallos": fallos})
        aprobados += ok_esc
        filas.append({"id": ident, "descripcion": desc, "aprobado": ok_esc, "turnos": detalle})
        estado = "OK " if ok_esc else "FALLA"
        print(f"{estado} {ident} {desc}" + ("" if ok_esc else "  -> " + " | ".join(f for d in detalle for f in d["fallos"])))
    lat = sorted(d["latencia_ms"] for f in filas for d in f["turnos"])
    resumen = {"version": args.version, "modelo": nlu.nombre, "escenarios": len(filas), "aprobados": aprobados,
               "tasa": aprobados / len(filas), "turnos": turnos_total,
               "latencia_media_ms": sum(lat) / len(lat), "latencia_p95_ms": lat[int(0.95 * (len(lat) - 1))]}
    print(json.dumps(resumen, ensure_ascii=False, indent=1))
    salida = RAIZ / "resultados" / (f"confirmacion_{args.version}.json" if args.confirmacion else f"aceptacion_{args.version}.json")
    salida.write_text(json.dumps({"resumen": resumen, "escenarios": filas}, ensure_ascii=False, indent=1, default=float), encoding="utf-8")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cli.py
"""Interfaz de consola de KuriBot.  Ejecutar:  python cli.py   (escriba «salir» para terminar)"""
from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from kuribot.dialogo import AVISO, KuriBot  # noqa: E402


def main():
    bot = KuriBot(registro=Path("registros/consola.jsonl"))
    print(f"KuriBot · {bot.nlu.nombre}\n{AVISO}\nEscriba «salir» para terminar.\n")
    while True:
        try:
            texto = input("Usted › ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not sys.stdin.isatty():
            print(texto)  # eco de la entrada cuando la consola se alimenta desde un archivo
        if texto.lower() in {"salir", "exit", "quit"}:
            break
        if not texto:
            continue
        r = bot.responder(texto)
        print(f"KuriBot › {r.texto}")
        for a in r.avisos:
            print(f"   ⚠ {a}")
        extra = f" · P(intrusión)={r.bayes['p_intrusion']:.3f}" if r.bayes else ""
        print(f"   [intención={r.intencion} conf={r.confianza:.2f} decisión={r.decision} reglas={','.join(r.reglas) or '—'}"
              f"{extra} · {r.latencia_ms:.0f} ms]\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile app_streamlit.py
"""Interfaz web de KuriBot (Streamlit).  Ejecutar:  streamlit run app_streamlit.py"""
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import streamlit as st

sys.path.insert(0, str(Path(__file__).resolve().parent))
from kuribot.conocimiento import BaseConocimiento  # noqa: E402
from kuribot.dialogo import AVISO, KuriBot  # noqa: E402
from kuribot.entidades import EVIDENCIAS  # noqa: E402
from kuribot.mapa import dibujar  # noqa: E402
from kuribot.nlu import ClasificadorIntenciones  # noqa: E402
from kuribot.razonamiento import DESCRIPCION_NODOS  # noqa: E402

st.set_page_config(page_title="KuriBot · Geomática", page_icon="🛰️", layout="wide")


@st.cache_resource(show_spinner="Cargando el modelo de lenguaje…")
def cargar_nlu():
    return ClasificadorIntenciones()


def bot() -> KuriBot:
    if "bot" not in st.session_state:
        st.session_state.bot = KuriBot(cargar_nlu(), BaseConocimiento(), registro=Path("registros/conversaciones.jsonl"))
        st.session_state.historial = []
    return st.session_state.bot


EJEMPLOS = ["Hola", "¿Cuál es el estado de la concesión La Ventana?", "¿El punto E 642600 N 9662480 PSAD56 está dentro de alguna concesión?",
            "Hay una retroexcavadora y gente extraña trabajando en Cerro Azul", "Necesito un vuelo de dron en San Gregorio mañana",
            "¿Cómo va la solicitud GEO-2026-0412?", "¿Qué EPSG uso para PSAD56 UTM 17 sur?"]

with st.sidebar:
    st.markdown("### 🛰️ KuriBot")
    st.caption(AVISO)
    st.markdown(f"**Modelo NLU:** {cargar_nlu().nombre}")
    ver_traza = st.toggle("Mostrar trazabilidad", value=True)
    st.markdown("**Ejemplos**")
    for i, e in enumerate(EJEMPLOS):
        if st.button(e, key=f"ej{i}", use_container_width=True):
            st.session_state.pendiente_ejemplo = e
    if st.button("🔄 Nueva conversación", use_container_width=True):
        for k in ("bot", "historial"):
            st.session_state.pop(k, None)
        st.rerun()
    st.caption("No escriba cédulas, teléfonos ni datos personales. Emergencias reales: ECU 911.")

st.markdown("## KuriBot · Asistente de Geomática y Catastro minero")
st.caption("Prototipo académico · datos ficticios · NLU con BETO ajustado + reglas + red bayesiana")
asistente = bot()


def mostrar_traza(r):
    c1, c2 = st.columns([1.15, 1])
    with c1:
        st.markdown(f"**Intención:** `{r.intencion}` · **confianza** {r.confianza:.2f} · **decisión** {r.decision}")
        st.dataframe(pd.DataFrame(r.top, columns=["intención", "probabilidad"]).round(3), hide_index=True, use_container_width=True)
        if r.entidades:
            def fmt(k, v):
                if hasattr(v, "texto"):
                    return f"coordenada={v.texto.strip()}"
                if k == "evidencias":
                    return "evidencias=" + "; ".join(f"{EVIDENCIAS[e][0]} ({'sí' if b else 'no'})" for e, b in v.items())
                return f"{k}={v}"
            st.markdown("**Entidades:** " + ", ".join(fmt(k, v) for k, v in r.entidades.items() if v not in (False,)))
        if r.traza_reglas:
            st.markdown("**Reglas disparadas**")
            for t in r.traza_reglas:
                st.markdown(f"- {t}")
        if r.bayes:
            b = r.bayes
            st.markdown(f"**Red bayesiana:** P(intrusión | evidencias) = **{b['p_intrusion']:.3f}**")
            filas = [{"nodo": k, "descripción": DESCRIPCION_NODOS[k], "valor": "sí" if v else "no"} for k, v in b["evidencia"].items()]
            st.dataframe(pd.DataFrame(filas), hide_index=True, use_container_width=True)
            if b["contribuciones"]:
                st.caption("Cambio del posterior al retirar cada evidencia: " + "; ".join(
                    f"{EVIDENCIAS[k][0]} {d:+.3f}" for k, d in b["contribuciones"]))
        if r.vuelo and r.vuelo.get("evaluados"):
            st.markdown("**Evaluación de ventanas de vuelo**")
            st.dataframe(pd.DataFrame([{"día": e["dia"], "apto": e["apto"], "viento m/s": e["viento"], "lluvia mm/h": e["lluvia"],
                                        "visib. km": e["visibilidad"], "motivos": "; ".join(e["motivos"])} for e in r.vuelo["evaluados"]]),
                         hide_index=True, use_container_width=True)
        st.caption(f"Latencia total {r.latencia_ms:.0f} ms (NLU {r.latencia_nlu_ms:.0f} ms)")
    with c2:
        if r.mapa and (r.mapa.get("concesion") or r.mapa.get("punto")):
            st.pyplot(dibujar(asistente.kb, r.mapa.get("concesion"), r.mapa.get("punto")), use_container_width=True)


for msg in st.session_state.historial:
    with st.chat_message(msg["rol"], avatar="🧑‍💼" if msg["rol"] == "user" else "🛰️"):
        st.markdown(msg["texto"])
        if msg.get("r") is not None and ver_traza:
            with st.expander("Trazabilidad de la respuesta", expanded=msg.get("ultimo", False)):
                mostrar_traza(msg["r"])

entrada = st.chat_input("Escriba su consulta…") or st.session_state.pop("pendiente_ejemplo", None)
if entrada:
    for m in st.session_state.historial:
        m["ultimo"] = False
    r = asistente.responder(entrada)
    st.session_state.historial.append({"rol": "user", "texto": entrada})
    texto = r.texto + ("\n\n" + "\n".join(f"⚠️ {a}" for a in r.avisos) if r.avisos else "")
    st.session_state.historial.append({"rol": "assistant", "texto": texto, "r": r, "ultimo": True})
    st.rerun()


In [ ]:
%%writefile requirements.txt
# Ejecución del ChatBot (inferencia)
torch>=2.2
transformers>=4.40
safetensors>=0.4
scikit-learn>=1.4
numpy>=1.26
pandas>=2.0
pyproj>=3.6
shapely>=2.0
pgmpy>=0.1.25
matplotlib>=3.8
streamlit>=1.35
joblib>=1.3
# Solo para reentrenar y reproducir experimentos
scipy>=1.11
pytest>=8.0
spacy>=3.8,<3.9
srsly>=2.4
python-docx>=1.1        # solo para generar el informe


## 3. BETO, base de conocimiento y corpus

In [ ]:
!python scripts/extraer_beto.py --salida modelos/beto_base
!python datos/construir_base.py
!python datos/construir_corpus.py

## 4. Modelado
Iteración 1: validación cruzada de M0–M3 y modelos finales. Iteración 2: ajustes tras el análisis de fallos.

In [ ]:
!python experimentos/entrenar_nlp.py --version v1
!python experimentos/entrenar_nlp.py --version v2 --aumento --sin-cv --modelos M0,M3 --iteracion 2 --epocas-m3 10 --paciencia-m3 3

## 5. Pruebas

In [ ]:
!python -m pytest pruebas/test_unitarias.py -q
!python pruebas/suite_aceptacion.py --version v2
!python experimentos/figuras.py --version v1 --v2 v2

## 6. Conversar con KuriBot

In [ ]:
from kuribot.dialogo import KuriBot
bot = KuriBot()
for t in ["hola", "hay una retroexcavadora y gente extraña trabajando en Cerro Azul",
          "¿el punto E 642600 N 9662480 PSAD56 en qué concesión queda?", "necesito un vuelo de dron en San Gregorio", "mañana"]:
    r = bot.responder(t)
    print("Usted  ›", t)
    print("KuriBot›", r.texto)
    print("        [", r.intencion, f"{r.confianza:.2f}", r.decision, r.reglas, "]\n")